# SMBHB Population Analysis & CGW SNR Diagnostics

This notebook analyzes SMBHB population outputs from the HPC pipeline and computes Continuous GW (CGW) SNR diagnostics.

## Quick Start

To run a complete CGW analysis:

1. **Activate the venv** (in terminal):
   ```bash
   . ~/jupyter/bin/activate
   cd /fred/oz005/users/bhoulden/SMBHB_population_injections
   export PYTHONPATH=$(pwd)
   jupyter lab
   ```

2. **Run cells in this order**:
   - Cell 1-2: **Setup** (imports, compatibility, plotting)
   - Cell 3-5: **Discovery & Loaders** (find result files, define unpickling)
   - Cell 6: **Load Binary Data** (populates `binary_df`)
   - Cell 7-8: **Simulation Summary** (aggregates to `sim_df`)
   - Cell 16-26: **CGW Analysis** (main plots: nearest distance, loudest binary, sky maps)

3. **Outputs**: Plots saved to `figures/` directory

## Formats Supported

- Old format: `data/YYYY-MM-DD/*/consistent_pop_synth*.pkl.gz`
- New Slurm format: `runs/YYYY-MM-DD_<scenario>/sim<NNN>/summary.pkl.gz` + `populations/subpop_*.pkl.gz`

Both are detected and loaded automatically.

In [1]:
from __future__ import annotations

import json
import gzip
import pickle
import re
import sys
from typing import Optional
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# NumPy compatibility: handle both NumPy 1.x (has numpy.core) and NumPy 2.x (has numpy._core)
NUMPY_VERSION_TUPLE = tuple(map(int, np.__version__.split('.')[:2]))
IS_NUMPY_2X = NUMPY_VERSION_TUPLE >= (2, 0)

if IS_NUMPY_2X:
    # We're on NumPy 2.x, create shim for unpickling NumPy 1.x pickles
    try:
        import numpy._core
        if not hasattr(np, 'core'):
            np.core = numpy._core
            sys.modules['numpy.core'] = numpy._core
    except (ImportError, AttributeError):
        pass

plt.style.use('default')
pd.set_option('display.max_columns', 50)

import types as _types

def _patch_numpy_modules():
    """Shim numpy.core → numpy._core for pickles created on NumPy 1.x."""
    if IS_NUMPY_2X:
        if 'numpy._core' not in sys.modules:
            _mod = _types.ModuleType('numpy._core')
            sys.modules['numpy._core'] = _mod
        if hasattr(np, '_core'):
            sys.modules['numpy._core'] = np._core
            if hasattr(np._core, 'multiarray'):
                sys.modules['numpy._core.multiarray'] = np._core.multiarray
    else:
        # NumPy 1.x: shim numpy._core → numpy.core
        if 'numpy._core' not in sys.modules:
            sys.modules['numpy._core'] = np.core
        if 'numpy._core.multiarray' not in sys.modules:
            sys.modules['numpy._core.multiarray'] = np.core.multiarray

_patch_numpy_modules()

SCENARIOS = ('optimistic', 'realistic', 'pessimistic')

In [2]:
import matplotlib as mpl

mpl.rcParams.update({
    # Font
    'font.family':       'serif',
    'font.serif':        ['Times New Roman', 'DejaVu Serif'],
    'mathtext.fontset':  'stix',        # matches Times for math/LaTeX symbols

    # Font sizes (ApJ single-column ~3.5in, so these scale well)
    'font.size':         12,
    'axes.titlesize':    12,
    'axes.labelsize':    12,
    'xtick.labelsize':   10,
    'ytick.labelsize':   10,
    'legend.fontsize':   10,

    # Figure size (single-column ApJ width = 3.5in)
    'figure.figsize':    (3.5, 2.8),    # use (7.0, 2.8) for double-column

    # Line/tick quality
    'axes.linewidth':    0.8,
    'xtick.major.width': 0.8,
    'ytick.major.width': 0.8,
    'xtick.minor.width': 0.6,
    'ytick.minor.width': 0.6,
    'xtick.direction':   'in',          # ApJ style: ticks point inward
    'ytick.direction':   'in',
    'xtick.top':         True,          # ticks on all four sides
    'ytick.right':       True,
    'axes.spines.top':    True,
    'axes.spines.right':  True,
    'axes.spines.left':   True,
    'axes.spines.bottom': True,

    # Output
    'savefig.dpi':       300,
    'savefig.bbox':      'tight',
})

In [3]:
def infer_scenario(path: Path) -> str:
    lower_parts = [p.lower() for p in path.parts]
    for s in SCENARIOS:
        if any(s in p for p in lower_parts):
            return s
        if s in path.name.lower():
            return s
    return 'unknown'


def infer_run_id(path: Path) -> str:
    date_cfg_re = re.compile(r'^\d{4}-\d{2}-\d{2}_.+$')
    date_re = re.compile(r'^\d{4}-\d{2}-\d{2}$')
    sim_re = re.compile(r'^sim\d+$')

    for p in path.parts:
        if date_cfg_re.match(p):
            return p
    for p in path.parts:
        if date_re.match(p):
            return p

    for parent in path.parents:
        name = parent.name
        if not name:
            continue
        if name in {'data', 'runs', 'results', 'output', 'outputs'}:
            continue
        if sim_re.match(name):
            continue
        return name
    return 'legacy'


def discover_result_files(data_roots = Path('data')):
    # Summary-only mode: only load summary.pkl.gz files from runs/.
    if isinstance(data_roots, (str, Path)):
        roots = [Path(data_roots)]
    else:
        roots = [Path(root) for root in data_roots]

    patterns = ('summary.pkl.gz',)

    ordered_files: list[Path] = []
    seen: set[Path] = set()
    for root in roots:
        if not root.exists():
            continue
        root_matches = set()
        for pattern in patterns:
            root_matches.update(root.rglob(pattern))
        for path in sorted(p for p in root_matches if p.is_file()):
            if path not in seen:
                seen.add(path)
                ordered_files.append(path)

    return ordered_files


# ANALYSIS_ROOTS = [Path('data'), Path('runs')]
ANALYSIS_ROOTS = [Path('runs')]
result_files = discover_result_files(ANALYSIS_ROOTS)
print(f'Discovered {len(result_files)} summary files')
for p in result_files[::-1][:20]:  # Show most recent 20 files
    print('-', p)
if len(result_files) > 20:
    print('...')

Discovered 1200 summary files
- runs/2026-06-11_pessimistic/sim548/summary.pkl.gz
- runs/2026-06-11_pessimistic/sim547/summary.pkl.gz
- runs/2026-06-10_realistic/sim510/summary.pkl.gz
- runs/2026-06-10_realistic/sim509/summary.pkl.gz
- runs/2026-06-10_realistic/sim508/summary.pkl.gz
- runs/2026-06-10_realistic/sim507/summary.pkl.gz
- runs/2026-06-10_realistic/sim506/summary.pkl.gz
- runs/2026-06-10_realistic/sim505/summary.pkl.gz
- runs/2026-06-10_realistic/sim504/summary.pkl.gz
- runs/2026-06-10_realistic/sim503/summary.pkl.gz
- runs/2026-06-10_realistic/sim502/summary.pkl.gz
- runs/2026-06-10_realistic/sim501/summary.pkl.gz
- runs/2026-06-10_realistic/sim500/summary.pkl.gz
- runs/2026-06-10_realistic/sim498/summary.pkl.gz
- runs/2026-06-10_realistic/sim497/summary.pkl.gz
- runs/2026-06-10_realistic/sim496/summary.pkl.gz
- runs/2026-06-10_realistic/sim495/summary.pkl.gz
- runs/2026-06-10_realistic/sim493/summary.pkl.gz
- runs/2026-06-10_realistic/sim492/summary.pkl.gz
- runs/2026-06-1

In [18]:
HUBBLE_CONSTANT_H   = 0.67
OMEGA_MATTER        = 0.3
OMEGA_LAMBDA        = 0.7
H0_KMS_MPC          = 100 * HUBBLE_CONSTANT_H          # km/s/Mpc
SPEED_OF_LIGHT_KMS  = 2.9979e5                          # km/s
SPEED_OF_LIGHT_MS   = SPEED_OF_LIGHT_KMS * 1e3          # m/s

def hubble_parameter(z: np.ndarray) -> np.ndarray:
    """H(z) for flat ΛCDM [km/s/Mpc]."""
    return H0_KMS_MPC * np.sqrt(OMEGA_MATTER * (1 + z)**3 + OMEGA_LAMBDA)

from scipy.integrate import quad 
from scipy.interpolate import interp1d
def build_comoving_distance_interpolator(z_max: float = 20.0,
                                         n_points: int = 2_000_000):
    """Cubic interpolator: z → comoving distance [Mpc]."""
    z_grid   = np.linspace(0, z_max, n_points)
    chi_grid = np.array([
        quad(lambda zp: SPEED_OF_LIGHT_KMS / hubble_parameter(zp), 0, zi)[0]
        for zi in z_grid
    ])
    interp = interp1d(z_grid, chi_grid, kind='cubic', fill_value='extrapolate')
    return lambda z: interp(np.atleast_1d(z)).squeeze()[()]
 
# Build module-level interpolation grids once
_CHI_FN   = build_comoving_distance_interpolator(z_max=20.0)
 
# Fine grid for fast vectorised inversion (chi → z via np.interp)
_Z_GRID   = np.linspace(0, 20.0, 10_000_000)
_CHI_GRID = _CHI_FN(_Z_GRID)          # monotone increasing, shape (10_000_000,)
 
# Expose for backward compatibility
COMOVING_DISTANCE_FN         = _CHI_FN
Z_GRID_NUMBA                 = _Z_GRID
CHI_GRID_NUMBA               = _CHI_GRID

/tmp/ipykernel_3626116/2576969880.py:38: RuntimeWarning: divide by zero encountered in true_divide
  plt.loglog(_Z_GRID, (1 + _Z_GRID)**(2/3) / _CHI_GRID)


In [21]:

plt.loglog(_Z_GRID, (1 + _Z_GRID)**(2/3) / _CHI_GRID)
plt.ylabel(r"$(1+z)^{2/3} / D_{\rm{comov}}$")
plt.xlabel("z")
plt.xlim(1e-2, 10)
plt.savefig('figures/comov_z.png')
plt.show()

/tmp/ipykernel_3626116/2135719142.py:1: RuntimeWarning: divide by zero encountered in true_divide
  plt.loglog(_Z_GRID, (1 + _Z_GRID)**(2/3) / _CHI_GRID)


In [14]:
np.argmin((1 + _Z_GRID)**(2/3) / _CHI_GRID)
_Z_GRID[1325029]

/tmp/ipykernel_3626116/1095686897.py:1: RuntimeWarning: divide by zero encountered in true_divide
  np.argmin((1 + _Z_GRID)**(2/3) / _CHI_GRID)


2.6500582650058266

In [114]:
def _extract_array_from_population_string(pop_text: str, key: str) -> Optional[np.ndarray]:
    # Extract key=array([ ... ]) from stringified PopulationArrays(...) output.
    pattern = rf"{re.escape(key)}=array\(\[(.*?)\]"
    match = re.search(pattern, pop_text, flags=re.DOTALL)
    if not match:
        return None

    raw = match.group(1).replace('\n', ' ')

    try:
        arr = np.fromstring(raw, sep=',')
    except ValueError:
        arr = np.array([], dtype=float)

    if arr.size == 0 and raw.strip():
        try:
            arr = np.fromstring(raw.replace(',', ' '), sep=' ')
        except ValueError:
            arr = np.array([], dtype=float)

    if arr.size == 0 and raw.strip():
        tokens = re.findall(
            r'[-+]?(?:\d*\.\d+|\d+)(?:[eE][-+]?\d+)?|[-+]?inf|nan',
            raw,
            flags=re.IGNORECASE,
        )
        if tokens:
            arr = np.asarray([float(t) for t in tokens], dtype=float)

    return arr if arr.size > 0 else None


def _population_to_arrays(population_obj) -> Optional[dict[str, np.ndarray]]:
    # Case 0: Slurm summary payload written by stage2_inject.py.
    if isinstance(population_obj, dict):
        arrays = population_obj.get('arrays')
        if isinstance(arrays, dict):
            out = {}
            for k in ('f', 'Mc', 'D', 'D_comov', 'h0', 'z', 'Mtot', 'cgw_snr', 'ra', 'dec', 'psi', 'iota', 'phi0'):
                arr = arrays.get(k)
                if arr is not None:
                    out[k] = np.asarray(arr, dtype=float)
            for k in ('global_idx', 'sim_id'):
                arr = arrays.get(k)
                if arr is not None:
                    out[k] = np.asarray(arr)
            return out if out else None

    # Case 1: population stored as dict with list/array values (compact representative format)
    if isinstance(population_obj, dict):
        has_pop_fields = any(k in population_obj for k in ('f', 'Mc', 'Mtot', 'D_comov', 'h0', 'z'))
        if has_pop_fields:
            out = {}
            for k in ('f', 'Mc', 'D', 'D_comov', 'h0', 'z', 'Mtot', 'cgw_snr', 'ra', 'dec', 'psi', 'iota', 'phi0'):
                if k in population_obj:
                    arr = population_obj[k]
                    if arr is not None:
                        out[k] = np.asarray(arr, dtype=float)
            return out if out else None

    # Case 2: population stored as list of dict rows (legacy format)
    if isinstance(population_obj, list):
        if not population_obj:
            return None
        out = {}
        for k in ('f', 'Mc', 'D', 'D_comov', 'h0', 'z', 'cgw_snr', 'ra', 'dec', 'psi', 'iota', 'phi0'):
            vals = [row.get(k, np.nan) for row in population_obj]
            out[k] = np.asarray(vals, dtype=float)
        return out

    # Case 3: true PopulationArrays object (via pickle)
    if hasattr(population_obj, 'f') and hasattr(population_obj, 'Mc'):
        out = {}
        for k in ('f', 'Mc', 'D', 'D_comov', 'h0', 'z', 'cgw_snr', 'ra', 'dec', 'psi', 'iota', 'phi0'):
            if hasattr(population_obj, k):
                arr = getattr(population_obj, k)
                if arr is not None:
                    out[k] = np.asarray(arr, dtype=float)
        return out if out else None

    # Case 4: stringified PopulationArrays(...) (legacy JSON fallback)
    if isinstance(population_obj, str):
        out = {}
        for k in ('f', 'Mc', 'D', 'D_comov', 'h0', 'z', 'ra', 'dec', 'psi', 'iota', 'phi0'):
            arr = _extract_array_from_population_string(population_obj, k)
            if arr is not None:
                out[k] = arr.astype(float)
        return out if out else None

    return None


def _entry_to_binary_rows(entry: dict, source_file: Path, scenario: str, run_id: str, fallback_sim_indexx: int) -> list[dict]:
    """Build a binary dataframe from array-backed summary data without a Python row loop."""
    if not arrays or 'f' not in arrays:
        return pd.DataFrame()

    n = len(arrays['f'])
    d = arrays.get('D_comov')
    if d is None:
        d = arrays.get('D')

    return pd.DataFrame({
        'scenario': np.repeat(scenario, n),
        'run_id': np.repeat(run_id, n),
        'source_file': np.repeat(str(source_file), n),
        'sim_index': np.repeat(np.int32(sim_index), n),
        'sim_key': np.repeat(f'{scenario}:{run_id}:{sim_index}', n),
        'binary_index': np.arange(n, dtype=np.int32),
        'f': np.asarray(arrays.get('f', np.full(n, np.nan)), dtype=float),
        'Mc': np.asarray(arrays.get('Mc', np.full(n, np.nan)), dtype=float),
        'D': np.asarray(d if d is not None else np.full(n, np.nan), dtype=float),
        'h0': np.asarray(arrays.get('h0', np.full(n, np.nan)), dtype=float),
        'z': np.asarray(arrays.get('z', np.full(n, np.nan)), dtype=float),
        'cgw_snr': np.asarray(arrays.get('cgw_snr', np.full(n, np.nan)), dtype=float),
        'ra': np.asarray(arrays.get('ra', np.full(n, np.nan)), dtype=float),
        'dec': np.asarray(arrays.get('dec', np.full(n, np.nan)), dtype=float),
        'psi': np.asarray(arrays.get('psi', np.full(n, np.nan)), dtype=float),
        'iota': np.asarray(arrays.get('iota', np.full(n, np.nan)), dtype=float),
        'phi0': np.asarray(arrays.get('phi0', np.full(n, np.nan)), dtype=float),
        'Mtot': np.asarray(arrays.get('Mtot', np.full(n, np.nan)), dtype=float),
        'global_idx': np.asarray(arrays.get('global_idx', np.arange(n, dtype=np.int64)), dtype=np.int64),
    })

In [115]:
def _summary_arrays_to_dataframe(arrays: dict, scenario: str, run_id: str, source_file: Path, sim_index: int = -1) -> pd.DataFrame:
    """Build a binary dataframe from array-backed summary data without a Python row loop."""
    if not arrays or 'f' not in arrays:
        return pd.DataFrame()

    n = len(arrays['f'])
    d = arrays.get('D_comov')
    if d is None:
        d = arrays.get('D')

    run_scope = _infer_run_scope_from_path(source_file)
    sim_name = source_file.parent.name
    sim_key = f'{run_scope}/{sim_name}' if run_scope else sim_name

    frame = pd.DataFrame({
        'scenario': np.repeat(scenario, n),
        'run_id': np.repeat(run_id, n),
        'run_scope': np.repeat(run_scope, n),
        'sim_name': np.repeat(sim_name, n),
        'source_file': np.repeat(str(source_file), n),
        'sim_index': np.repeat(np.int32(sim_index), n),
        'sim_key': np.repeat(sim_key, n),
        'binary_index': np.arange(n, dtype=np.int32),
        'f': np.asarray(arrays.get('f', np.full(n, np.nan)), dtype=np.float32),
        'Mc': np.asarray(arrays.get('Mc', np.full(n, np.nan)), dtype=np.float32),
        'D': np.asarray(d if d is not None else np.full(n, np.nan), dtype=np.float32),
        'h0': np.asarray(arrays.get('h0', np.full(n, np.nan)), dtype=np.float32),
        'z': np.asarray(arrays.get('z', np.full(n, np.nan)), dtype=np.float32),
        'cgw_snr': np.asarray(arrays.get('cgw_snr', np.full(n, np.nan)), dtype=np.float32),
        'ra': np.asarray(arrays.get('ra', np.full(n, np.nan)), dtype=np.float32),
        'dec': np.asarray(arrays.get('dec', np.full(n, np.nan)), dtype=np.float32),
        'psi': np.asarray(arrays.get('psi', np.full(n, np.nan)), dtype=np.float32),
        'iota': np.asarray(arrays.get('iota', np.full(n, np.nan)), dtype=np.float32),
        'phi0': np.asarray(arrays.get('phi0', np.full(n, np.nan)), dtype=np.float32),
        'Mtot': np.asarray(arrays.get('Mtot', np.full(n, np.nan)), dtype=np.float32),
        'global_idx': np.asarray(arrays.get('global_idx', np.arange(n, dtype=np.int32)), dtype=np.int32),
    })

    for column in ('scenario', 'run_id', 'run_scope', 'sim_name', 'source_file', 'sim_key'):
        frame[column] = frame[column].astype('category')

    return frame


In [116]:
from dataclasses import dataclass, field

@dataclass
class PopulationArrays:
    f       : np.ndarray
    Mc      : np.ndarray
    Mtot    : np.ndarray
    D_comov : np.ndarray
    z       : np.ndarray
    h0      : np.ndarray
    ra      : np.ndarray
    dec     : np.ndarray
    psi     : np.ndarray
    iota    : np.ndarray
    phi0    : np.ndarray
    cgw_snr : np.ndarray
    amp_A   : Dict[str, np.ndarray] = field(default_factory=dict)
    amp_B   : Dict[str, np.ndarray] = field(default_factory=dict)

    def __len__(self): return len(self.f)

    def __getitem__(self, idx):
        new = PopulationArrays(
            f=self.f[idx], Mc=self.Mc[idx], Mtot=self.Mtot[idx],
            D_comov=self.D_comov[idx], z=self.z[idx], h0=self.h0[idx],
            ra=self.ra[idx], dec=self.dec[idx], psi=self.psi[idx],
            iota=self.iota[idx], phi0=self.phi0[idx], cgw_snr=self.cgw_snr[idx],
        )
        for k, v in self.amp_A.items(): new.amp_A[k] = v[idx]
        for k, v in self.amp_B.items(): new.amp_B[k] = v[idx]
        return new
    
from pathlib import Path

def _summary_rows_from_sim_directory(
    summary_file: Path,
    payload: dict,
    scenario: str,
    run_id: str,
    verbose: bool = False,
) -> pd.DataFrame:
    arrays = payload.get('arrays', {}) if isinstance(payload, dict) else {}
    if not isinstance(arrays, dict):
        return pd.DataFrame()

    global_idx = arrays.get('global_idx')
    if global_idx is None:
        if verbose:
            print(f'  No global_idx in summary: {summary_file}')
        return pd.DataFrame()

    global_idx = np.asarray(global_idx, dtype=np.int64)
    if global_idx.size == 0:
        return pd.DataFrame()

    FIELDS = ['f','Mc','Mtot','D_comov','z','h0','ra','dec','psi','iota','phi0','cgw_snr']

    # ── Fast path: physical arrays already in summary ──────────────────────
    if 'f' in arrays:
        if verbose:
            print(f'  Fast path: reading arrays directly from summary')
        frame = _summary_arrays_to_dataframe(arrays, scenario, run_id, summary_file, sim_index=-1)
        if not frame.empty:
            frame['global_idx'] = np.asarray(global_idx, dtype=np.int64)
        return frame

    # ── Fallback: read physical arrays from shard files ────────────────────
    pop_dir = summary_file.parent / 'populations'
    if not pop_dir.exists():
        if verbose:
            print(f'  No arrays in summary and no populations/ dir: {summary_file}')
        return pd.DataFrame()

    if verbose:
        print(f'  Sparse summary — reading from shards in {pop_dir}')

    # Build a map: global_idx → shard file + local position
    shard_files = sorted(pop_dir.glob('subpop_*.pkl.gz'))
    needed = set(global_idx.tolist())
    collected: dict[int, dict] = {}
    global_counter = 0

    for shard_path in shard_files:
        if not needed:
            break
        try:
            with gzip.open(shard_path, 'rb') as fh:
                pop = _CompatibilityUnpickler(fh).load()
        except Exception as e:
            if verbose:
                print(f'    Failed to load {shard_path.name}: {e}')
            global_counter += 0
            continue

        n = len(pop.f)
        for local_i in range(n):
            gidx = global_counter + local_i
            if gidx in needed:
                collected[gidx] = {
                    f: float(getattr(pop, f)[local_i])
                    for f in FIELDS
                    if hasattr(pop, f) and getattr(pop, f) is not None
                }
                needed.discard(gidx)
        global_counter += n
        del pop

    if verbose:
        print(f'  Recovered {len(collected)}/{global_idx.size} binaries from shards')

    # Extract sim_index from directory name
    sim_match = re.search(r'sim(\d+)', summary_file.parent.name)
    sim_index = int(sim_match.group(1)) if sim_match else -1

    rows = []
    for i, gidx in enumerate(global_idx.tolist()):
        row_data = collected.get(int(gidx), {})
        rows.append({
            'scenario': scenario, 'run_id': run_id,
            'source_file': str(summary_file),
            'sim_index': np.int32(sim_index),
            'binary_index': np.int32(i),
            'global_idx': np.int64(gidx),
            **{f: float(row_data.get(f, np.nan)) for f in FIELDS},
            'D': float(row_data.get('D_comov', np.nan)),
        })
    return pd.DataFrame(rows)

def _load_payload(path: Path, verbose: bool = False):
    """Load a .pkl.gz or .pkl file, trying CompatibilityUnpickler first."""
    import gzip, pickle
    try:
        with gzip.open(path, 'rb') as f:
            try:
                return _CompatibilityUnpickler(f).load()
            except Exception:
                pass
        with gzip.open(path, 'rb') as f:
            return pickle.load(f)
    except Exception as e:
        if verbose:
            print(f"  Failed to load {path}: {e}")
        return None
    
class _CompatibilityUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        # Redirect PopulationArrays from ANY module path to our local definition
        if name == 'PopulationArrays':
            return PopulationArrays
        # numpy core shim
        if module.startswith('numpy._core'):
            module = module.replace('numpy._core', 'numpy.core')
        elif module.startswith('numpy.core') and IS_NUMPY_2X:
            try:
                return super().find_class(module, name)
            except Exception:
                module = module.replace('numpy.core', 'numpy._core')
        # stub anything missing (numba etc.)
        try:
            return super().find_class(module, name)
        except Exception:
            _stub_missing_modules(module)
            try:
                return super().find_class(module, name)
            except Exception:
                return type(name, (), {'__module__': module})

In [117]:
# Override loader stubs for the Jupyter venv.
# The shard pickles import numba decorators such as `from numba import njit`.
# If we create `numba.njit` as a submodule, Python binds it as a module object
# and unpickling fails with `'module' object is not callable`.
# Keep the base `numba` module callable via decorator attributes, but do not
# create `numba.njit`/`numba.prange` submodules.

for _bad_mod in ('numba.njit', 'numba.prange', 'numba.vectorize', 'numba.guvectorize'):
    sys.modules.pop(_bad_mod, None)


def _stub_missing_modules(*module_names):
    for name in module_names:
        if name in sys.modules:
            continue
        try:
            __import__(name)
            continue
        except Exception:
            parts = name.split('.')
            if parts[0] == 'numba':
                stub = sys.modules.get('numba')
                if stub is None:
                    stub = types.ModuleType('numba')
                    stub.__path__ = []
                    sys.modules['numba'] = stub
                stub.jit = lambda *a, **k: (lambda f: f)
                stub.njit = lambda *a, **k: (lambda f: f)
                stub.vectorize = lambda *a, **k: (lambda f: f)
                stub.guvectorize = lambda *a, **k: (lambda f: f)
                stub.prange = range
                continue

            for i in range(len(parts)):
                parent = '.'.join(parts[: i + 1])
                if parent not in sys.modules:
                    stub = types.ModuleType(parent)
                    stub.__path__ = []
                    sys.modules[parent] = stub


# Re-run the numpy shim just in case this cell is executed standalone.
_patch_numpy_modules()

In [118]:
# DEBUG: Inspect what's in the discovered files
result_files_all = discover_result_files([Path('runs')])
print(f"Total files discovered: {len(result_files_all)}")

# Group by parent structure
from collections import defaultdict
by_sim = defaultdict(list)
for f in result_files_all:
    sim_dir = f.parent.name if 'sim' in f.parent.name else 'other'
    by_sim[sim_dir].append(f)

print(f"\nFile organization:")
for sim, files in sorted(by_sim.items())[:5]:
    print(f"  {sim}: {len(files)} files")
    for f in files[:2]:
        print(f"    - {f.name}")

# Try loading first few files and inspect their structure
print(f"\nInspecting file contents:")
for i, fp in enumerate(result_files_all[:3]):
    print(f"\n{i}. {fp.relative_to('.')}")
    try:
        payload = _load_payload(fp, verbose=False)
        print(f"   Type: {type(payload)}")
        if isinstance(payload, dict):
            print(f"   Keys: {list(payload.keys())[:10]}")
            if 'arrays' in payload:
                arrays = payload['arrays']
                print(f"   arrays type: {type(arrays)}")
                if isinstance(arrays, dict):
                    print(f"   arrays keys: {list(arrays.keys())}")
                    if 'global_idx' in arrays:
                        print(f"   global_idx: {type(arrays['global_idx'])}, len={len(arrays['global_idx']) if hasattr(arrays['global_idx'], '__len__') else '?'}")
        elif hasattr(payload, '__dict__'):
            print(f"   Attrs: {list(vars(payload).keys())[:5]}")
    except Exception as e:
        print(f"   Error: {type(e).__name__}: {e}")


Total files discovered: 1200

File organization:
  sim000: 3 files
    - summary.pkl.gz
    - summary.pkl.gz
  sim002: 2 files
    - summary.pkl.gz
    - summary.pkl.gz
  sim003: 2 files
    - summary.pkl.gz
    - summary.pkl.gz
  sim004: 2 files
    - summary.pkl.gz
    - summary.pkl.gz
  sim005: 1 files
    - summary.pkl.gz

Inspecting file contents:

0. runs/2026-06-05_optimistic/sim000/summary.pkl.gz
   Type: <class 'dict'>
   Keys: ['arrays', 'meta']
   arrays type: <class 'dict'>
   arrays keys: ['f', 'Mc', 'Mtot', 'D_comov', 'z', 'h0', 'ra', 'dec', 'psi', 'iota', 'phi0', 'cgw_snr', 'cgw_proxy', 'cgw_snr_5x_cadence', 'cgw_snr_4x_precision', 'cgw_snr_5x_cad_4x_prec', 'global_idx']
   global_idx: <class 'numpy.ndarray'>, len=13168

1. runs/2026-06-05_optimistic/sim002/summary.pkl.gz
   Type: <class 'dict'>
   Keys: ['arrays', 'meta']
   arrays type: <class 'dict'>
   arrays keys: ['f', 'Mc', 'Mtot', 'D_comov', 'z', 'h0', 'ra', 'dec', 'psi', 'iota', 'phi0', 'cgw_snr', 'cgw_proxy', '

In [11]:
import gzip, pickle, traceback

path = 'runs/2026-05-20_optimistic/sim000/summary.pkl.gz'
try:
    with gzip.open(path, 'rb') as f:
        _CompatibilityUnpickler(f).load()
except Exception:
    traceback.print_exc()

Traceback (most recent call last):
  File "/tmp/ipykernel_642334/638422092.py", line 5, in <module>
    with gzip.open(path, 'rb') as f:
  File "/apps/modules/software/Python/3.10.4-GCCcore-11.3.0/lib/python3.10/gzip.py", line 58, in open
    binary_file = GzipFile(filename, gz_mode, compresslevel)
  File "/apps/modules/software/Python/3.10.4-GCCcore-11.3.0/lib/python3.10/gzip.py", line 174, in __init__
    fileobj = self.myfileobj = builtins.open(filename, mode or 'rb')
FileNotFoundError: [Errno 2] No such file or directory: 'runs/2026-05-20_optimistic/sim000/summary.pkl.gz'


In [119]:
# ============================================
# STEP 1: LOAD BINARY DATAFRAME FROM RUNS
# ============================================

import gc


def _infer_run_scope_from_path(path: Path) -> str:
    try:
        return path.parent.parent.relative_to(Path('runs')).as_posix()
    except Exception:
        return path.parent.parent.as_posix()


def _infer_sim_index_from_path(path: Path) -> int:
    match = re.search(r'sim(\d+)', path.parent.name)
    return int(match.group(1)) if match else -1


def load_population_binary_table_robust(files, verbose=False):
    """
    Load binaries from summary files, handling multiple formats:
    1. Sparse summary: dict with 'arrays' key
    2. Compact format: dict with 'populations' list
    3. Direct array format: PopulationArrays-like object
    """
    frames = []

    for fp in files:
        if verbose:
            print(f"Loading: {fp.name}")

        payload = None
        try:
            scenario = infer_scenario(fp)
            run_id = infer_run_id(fp)
            run_scope = _infer_run_scope_from_path(fp)
            sim_name = fp.parent.name
            sim_index = _infer_sim_index_from_path(fp)
            sim_key = f'{run_scope}/{sim_name}' if run_scope else sim_name
            payload = _load_payload(fp, verbose=False)

            if payload is None or (isinstance(payload, dict) and not payload):
                if verbose:
                    print("  → Empty payload")
                continue

            # FORMAT 1: Sparse summary (dict with 'arrays' key)
            if isinstance(payload, dict) and 'arrays' in payload:
                if verbose:
                    print("  → Sparse format detected")
                df_rows = _summary_rows_from_sim_directory(fp, payload, scenario, run_id, verbose=verbose)
                if not df_rows.empty:
                    df_rows['run_scope'] = run_scope
                    df_rows['sim_name'] = sim_name
                    df_rows['sim_index'] = np.int32(sim_index)
                    df_rows['sim_key'] = sim_key
                    if 'source_file' in df_rows.columns:
                        df_rows['source_file'] = str(fp)
                    frames.append(df_rows)
                    if verbose:
                        print(f"    → {len(df_rows)} rows from sparse")
                continue

            # FORMAT 2: Old compact format (dict with 'populations' list)
            if isinstance(payload, dict) and 'populations' in payload:
                pops = payload.get('populations', [])
                if isinstance(pops, list) and pops:
                    if verbose:
                        print(f"  → Compact format: {len(pops)} populations")
                    rows = []

                    for pop_idx, pop_entry in enumerate(pops):
                        if not isinstance(pop_entry, dict):
                            continue
                        pop_obj = pop_entry.get('population', pop_entry)
                        arrays = _population_to_arrays(pop_obj)
                        if not arrays or 'f' not in arrays:
                            continue

                        n = len(arrays['f'])
                        for bi in range(n):
                            rows.append({
                                'scenario': scenario,
                                'run_id': run_id,
                                'run_scope': run_scope,
                                'sim_name': sim_name,
                                'source_file': str(fp),
                                'sim_index': np.int32(sim_index),
                                'binary_index': np.int32(bi),
                                'global_idx': np.int32(bi),
                                'sim_key': sim_key,
                                'f': np.float32(arrays['f'][bi]),
                                'Mc': np.float32(arrays.get('Mc', [np.nan] * n)[bi]),
                                'D': np.float32(arrays.get('D_comov', arrays.get('D', [np.nan] * n))[bi]),
                                'h0': np.float32(arrays.get('h0', [np.nan] * n)[bi]),
                                'z': np.float32(arrays.get('z', [np.nan] * n)[bi]),
                                'cgw_snr': np.float32(arrays.get('cgw_snr', [np.nan] * n)[bi]),
                                'ra': np.float32(arrays.get('ra', [np.nan] * n)[bi]),
                                'dec': np.float32(arrays.get('dec', [np.nan] * n)[bi]),
                                'psi': np.float32(arrays.get('psi', [np.nan] * n)[bi]),
                                'iota': np.float32(arrays.get('iota', [np.nan] * n)[bi]),
                                'phi0': np.float32(arrays.get('phi0', [np.nan] * n)[bi]),
                                'Mtot': np.float32(arrays.get('Mtot', [np.nan] * n)[bi]),
                            })

                    if rows:
                        frames.append(pd.DataFrame(rows))
                        if verbose:
                            print(f"    → {len(rows)} rows from compact")
                    continue

            # FORMAT 3: Direct array object (has f, Mc, D attributes)
            if hasattr(payload, 'f'):
                if verbose:
                    print(f"  → Direct array format: {type(payload).__name__}")
                rows = []
                n = len(payload.f)
                for bi in range(n):
                    rows.append({
                        'scenario': scenario,
                        'run_id': run_id,
                        'run_scope': run_scope,
                        'sim_name': sim_name,
                        'source_file': str(fp),
                        'sim_index': np.int32(sim_index),
                        'binary_index': np.int32(bi),
                        'global_idx': np.int32(bi),
                        'sim_key': sim_key,
                        'f': np.float32(payload.f[bi]),
                        'Mc': np.float32(payload.Mc[bi] if hasattr(payload, 'Mc') else np.nan),
                        'D': np.float32(payload.D_comov[bi] if hasattr(payload, 'D_comov') else np.nan),
                        'h0': np.float32(payload.h0[bi] if hasattr(payload, 'h0') else np.nan),
                        'z': np.float32(payload.z[bi] if hasattr(payload, 'z') else np.nan),
                        'cgw_snr': np.float32(payload.cgw_snr[bi] if hasattr(payload, 'cgw_snr') else np.nan),
                        'ra': np.float32(payload.ra[bi] if hasattr(payload, 'ra') else np.nan),
                        'dec': np.float32(payload.dec[bi] if hasattr(payload, 'dec') else np.nan),
                        'psi': np.float32(payload.psi[bi] if hasattr(payload, 'psi') else np.nan),
                        'iota': np.float32(payload.iota[bi] if hasattr(payload, 'iota') else np.nan),
                        'phi0': np.float32(payload.phi0[bi] if hasattr(payload, 'phi0') else np.nan),
                        'Mtot': np.float32(payload.Mtot[bi] if hasattr(payload, 'Mtot') else np.nan),
                    })
                if rows:
                    frames.append(pd.DataFrame(rows))
                    if verbose:
                        print(f"    → {n} rows from direct arrays")
                continue

            if verbose:
                print(f"  → Unrecognized format: {type(payload)}")

        except Exception as e:
            if verbose:
                print(f"  Error: {type(e).__name__}: {e}")
        finally:
            del payload
            if len(frames) % 8 == 0:
                gc.collect()

    if not frames:
        if verbose:
            print("→ No rows loaded from any file")
        return pd.DataFrame()

    df = pd.concat(frames, ignore_index=True)
    if verbose:
        print(f"\n✓ Total: {len(df)} rows across {len(files)} files")
    return df


def _population_to_arrays(pop_obj):
    """Convert various population formats to a dict of arrays."""
    if pop_obj is None:
        return None

    if isinstance(pop_obj, dict) and 'f' in pop_obj:
        return pop_obj

    if hasattr(pop_obj, 'f'):
        return {
            'f': pop_obj.f,
            'Mc': getattr(pop_obj, 'Mc', None),
            'D_comov': getattr(pop_obj, 'D_comov', getattr(pop_obj, 'D', None)),
            'h0': getattr(pop_obj, 'h0', None),
            'z': getattr(pop_obj, 'z', None),
            'cgw_snr': getattr(pop_obj, 'cgw_snr', None),
            'ra': getattr(pop_obj, 'ra', None),
            'dec': getattr(pop_obj, 'dec', None),
            'psi': getattr(pop_obj, 'psi', None),
            'iota': getattr(pop_obj, 'iota', None),
            'phi0': getattr(pop_obj, 'phi0', None),
            'Mtot': getattr(pop_obj, 'Mtot', None),
        }

    if isinstance(pop_obj, str) and 'PopulationArrays' in pop_obj:
        try:
            import ast
            pass
        except Exception:
            pass

    return None


print("Step 1: Discovering result files...")
result_files = list(discover_result_files([Path('runs')]))
print(f"  → Found {len(result_files)} result files")

if result_files:
    print("\nStep 2: Loading population binary data with robust parser...")
    binary_df = load_population_binary_table_robust(result_files, verbose=True)
    print(f"\n  → Final: {len(binary_df)} binary rows loaded")
    if len(binary_df) > 0:
        print(f"  → Scenarios: {sorted(binary_df['scenario'].unique().tolist())}")
        if 'cgw_snr' in binary_df.columns:
            print(f"  → CGW SNR > 0: {(binary_df['cgw_snr'] > 0).sum()} binaries")
        sky_cols = [c for c in ('ra', 'dec', 'psi', 'iota', 'phi0') if c in binary_df.columns]
        if sky_cols:
            print(f"  → Sky-location columns loaded: {sky_cols}")
        display(binary_df.head())
else:
    print("ERROR: No result files found in runs/")
    binary_df = pd.DataFrame()

print(f"\nDataFrame shape: {binary_df.shape}")
print(f"Columns: {list(binary_df.columns) if not binary_df.empty else 'N/A'}")


Step 1: Discovering result files...
  → Found 1200 result files

Step 2: Loading population binary data with robust parser...
Loading: summary.pkl.gz
  → Sparse format detected
  Fast path: reading arrays directly from summary
    → 13168 rows from sparse
Loading: summary.pkl.gz
  → Sparse format detected
  Fast path: reading arrays directly from summary
    → 13170 rows from sparse
Loading: summary.pkl.gz
  → Sparse format detected
  Fast path: reading arrays directly from summary
    → 13172 rows from sparse
Loading: summary.pkl.gz
  → Sparse format detected
  Fast path: reading arrays directly from summary
    → 13172 rows from sparse
Loading: summary.pkl.gz
  → Sparse format detected
  Fast path: reading arrays directly from summary
    → 13176 rows from sparse
Loading: summary.pkl.gz
  → Sparse format detected
  Fast path: reading arrays directly from summary
    → 13165 rows from sparse
Loading: summary.pkl.gz
  → Sparse format detected
  Fast path: reading arrays directly from s

,scenario,run_id,run_scope,sim_name,source_file,sim_index,sim_key,binary_index,f,Mc,D,h0,z,cgw_snr,ra,dec,psi,iota,phi0,Mtot,global_idx
0,optimistic,2026-06-05_optimistic,2026-06-05_optimistic,sim000,runs/2026-06-05_optimistic/sim000/summary.pkl.gz,0,2026-06-05_optimistic/sim000,0,3.482308e-09,2.336839e+09,912.563049,7.505936e-17,0.21468,0.074480,4.605469,-0.916992,2.257812,2.935547,2.343750,5.485993e+09,0
1,optimistic,2026-06-05_optimistic,2026-06-05_optimistic,sim000,runs/2026-06-05_optimistic/sim000/summary.pkl.gz,0,2026-06-05_optimistic/sim000,1,3.030460e-09,3.107416e+09,818.346313,1.211183e-16,0.19140,0.096819,4.121094,-0.112244,1.085938,0.545410,4.011719,7.159153e+09,1
2,optimistic,2026-06-05_optimistic,2026-06-05_optimistic,sim000,runs/2026-06-05_optimistic/sim000/summary.pkl.gz,0,2026-06-05_optimistic/sim000,2,2.684432e-09,2.729970e+09,843.065491,8.767283e-17,0.19748,0.058116,5.160156,0.192139,2.357422,2.423828,3.546875,6.767401e+09,2
3,optimistic,2026-06-05_optimistic,2026-06-05_optimistic,sim000,runs/2026-06-05_optimistic/sim000/summary.pkl.gz,0,2026-06-05_optimistic/sim000,3,3.070840e-09,1.789117e+09,925.583008,4.368369e-17,0.21792,0.039927,5.113281,0.800781,0.804199,0.200928,1.036133,4.674422e+09,3
4,optimistic,2026-06-05_optimistic,2026-06-05_optimistic,sim000,runs/2026-06-05_optimistic/sim000/summary.pkl.gz,0,2026-06-05_optimistic/sim000,4,2.291993e-09,6.252604e+09,791.749756,3.320129e-16,0.18488,0.106142,5.679688,-0.681641,2.333984,1.996094,5.898438,1.469308e+10,4



DataFrame shape: (52859506, 21)
Columns: ['scenario', 'run_id', 'run_scope', 'sim_name', 'source_file', 'sim_index', 'sim_key', 'binary_index', 'f', 'Mc', 'D', 'h0', 'z', 'cgw_snr', 'ra', 'dec', 'psi', 'iota', 'phi0', 'Mtot', 'global_idx']


In [120]:
# ============================================
# STEP 1B: CGW HELPER FUNCTIONS
# ============================================


def build_cgw_simulation_table_simple(binary_df: pd.DataFrame) -> pd.DataFrame:
    """Build CGW simulation table from the loaded binary dataframe."""
    if binary_df.empty or 'cgw_snr' not in binary_df.columns:
        return pd.DataFrame()

    df_cgw = binary_df[binary_df['cgw_snr'] > 0].copy()
    if df_cgw.empty:
        return pd.DataFrame()

    records = []
    for (scenario, sim_key), group in df_cgw.groupby(['scenario', 'sim_key'], dropna=False):
        nearest_d = group['D'].min() if 'D' in group.columns else np.nan
        loudest_idx = group['cgw_snr'].idxmax()
        loudest = group.loc[loudest_idx]

        records.append({
            'scenario': scenario,
            'run_id': loudest.get('run_id', ''),
            'run_scope': loudest.get('run_scope', ''),
            'sim_name': loudest.get('sim_name', ''),
            'source_file': str(loudest.get('source_file', '')),
            'sim_index': int(loudest.get('sim_index', -1)),
            'sim_key': sim_key,
            'nearest_D': float(nearest_d),
            'loudest_cgw_snr': float(loudest['cgw_snr']),
            'loudest_h0': float(loudest.get('h0', np.nan)),
            'loudest_Mc': float(loudest.get('Mc', np.nan)),
            'loudest_D': float(loudest.get('D', np.nan)),
            'loudest_f': float(loudest.get('f', np.nan)),
            'loudest_ra': float(loudest.get('ra', np.nan)) if pd.notna(loudest.get('ra')) else np.nan,
            'loudest_dec': float(loudest.get('dec', np.nan)) if pd.notna(loudest.get('dec')) else np.nan,
            'loudest_psi': float(loudest.get('psi', np.nan)) if pd.notna(loudest.get('psi')) else np.nan,
            'loudest_iota': float(loudest.get('iota', np.nan)) if pd.notna(loudest.get('iota')) else np.nan,
            'loudest_phi0': float(loudest.get('phi0', np.nan)) if pd.notna(loudest.get('phi0')) else np.nan,
        })

    return pd.DataFrame.from_records(records) if records else pd.DataFrame()



def build_cgw_sim_inventory_table_simple(binary_df: pd.DataFrame) -> pd.DataFrame:
    """Track whether each simulation has any nonzero CGW SNR binaries."""
    if binary_df.empty or 'cgw_snr' not in binary_df.columns:
        return pd.DataFrame()

    records = []
    for (scenario, sim_key), group in binary_df.groupby(['scenario', 'sim_key'], dropna=False):
        records.append({
            'scenario': scenario,
            'run_id': group['run_id'].iloc[0] if 'run_id' in group.columns and not group.empty else '',
            'run_scope': group['run_scope'].iloc[0] if 'run_scope' in group.columns and not group.empty else '',
            'sim_name': group['sim_name'].iloc[0] if 'sim_name' in group.columns and not group.empty else '',
            'sim_index': int(group['sim_index'].iloc[0]) if 'sim_index' in group.columns and not group.empty else -1,
            'sim_key': sim_key,
            'has_cgw': bool((group['cgw_snr'] > 0).any()),
        })

    return pd.DataFrame.from_records(records) if records else pd.DataFrame()


In [121]:
# ============================================
# STEP 2: BUILD CGW ANALYSIS TABLE
# ============================================

print("Building CGW analysis table...")

# Always reload from the summary files so Step 2 cannot use stale binary_df state.
binary_df = load_population_binary_table_robust(result_files, verbose=True)
print(f"\n  → Reloaded: {len(binary_df)} binary rows loaded from summary.pkl.gz files")
if not binary_df.empty:
    print(f"  → Columns: {list(binary_df.columns)}")
    if 'cgw_snr' in binary_df.columns:
        finite_cgw = binary_df['cgw_snr'].replace([np.inf, -np.inf], np.nan).dropna()
        print(f"  → cgw_snr present: {len(finite_cgw)} finite values, {int((binary_df['cgw_snr'] > 0).sum())} positive")
    sky_cols = [c for c in ('ra', 'dec', 'psi', 'iota', 'phi0') if c in binary_df.columns]
    print(f"  → Sky-location columns: {sky_cols if sky_cols else 'none'}")

# Build tables
cgw_sim_inventory_df = build_cgw_sim_inventory_table_simple(binary_df)
cgw_sim_df = build_cgw_simulation_table_simple(binary_df)
# Keep generic aliases for later cells that still expect the old names.
sim_df = cgw_sim_df.copy()
result_files_CGW = result_files.copy()

if not cgw_sim_df.empty:
    print(f"\n✓ CGW simulations found: {len(cgw_sim_df)}")
    print(f"  Scenarios: {sorted(cgw_sim_df['scenario'].unique().tolist())}")

    summary_cgw = (
        cgw_sim_df.groupby('scenario', as_index=False)
        .agg(
            n_sims=('sim_key', 'nunique'),
            median_cgw_snr=('loudest_cgw_snr', 'median'),
            max_cgw_snr=('loudest_cgw_snr', 'max'),
            median_nearest_D=('nearest_D', 'median'),
        )
    )
    print("\nCGW Summary by Scenario:")
    display(summary_cgw)
    print("\nFirst 10 CGW sources:")
    display(cgw_sim_df[['scenario', 'sim_index', 'loudest_cgw_snr', 'loudest_h0', 'loudest_Mc', 'loudest_f', 'loudest_ra', 'loudest_dec', 'loudest_psi', 'loudest_iota', 'loudest_phi0']].head(10))
else:
    print("✗ No CGW data found in summary.pkl.gz files (all cgw_snr ≤ 0, or the summary payload does not include cgw_snr/sky-location arrays)")

Building CGW analysis table...
Loading: summary.pkl.gz
  → Sparse format detected
  Fast path: reading arrays directly from summary
    → 13168 rows from sparse
Loading: summary.pkl.gz
  → Sparse format detected
  Fast path: reading arrays directly from summary
    → 13170 rows from sparse
Loading: summary.pkl.gz
  → Sparse format detected
  Fast path: reading arrays directly from summary
    → 13172 rows from sparse
Loading: summary.pkl.gz
  → Sparse format detected
  Fast path: reading arrays directly from summary
    → 13172 rows from sparse
Loading: summary.pkl.gz
  → Sparse format detected
  Fast path: reading arrays directly from summary
    → 13176 rows from sparse
Loading: summary.pkl.gz
  → Sparse format detected
  Fast path: reading arrays directly from summary
    → 13165 rows from sparse
Loading: summary.pkl.gz
  → Sparse format detected
  Fast path: reading arrays directly from summary
    → 13163 rows from sparse
Loading: summary.pkl.gz
  → Sparse format detected
  Fast p

,scenario,n_sims,median_cgw_snr,max_cgw_snr,median_nearest_D
0,optimistic,400,4.876094,559.732849,15.112256
1,pessimistic,400,2.520076,332.191101,1.968580
2,realistic,400,3.831325,550.058044,11.179922



First 10 CGW sources:


,scenario,sim_index,loudest_cgw_snr,loudest_h0,loudest_Mc,loudest_f,loudest_ra,loudest_dec,loudest_psi,loudest_iota,loudest_phi0
0,optimistic,0,13.384206,2.905476e-16,3.607003e+09,9.748899e-09,4.636719,0.228271,0.669434,0.498535,0.205566
1,optimistic,2,3.299688,1.600221e-15,9.054749e+09,1.466140e-08,4.078125,0.078491,1.989258,2.689453,4.722656
2,optimistic,3,15.722937,1.508231e-16,6.386545e+09,5.275981e-09,5.152344,0.391846,0.281494,3.070312,5.714844
3,optimistic,4,11.315403,3.202127e-16,2.877095e+09,7.155658e-09,5.914062,0.479980,0.590820,0.964844,0.752441
4,optimistic,5,8.719792,3.642227e-15,9.171278e+09,1.498355e-08,4.917969,0.516113,3.136719,0.084839,4.484375
5,optimistic,6,2.429806,7.534151e-17,3.981066e+09,2.176630e-09,4.523438,0.139404,0.507324,0.044617,3.015625
6,optimistic,7,13.690153,8.756860e-15,5.457064e+09,9.347203e-09,4.691406,-1.240234,0.478516,2.601562,4.550781
7,optimistic,8,8.298971,1.454141e-14,1.032554e+10,7.019472e-08,5.472656,0.736816,0.727539,0.907715,2.939453
8,optimistic,11,1.849673,1.909012e-15,7.572372e+09,2.969458e-08,5.644531,0.537598,2.511719,0.083374,2.011719
9,optimistic,12,3.741971,1.801556e-15,1.044269e+10,1.420799e-08,5.500000,0.346191,2.259766,0.430176,3.851562


In [122]:
# ============================================
# STEP 3: PLOT CGW ANALYSIS RESULTS  
# ============================================

if cgw_sim_df.empty:
    print("Skipping plots: No CGW data available")
else:
    from pathlib import Path
    Path('figures').mkdir(exist_ok=True)
    
    print("\nGenerating plots...")
    
    # Helper: extract values by scenario
    def _extract(df, col):
        return {
            s: df.loc[df['scenario'] == s, col].to_numpy()
            for s in SCENARIOS
        }
    
    # 1. Nearest distance
    fig, ax = plt.subplots(figsize=(9, 4.8))
    for scenario in SCENARIOS:
        sub = cgw_sim_df[cgw_sim_df['scenario'] == scenario]['nearest_D'].dropna()
        if len(sub) > 0:
            ax.hist(sub, bins=30, alpha=0.5, label=scenario, density=False)
    ax.set_xscale('log')
    ax.set_xlabel('Nearest SMBHB comoving distance [Mpc]')
    ax.set_ylabel('Count')
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig("figures/cgw_nearest_SMBHB.pdf", dpi=300, bbox_inches='tight')
    print("  ✓ figures/cgw_nearest_SMBHB.pdf")
    
    # 2. CGW SNR distribution
    fig, ax = plt.subplots(figsize=(9, 4.8))
    for scenario in SCENARIOS:
        sub = cgw_sim_df[cgw_sim_df['scenario'] == scenario]['loudest_cgw_snr'].dropna()
        if len(sub) > 0:
            ax.hist(sub, bins=30, alpha=0.5, label=scenario, density=False)
    ax.set_xscale('log')
    ax.set_xlabel('Loudest binary CGW SNR')
    ax.set_ylabel('Count')
    ax.axvline(5.0, color='red', linestyle='--', linewidth=2, label='SNR=5 threshold')
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig("figures/cgw_snr_distribution.pdf", dpi=300, bbox_inches='tight')
    print("  ✓ figures/cgw_snr_distribution.pdf")
    
    # 3. Loudest binary parameters
    fig, axes = plt.subplots(2, 2, figsize=(13, 8))
    plot_cfg = [
        ('loudest_h0', '$h_0$', True),
        ('loudest_Mc', r'$\mathcal{M}_c$ [$M_\odot$]', True),
        ('loudest_D', r'$D_{\rm{comov}}$ [Mpc]', True),
        ('loudest_f', '$f$ [Hz]', True),
    ]
    for ax, (col, label, logx) in zip(axes.ravel(), plot_cfg):
        for scenario in SCENARIOS:
            sub = cgw_sim_df[cgw_sim_df['scenario'] == scenario][col].dropna()
            if len(sub) > 0:
                ax.hist(sub, bins=20, alpha=0.5, label=scenario, density=False)
        ax.set_xlabel(label)
        ax.set_ylabel('Count')
        if logx:
            ax.set_xscale('log')
        ax.legend(frameon=False)
    fig.suptitle('Loudest Binary (max CGW SNR) Properties', fontsize=14, y=1.01)
    fig.tight_layout()
    fig.savefig("figures/cgw_loudest_binary_parameters.pdf", dpi=300, bbox_inches='tight')
    print("  ✓ figures/cgw_loudest_binary_parameters.pdf")
    
    print("\n✓ All plots saved to figures/")
    print("  - cgw_nearest_SMBHB.pdf")
    print("  - cgw_snr_distribution.pdf")
    print("  - cgw_loudest_binary_parameters.pdf")


Generating plots...
  ✓ figures/cgw_nearest_SMBHB.pdf
  ✓ figures/cgw_snr_distribution.pdf
  ✓ figures/cgw_loudest_binary_parameters.pdf

✓ All plots saved to figures/
  - cgw_nearest_SMBHB.pdf
  - cgw_snr_distribution.pdf
  - cgw_loudest_binary_parameters.pdf


In [123]:
import sys, types
import numpy as np

# Create compatibility modules that older pickles expect.
if 'numpy._core' not in sys.modules:
    mod_core = types.ModuleType('numpy._core')
    sys.modules['numpy._core'] = mod_core

# Point numpy._core.multiarray to the real implementation
sys.modules['numpy._core.multiarray'] = np.core.multiarray
setattr(sys.modules['numpy._core'], 'multiarray', np.core.multiarray)

# Also ensure attribute aliasing some pickles use
if not hasattr(np.core.multiarray, '_reconstruct') and hasattr(np.core.multiarray, 'reconstruct'):
    setattr(np.core.multiarray, '_reconstruct', getattr(np.core.multiarray, 'reconstruct'))

In [14]:
import gzip, pickle
p = Path('runs/2026-05-20_optimistic/sim000/summary.pkl.gz')
payload = pickle.load(gzip.open(p,'rb'))
print(payload.keys())
print(sorted(payload.get('arrays', {}).keys()))

FileNotFoundError: [Errno 2] No such file or directory: 'runs/2026-05-20_optimistic/sim000/summary.pkl.gz'

In [124]:
def hist_by_scenario(data_by_scenario: dict[str, np.ndarray], bins, xlabel: str, title: str, save = False, savename = None, logx: bool = False):
    fig, ax = plt.subplots(figsize=(9, 5))
    scenario_labels = (r"$p(M) \propto M^{-1.21}$", 
         r"$p(M) \propto M^{-1.21}\exp(-\frac{M}{10^{10}\;\mathrm{M}_{\odot}})$", 
         r"$p(M) \propto M^{-1.21}\exp(-\frac{M}{10^{9}\;\mathrm{M}_{\odot}})$")
    for (scenario, label) in zip(SCENARIOS, scenario_labels):
        vals = data_by_scenario.get(scenario)
        if vals is None or len(vals) == 0:
            continue
        ax.hist(vals, bins=bins, alpha=0.45, density=False, label=label)

    if logx:
        ax.set_xscale('log')
    ax.set_xlabel(xlabel)
    ax.set_ylabel('PDF')
    # ax.set_title(title)
    ax.legend(frameon=False)
    plt.tight_layout()
    if save:
        plt.savefig(savename)
    return fig, ax



nearest_by_scenario = {
    s: sim_df.loc[sim_df['scenario'] == s, 'nearest_D'].dropna().to_numpy()
    for s in SCENARIOS
}

hist_by_scenario(
    nearest_by_scenario,
    bins=list(np.logspace(0, 4, 50)),
    xlabel=r'$D_{\rm{comov}}$',
    title='Nearest Binary Per Simulation by Scenario',
    logx=True,
    save = True,
    savename="data/2026-04-01/nearest_SMBHB.pdf"
)


FileNotFoundError: [Errno 2] No such file or directory: 'data/2026-04-01/nearest_SMBHB.pdf'

In [ ]:
loudest_by_scenario = {
    s: sim_df.loc[sim_df['scenario'] == s, 'loudest_h0'].dropna().to_numpy()
    for s in SCENARIOS
}

_ = hist_by_scenario(
    loudest_by_scenario,
    bins=list(np.logspace(-15, -12, 50)),
    xlabel='Maximum h0 per simulation',
    title='Loudest Binary Proxy Per Simulation by Scenario',
    logx=True,
    save = True,
    savename="data/2026-04-01/biggest_h0_SMBHB.pdf"
)

In [ ]:
def add_distribution_weights(df: pd.DataFrame, mode: str = 'all_binaries') -> pd.DataFrame:
    out = df.copy()
    if mode == 'all_binaries':
        out['weight'] = 1.0
        return out

    if mode == 'per_sim_equal':
        counts = out.groupby('sim_index')['binary_index'].transform('count').astype(float)
        out['weight'] = 1.0 / counts
        return out

    raise ValueError("mode must be 'all_binaries' or 'per_sim_equal'")


weight_mode = 'per_sim_equal'  # change to 'per_sim_equal' if desired
dist_df = add_distribution_weights(binary_df, mode=weight_mode)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
plot_specs = [
    ('Mc', 'Chirp mass Mc', True),
    ('f', 'GW frequency f [Hz]', True),
    ('D', 'Comoving distance D', True),
]
bins_mass = list(np.logspace(6, 11, 50))
bins_f = list(np.logspace(-9, -6, 50))
bins_dist = list(np.logspace(1, 5, 50))
bins = [bins_mass, bins_f, bins_dist]


for ax, (col, xlabel, logx), _bin in zip(axes, plot_specs, bins):
    for scenario in SCENARIOS:
        sub = dist_df[(dist_df['scenario'] == scenario) & np.isfinite(dist_df[col])]
        if sub.empty:
            continue
        ax.hist(
            sub[col].to_numpy(),
            bins=_bin,
            weights=sub['weight'].to_numpy(),
            density=True,
            alpha=0.35,
            label=scenario,
        )
    if logx:
        ax.set_xscale('log')
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Density')
    ax.set_xlim(min(_bin), max(_bin))

axes[0].legend(frameon=False)
fig.suptitle(f'Population Distributions by Scenario (weight_mode={weight_mode})')
plt.tight_layout()
plt.savefig("data/2026-04-01/distributions.pdf")

In [ ]:
summary = (
    sim_df.groupby('scenario', as_index=False)
    .agg(
        n_sims=('sim_index', 'nunique'),
        nearest_D_median=('nearest_D', 'median'),
        nearest_D_p10=('nearest_D', lambda x: np.nanpercentile(x, 10)),
        nearest_D_p90=('nearest_D', lambda x: np.nanpercentile(x, 90)),
        loudest_h0_median=('loudest_h0', 'median'),
        loudest_h0_p10=('loudest_h0', lambda x: np.nanpercentile(x, 10)),
        loudest_h0_p90=('loudest_h0', lambda x: np.nanpercentile(x, 90)),
    )
)

summary

## Notes on Combining Across Simulations

Default in this notebook:
- nearest/loudest metrics are per simulation (one point per sim), which is robust for scenario comparison
- distribution plots use all binaries by default (`weight_mode='all_binaries'`)

Alternative:
- set `weight_mode='per_sim_equal'` to avoid simulations with more binaries dominating parameter distributions

If you later add SNR contribution diagnostics, keep the same grouping key (`sim_id`) so all summaries stay aligned.

## Storage Recommendation

For your workflow, use one primary format by default: **compressed pickle (`.pkl.gz`)**.

Why this is the best default here:
- full-fidelity recovery of `PopulationArrays` objects
- much smaller than plain `.pkl`
- still straightforward to load in Python

When to additionally save `.npz`:
- only if disk size or data transfer is a bottleneck
- only for plotting/array workflows (not full object reconstruction)

In [70]:
# Minimal examples
from pathlib import Path

from utils import load_results_pickle_gz, save_results_compact_npz

# 1) Preferred: load full-fidelity compressed pickle
pkl_gz_path = Path('data/2026-03-31/realistic/consistent_population_realistic_targetSNR4.0_sims10.pkl.gz')
if pkl_gz_path.exists():
    results_obj = load_results_pickle_gz(pkl_gz_path)
    print('Loaded:', pkl_gz_path)
    print('n sims:', len(results_obj.get('populations', [])))
else:
    print('Example file not found:', pkl_gz_path)

# 2) Optional: create compact NPZ for plotting-only pipelines
if 'results_obj' in locals():
    npz_path = pkl_gz_path.with_suffix('').with_suffix('.npz')
    save_results_compact_npz(results_obj, npz_path)
    print('Saved compact npz:', npz_path)

Example file not found: data/2026-03-31/realistic/consistent_population_realistic_targetSNR4.0_sims10.pkl.gz


## Continuous GW (CGW) SNR Diagnostics

This section reads CGW diagnostics saved in the same `compact_results` structure used in `main.py`, i.e. per simulation:
- `compact_results['populations'][i]['population']`
- `compact_results['populations'][i]['cgw_analysis']['top_sources']`

It builds per-simulation metrics across scenarios and then plots:
- nearest SMBHB distance per simulation,
- loudest-by-CGW binary (`max SNR`) per simulation: `h0`, `Mc`, `D`, and `f`,
- two sky maps for loudest-by-CGW binaries: plain dots and SNR-colored.


In [125]:
len(result_files_CGW)

1200

In [127]:
# Simplified CGW analysis: Build table directly from binary_df (works for summary.pkl.gz payloads)

def build_cgw_simulation_table_simple(binary_df: pd.DataFrame) -> pd.DataFrame:
    """Build CGW simulation table from the loaded binary dataframe."""
    if binary_df.empty or 'cgw_snr' not in binary_df.columns:
        return pd.DataFrame()

    df_cgw = binary_df[binary_df['cgw_snr'] > 0].copy()
    if df_cgw.empty:
        return pd.DataFrame()

    records = []
    for (scenario, run_id, sim_index), group in df_cgw.groupby(['scenario', 'run_id', 'sim_index'], dropna=False):
        nearest_d = group['D'].min() if 'D' in group.columns else np.nan
        loudest_idx = group['cgw_snr'].idxmax()
        loudest = group.loc[loudest_idx]

        records.append({
            'scenario': scenario,
            'run_id': run_id,
            'source_file': str(loudest.get('source_file', '')),
            'sim_index': int(sim_index),
            'sim_key': f"{scenario}:{run_id}:{sim_index}",
            'nearest_D': float(nearest_d),
            'loudest_cgw_snr': float(loudest['cgw_snr']),
            'loudest_h0': float(loudest.get('h0', np.nan)),
            'loudest_Mc': float(loudest.get('Mc', np.nan)),
            'loudest_D': float(loudest.get('D', np.nan)),
            'loudest_f': float(loudest.get('f', np.nan)),
            'loudest_ra': float(loudest.get('ra', np.nan)) if pd.notna(loudest.get('ra')) else np.nan,
            'loudest_dec': float(loudest.get('dec', np.nan)) if pd.notna(loudest.get('dec')) else np.nan,
            'loudest_psi': float(loudest.get('psi', np.nan)) if pd.notna(loudest.get('psi')) else np.nan,
            'loudest_iota': float(loudest.get('iota', np.nan)) if pd.notna(loudest.get('iota')) else np.nan,
            'loudest_phi0': float(loudest.get('phi0', np.nan)) if pd.notna(loudest.get('phi0')) else np.nan,
        })

    return pd.DataFrame.from_records(records) if records else pd.DataFrame()


def build_cgw_sim_inventory_table_simple(binary_df: pd.DataFrame) -> pd.DataFrame:
    """Track whether each simulation has any nonzero CGW SNR binaries."""
    if binary_df.empty or 'cgw_snr' not in binary_df.columns:
        return pd.DataFrame()

    records = []
    for (scenario, run_id, sim_index), group in binary_df.groupby(['scenario', 'run_id', 'sim_index'], dropna=False):
        records.append({
            'scenario': scenario,
            'run_id': run_id,
            'sim_index': int(sim_index),
            'sim_key': f"{scenario}:{run_id}:{sim_index}",
            'has_cgw': bool((group['cgw_snr'] > 0).any()),
        })

    return pd.DataFrame.from_records(records) if records else pd.DataFrame()


# Build tables from summary.pkl.gz-backed rows
print('Reloading binary_df from summary.pkl.gz files...')
binary_df = load_population_binary_table_robust(result_files, verbose=False)
print(f'  → Reloaded {len(binary_df)} binary rows')
if not binary_df.empty:
    sky_cols = [c for c in ('ra', 'dec', 'psi', 'iota', 'phi0') if c in binary_df.columns]
    print(f"  → Sky-location columns loaded: {sky_cols if sky_cols else 'none'}")

cgw_sim_inventory_df = build_cgw_sim_inventory_table_simple(binary_df)
cgw_sim_df = build_cgw_simulation_table_simple(binary_df)
# Keep generic aliases for later cells that still expect the old names.
sim_df = cgw_sim_df.copy()
result_files_CGW = result_files.copy()

if not cgw_sim_df.empty:
    print(f"\n✓ CGW simulations found: {len(cgw_sim_df)}")
    print(f"  Scenarios: {sorted(cgw_sim_df['scenario'].unique().tolist())}")
    
    summary_cgw = (
        cgw_sim_df.groupby('scenario', as_index=False)
        .agg(
            n_sims=('sim_key', 'nunique'),
            median_cgw_snr=('loudest_cgw_snr', 'median'),
            max_cgw_snr=('loudest_cgw_snr', 'max'),
            median_nearest_D=('nearest_D', 'median'),
        )
    )
    print("\nCGW Summary by Scenario:")
    display(summary_cgw)
    print("\nFirst 10 CGW sources:")
    display(cgw_sim_df[['scenario', 'sim_index', 'loudest_cgw_snr', 'loudest_h0', 'loudest_Mc', 'loudest_f', 'loudest_ra', 'loudest_dec']].head(10))
else:
    print("✗ No CGW data found in summary.pkl.gz files (all cgw_snr ≤ 0, or the summary payload does not include cgw_snr/sky-location arrays)")

Reloading binary_df from summary.pkl.gz files...
  → Reloaded 52859506 binary rows
  → Sky-location columns loaded: ['ra', 'dec', 'psi', 'iota', 'phi0']

✓ CGW simulations found: 1200
  Scenarios: ['optimistic', 'pessimistic', 'realistic']

CGW Summary by Scenario:


,scenario,n_sims,median_cgw_snr,max_cgw_snr,median_nearest_D
0,optimistic,400,4.876094,559.732849,15.112256
1,pessimistic,400,2.520076,332.191101,1.968580
2,realistic,400,3.831325,550.058044,11.179922



First 10 CGW sources:


,scenario,sim_index,loudest_cgw_snr,loudest_h0,loudest_Mc,loudest_f,loudest_ra,loudest_dec
0,optimistic,0,13.384206,2.905476e-16,3.607003e+09,9.748899e-09,4.636719,0.228271
1,optimistic,2,3.299688,1.600221e-15,9.054749e+09,1.466140e-08,4.078125,0.078491
2,optimistic,3,15.722937,1.508231e-16,6.386545e+09,5.275981e-09,5.152344,0.391846
3,optimistic,4,11.315403,3.202127e-16,2.877095e+09,7.155658e-09,5.914062,0.479980
4,optimistic,5,8.719792,3.642227e-15,9.171278e+09,1.498355e-08,4.917969,0.516113
5,optimistic,6,2.429806,7.534151e-17,3.981066e+09,2.176630e-09,4.523438,0.139404
6,optimistic,7,13.690153,8.756860e-15,5.457064e+09,9.347203e-09,4.691406,-1.240234
7,optimistic,8,8.298971,1.454141e-14,1.032554e+10,7.019472e-08,5.472656,0.736816
8,optimistic,11,1.849673,1.909012e-15,7.572372e+09,2.969458e-08,5.644531,0.537598
9,optimistic,12,3.741971,1.801556e-15,1.044269e+10,1.420799e-08,5.500000,0.346191


In [166]:
# Okabe-Ito colourblind-safe palette
scenario_labels = (r"$m_{\mathrm{char}} = 10^{9.3}\;\mathrm{M}_{\odot}$", 
         r"$m_{\mathrm{char}} = 10^{9.0}\;\mathrm{M}_{\odot}$", 
         r"$m_{\mathrm{char}} = 10^{8.7}\;\mathrm{M}_{\odot}$")

scenario_styles = {
    r"$m_{\mathrm{char}} = 10^{9.3}\;\mathrm{M}_{\odot}$" :  {'color': '#0072B2', 'linestyle': '-',  'linewidth': 2.2},
    r"$m_{\mathrm{char}} = 10^{9.0}\;\mathrm{M}_{\odot}$":   {'color': '#E69F00', 'linestyle': '--', 'linewidth': 2.2},
    r"$m_{\mathrm{char}} = 10^{8.7}\;\mathrm{M}_{\odot}$": {'color': '#D55E00', 'linestyle': ':',  'linewidth': 2.5},
}
scenario_styles = {
    'optimistic':  {'color': 'lime', 'linestyle': '-',  'linewidth': 2.2},
    'realistic':   {'color': 'magenta', 'linestyle': '--', 'linewidth': 2.2},
    'pessimistic': {'color': 'navy', 'linestyle': ':',  'linewidth': 2.5},
}

In [167]:
def _scenario_histogram(
    ax,
    scenario_vals: dict,
    xlabel: str,
    logx: bool = False,
    show_legend: bool = True,
    vline: float = None,
    loc: str = 'best',
    bbox_anch_coords=None,
    show_ci_band: bool = False,
    ci_alpha: float = 0.08,
):
    all_vals = np.concatenate([v for v in scenario_vals.values() if len(v) > 0])
    all_vals = all_vals[np.isfinite(all_vals)]
    if len(all_vals) == 0:
        return
    if logx:
        bins = np.logspace(np.log10(all_vals.min()), np.log10(all_vals.max()), 25)
    else:
        bins = np.linspace(all_vals.min(), all_vals.max(), 25)
    if show_ci_band:
        print(f"\n{xlabel}:")
    for s in scenario_order:
        vals = scenario_vals.get(s, np.array([]))
        vals = vals[np.isfinite(vals)]
        if len(vals) == 0:
            continue
        style = scenario_styles.get(s, {'color': '#888888', 'linestyle': '-', 'linewidth': 1.5})

        counts, edges = np.histogram(vals, bins=bins)
        ax.hist(
            vals,
            bins=bins,
            histtype='step',
            color=style['color'],
            linestyle=style['linestyle'],
            linewidth=style['linewidth'],
            label=s,
            density=False,
            zorder=2,
        )

        if show_ci_band:
            lo, hi = np.nanpercentile(vals, [2.5, 97.5])
            median = np.nanmedian(vals)
            print(f"  {s}: median = {median:.3g}, 95% CI = [{lo:.3g}, {hi:.3g}]")
            counts, edges = np.histogram(vals, bins=bins)
            lo_idx = np.clip(np.searchsorted(edges, lo, side='right') - 1, 0, len(counts) - 1)
            hi_idx = np.clip(np.searchsorted(edges, hi, side='right') - 1, 0, len(counts) - 1)
            for i in range(lo_idx, hi_idx + 1):
                ax.bar(
                    edges[i],
                    counts[i],
                    width=edges[i + 1] - edges[i],
                    align='edge',
                    color=style['color'],
                    alpha=0.12,
                    zorder=1,
                    linewidth=0,
                )

    ax.set_xlabel(xlabel, fontsize=11)
    ax.set_ylabel('Number of simulations', fontsize=11)
    ax.tick_params(labelsize=8)
    ax.spines['top'].set_visible(True)
    ax.spines['right'].set_visible(True)
    if logx:
        ax.set_xscale('log')
    if vline is not None:
        ax.axvline(vline, color='black', linestyle='--', linewidth=1.5, zorder=3)
    if show_legend:
        handles = [
            mlines.Line2D(
                [], [],
                color=scenario_styles[s]['color'],
                linestyle=scenario_styles[s]['linestyle'],
                linewidth=scenario_styles[s]['linewidth'],
                label=scenario_label_map[s],
            )
            for s in scenario_order if s in scenario_styles
        ]
        legend_kwargs = dict(frameon=False, fontsize=8, loc=loc)
        if bbox_anch_coords is not None:
            legend_kwargs['bbox_to_anchor'] = bbox_anch_coords
        ax.legend(handles=handles, **legend_kwargs)


_save_hist_figure(
    'nearest_SMBHB.pdf',
    _extract(cgw_sim_df, 'nearest_D'),
    r'$D_{\mathrm{comov}}$ [Mpc]',
    logx=True,
    figsize=(3.5, 2.8),
    legend_loc='upper right',
    show_legend=True,
    show_ci_band=True
)
_save_hist_figure(
    'loudest_h0_CGW.pdf',
    _extract(cgw_sim_df, 'loudest_h0'),
    r'$h_0$',
    logx=True,
    figsize=(3.5, 2.8),
    legend_loc='upper right',
    show_legend=False,
    show_ci_band=True,
)
_save_hist_figure(
    'loudest_Mc_CGW.pdf',
    _extract(cgw_sim_df, 'loudest_Mc'),
    r'$\mathcal{M}_c$ [$M_\odot$]',
    logx=True,
    figsize=(3.5, 2.8),
    legend_loc='upper right',
    show_legend=False,
    show_ci_band=True,
)
_save_hist_figure(
    'loudest_D_CGW.pdf',
    _extract(cgw_sim_df, 'loudest_D'),
    r'$D_{\mathrm{comov}}$ [Mpc]',
    logx=True,
    figsize=(3.5, 2.8),
    legend_loc='upper left',
    show_legend=True,
    show_ci_band=True,
)
_save_hist_figure(
    'loudest_f_CGW.pdf',
    _extract(cgw_sim_df, 'loudest_f'),
    r'$f$ [Hz]',
    logx=True,
    figsize=(3.5, 2.8),
    legend_loc='upper right',
    show_legend=False,
    show_ci_band=True,
)
_save_hist_figure(
    'loudest_cgw_snr.pdf',
    _extract(cgw_sim_df, 'loudest_cgw_snr'),
    r'$(\mathrm{S/N})_s$',
    logx=True,
    vline=5.94,
    figsize=(3.5, 2.8),
    legend_loc='upper right',
    show_legend=True,
)


$D_{\mathrm{comov}}$ [Mpc]:
  optimistic: median = 15.1, 95% CI = [1.79, 79.3]
  realistic: median = 11.2, 95% CI = [1.07, 45.9]
  pessimistic: median = 1.97, 95% CI = [0.358, 10.6]
  saved figures/nearest_SMBHB.pdf

$h_0$:
  optimistic: median = 1.58e-15, 95% CI = [2.13e-17, 1.19e-14]
  realistic: median = 3.5e-16, 95% CI = [1.95e-17, 6.98e-15]
  pessimistic: median = 1.21e-16, 95% CI = [1.07e-17, 4.24e-15]
  saved figures/loudest_h0_CGW.pdf

$\mathcal{M}_c$ [$M_\odot$]:
  optimistic: median = 6.65e+09, 95% CI = [1.62e+09, 1.5e+10]
  realistic: median = 2.66e+09, 95% CI = [1.04e+09, 7.31e+09]
  pessimistic: median = 1.63e+09, 95% CI = [5.81e+08, 4.03e+09]
  saved figures/loudest_Mc_CGW.pdf

$D_{\mathrm{comov}}$ [Mpc]:
  optimistic: median = 415, 95% CI = [15.5, 2.8e+03]
  realistic: median = 497, 95% CI = [10.4, 1.12e+04]
  pessimistic: median = 196, 95% CI = [3.57, 2.25e+03]
  saved figures/loudest_D_CGW.pdf

$f$ [Hz]:
  optimistic: median = 8.61e-09, 95% CI = [2.26e-09, 3.01e-08]
 

In [180]:
fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.2), sharey=True)

for ax, s in zip(axes, scenario_order):
    sub   = cgw_sim_df[cgw_sim_df['scenario'] == s]
    style = scenario_styles.get(s, {'color': '#888888'})

    sc = ax.scatter(
        sub['loudest_Mc'],
        sub['loudest_h0'],
        c=sub['loudest_D'],
        cmap='viridis_r',
        s=4,
        alpha=0.5,
        rasterized=True,
    )
    cb = fig.colorbar(sc, ax=ax, pad=0.02)
    cb.set_label(r'$D_{\mathrm{comov}}$ [Mpc]', fontsize=8)
    cb.ax.tick_params(labelsize=7)

    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel(r'$\mathcal{M}_c\ [M_\odot]$', fontsize=9)
    ax.set_title(scenario_label_map.get(s, s), fontsize=8)
    ax.tick_params(labelsize=8)
    ax.spines['top'].set_visible(True)
    ax.spines['right'].set_visible(True)

axes[0].set_ylabel(r'$h_0$', fontsize=9)

fig.tight_layout(pad=0.4)
fig.savefig('figures/loudest_h0_vs_Mc_scatter.pdf', dpi=300, bbox_inches='tight')
plt.show(); plt.close(fig)
print('  saved loudest_h0_vs_Mc_scatter.pdf')

# also plot h0 vs frequency to check the other axis
fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.2), sharey=True)

for ax, s in zip(axes, scenario_order):
    sub   = cgw_sim_df[cgw_sim_df['scenario'] == s]
    style = scenario_styles.get(s, {'color': '#888888'})

    sc = ax.scatter(
        sub['loudest_f'],
        sub['loudest_h0'],
        c=sub['loudest_Mc'],
        cmap='plasma',
        norm=plt.matplotlib.colors.LogNorm(),
        s=4,
        alpha=0.5,
        rasterized=True,
    )
    cb = fig.colorbar(sc, ax=ax, pad=0.02)
    cb.set_label(r'$\mathcal{M}_c\ [M_\odot]$', fontsize=8)
    cb.ax.tick_params(labelsize=7)

    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel(r'$f$ [Hz]', fontsize=9)
    ax.set_title(scenario_label_map.get(s, s), fontsize=8)
    ax.tick_params(labelsize=8)
    ax.spines['top'].set_visible(True)
    ax.spines['right'].set_visible(True)

axes[0].set_ylabel(r'$h_0$', fontsize=9)

fig.tight_layout(pad=0.4)
fig.savefig('figures/loudest_h0_vs_f_scatter.pdf', dpi=300, bbox_inches='tight')
plt.show(); plt.close(fig)
print('  saved loudest_h0_vs_f_scatter.pdf')

  saved loudest_h0_vs_Mc_scatter.pdf
  saved loudest_h0_vs_f_scatter.pdf


In [176]:
# Sky maps for loudest-by-CGW binaries per simulation, with pulsar positions overlaid.

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patheffects import withStroke

def _wrap_ra(ra_rad):
    ra_shifted = (ra_rad - np.pi) % (2 * np.pi)
    ra_shifted = np.where(ra_shifted > np.pi, ra_shifted - 2 * np.pi, ra_shifted)
    return -ra_shifted

def _galactic_plane():
    ra_ngp  = np.deg2rad(192.859508)
    dec_ngp = np.deg2rad(27.128336)
    l_ncp   = np.deg2rad(122.932)
    l = np.linspace(0, 2 * np.pi, 1000)
    b = np.zeros(1000)
    sin_dec = (np.sin(b) * np.sin(dec_ngp)
               + np.cos(b) * np.cos(dec_ngp) * np.sin(l_ncp - l))
    dec = np.arcsin(np.clip(sin_dec, -1, 1))
    cos_ra_minus_rangp = (np.cos(b) * np.cos(l_ncp - l)) / np.cos(dec)
    sin_ra_minus_rangp = (np.sin(b) * np.cos(dec_ngp)
                          - np.cos(b) * np.sin(dec_ngp) * np.sin(l_ncp - l)) / np.cos(dec)
    ra = (np.arctan2(sin_ra_minus_rangp, cos_ra_minus_rangp) + ra_ngp) % (2 * np.pi)
    ra  = _wrap_ra(ra)
    order = np.argsort(ra)
    ra, dec = ra[order], dec[order]
    gaps = np.where(np.abs(np.diff(ra)) > np.pi / 2)[0] + 1
    ra  = np.insert(ra.astype(float),  gaps, np.nan)
    dec = np.insert(dec.astype(float), gaps, np.nan)
    return ra, dec

if cgw_sim_df.empty:
    raise RuntimeError('No CGW simulation table available to plot sky maps.')

sky_df = cgw_sim_df[
    np.isfinite(cgw_sim_df['loudest_ra']) & np.isfinite(cgw_sim_df['loudest_dec'])
].copy()
if sky_df.empty:
    raise RuntimeError('No valid RA/Dec values for loudest-by-CGW sources.')

data       = np.load('data/pulsar_sky_locations.npz')
pulsar_ra  = _wrap_ra(data['ras'])
pulsar_dec = data['decs']
gal_ra, gal_dec = _galactic_plane()

# white stroke used on all axis labels so they read over any background
_STROKE = [withStroke(linewidth=2.5, foreground='white')]

def _aitoff_right_rim_axes_x(dec_rad, ax, fig):
    """
    For a given declination, find the axes-fraction x coordinate of the
    right rim of the Aitoff ellipse, so dec labels can hug the curved border.
    The Aitoff projection maps (ra=+pi, dec) to display coords; we convert
    those to axes fraction.
    """
    # In our _wrap_ra convention the right rim is at ra_data = -pi
    # Project that point through the Aitoff transform
    import numpy as np
    # Use a point just inside the rim to avoid projection singularities
    ra_rim = -np.pi * 0.999
    # Aitoff formula for x at the rim given dec
    z     = np.sqrt(1 + np.cos(dec_rad) * np.cos(ra_rim / 2))
    x_display = 2 * np.cos(dec_rad) * np.sin(ra_rim / 2) / z
    y_display = np.sin(dec_rad) / z
    # Convert from display (-2..2, -1..1 for Aitoff) to axes fraction
    # matplotlib's Aitoff axes maps x in [-2,2] -> axes [0,1] and y [-1,1] -> [0,1]
    x_axes = (x_display / 2 + 1) / 2   # maps [-2,2] -> [0,1]  (but flip: right rim is negative ra)
    # right rim with negative ra_rim gives negative x_display, so we want the
    # positive counterpart — the rim is symmetric, just take abs
    x_axes = (abs(x_display) / 2 + 1) / 2  # right rim is positive x side... 
    # Actually recompute with ra_rim positive for the right physical rim
    ra_rim =  np.pi * 0.999
    z      = np.sqrt(1 + np.cos(dec_rad) * np.cos(ra_rim / 2))
    x_disp = 2 * np.cos(dec_rad) * np.sin(ra_rim / 2) / z
    x_axes = (x_disp / (2 * np.sqrt(2)) + 0.5)
    return x_axes

def _style_skyax(ax, fig):
    ax.set_xticklabels([])
    ax.set_yticklabels([])
    ax.grid(True, alpha=0.25, linestyle='--', linewidth=0.5)

    # ── RA labels: through the centre along dec = 0 ──────────────────────
    ra_hours = [2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22]
    for h in ra_hours:
        rad = _wrap_ra(np.deg2rad(h * 15))
        ax.annotate(
            f'{h}h',
            xy=(rad, 0),
            xycoords='data',
            ha='center', va='center',
            fontsize=7, fontweight='bold', color='dimgrey',
            path_effects=_STROKE,
        )

    # ── Dec labels: use matplotlib's own transform to find the rim exactly ─
    # We draw the figure first to populate the transform, then place labels
    # at the display coords of (ra ~ +pi rim, dec) converted back to axes fraction.
    fig.canvas.draw()                          # forces transform to be computed

    data_to_axes = ax.transData + ax.transAxes.inverted()

    for deg in range(-75, 76, 15):             # skip ±90, they're a single point
        dec_rad = np.deg2rad(deg)
        # use ra just inside the right rim in data coords
        ra_rim  = _wrap_ra(np.deg2rad(359.9))  # rightmost data coord

        # transform data -> axes fraction
        x_ax, y_ax = data_to_axes.transform((ra_rim, dec_rad))

        ax.annotate(
            f'{deg}°',
            xy=(x_ax + 0.012, y_ax),           # small nudge outside border
            xycoords='axes fraction',
            ha='left', va='center',
            fontsize=6.5, fontweight='bold', color='dimgrey',
            path_effects=_STROKE,
            annotation_clip=False,
        )

# ── ApJ two-column figure width = 3.5 in ─────────────────────────────────────
FIG_W, FIG_H = 3.5, 2.8

# ── Sky map 1: loudest binaries colored by scenario ──────────────────────────
fig = plt.figure(figsize=(FIG_W, FIG_H))
ax  = fig.add_subplot(111, projection='aitoff')

for scenario in scenario_order:
    sub = sky_df[sky_df['scenario'] == scenario]
    if sub.empty:
        continue
    style = scenario_styles.get(scenario, {'color': '#888888'})
    ax.scatter(
        _wrap_ra(sub['loudest_ra'].to_numpy()),
        sub['loudest_dec'].to_numpy(),
        s=8, alpha=0.65,
        color=style['color'],
        edgecolors='none',
        label=scenario_label_map.get(scenario, scenario),
        zorder=3,
    )

if pulsar_ra.size > 0:
    ax.scatter(
        pulsar_ra, pulsar_dec,
        s=20, marker='*',
        color='white',
        edgecolors='black',
        linewidths=0.4,
        alpha=0.95,
        label='Pulsars',
        zorder=4,
    )

_style_skyax(ax, fig)
ax.legend(
    loc='lower center', frameon=False, fontsize=7,
    bbox_to_anchor=(0.5, -0.32), ncol=2,   # was -0.22
)
fig.tight_layout()
fig.savefig("figures/SNR_loudest_binary_CGW.pdf", dpi=300, bbox_inches='tight')
plt.show()

# ── Sky map 2: loudest binaries colored by CGW SNR ───────────────────────────
fig = plt.figure(figsize=(FIG_W + 0.6, FIG_H))   # slight extra width for colorbar
ax  = fig.add_subplot(111, projection='aitoff')

sc = ax.scatter(
    _wrap_ra(sky_df['loudest_ra'].to_numpy()),
    sky_df['loudest_dec'].to_numpy(),
    c=sky_df['loudest_cgw_snr'].to_numpy(),
    cmap='plasma',
    s=8, alpha=0.8,
    edgecolors='none',
    zorder=3,
)

if pulsar_ra.size > 0:
    ax.scatter(
        pulsar_ra, pulsar_dec,
        s=20, marker='*',
        color='white',
        edgecolors='black',
        linewidths=0.4,
        alpha=0.95,
        label='Pulsars',
        zorder=4,
    )

cbar = plt.colorbar(sc, ax=ax, pad=0.05, shrink=0.7, orientation='vertical')
cbar.set_label('CW SNR', fontsize=8)
cbar.ax.tick_params(labelsize=7)

_style_skyax(ax, fig)
if pulsar_ra.size > 0:
    ax.legend(
        loc='lower center', frameon=False, fontsize=7,
        bbox_to_anchor=(0.5, -0.22), ncol=2,
    )
fig.tight_layout()
fig.savefig("figures/coloured_SNR_loudest_binary_CGW.pdf", dpi=300, bbox_inches='tight')
plt.show()

## Continuous GW (CGW) SNR Diagnostics with Threshold

This section mirrors the CGW diagnostics above, but only keeps the loudest CGW source in each simulation if its CGW SNR is at or above a configurable threshold. Change `CGW_SNR_THRESHOLD` in the next cell whenever you want to tighten or relax the cut.


In [173]:
CGW_SNR_THRESHOLD = 5.94


def build_cgw_threshold_simulation_table(sim_df: pd.DataFrame, snr_threshold: float) -> pd.DataFrame:
    """Filter the per-simulation CGW summary table to simulations above the SNR cut."""
    if sim_df.empty or 'loudest_cgw_snr' not in sim_df.columns:
        return pd.DataFrame()

    thresholded = sim_df.loc[sim_df['loudest_cgw_snr'] >= snr_threshold].copy()
    if thresholded.empty:
        return pd.DataFrame()

    thresholded['threshold_snr'] = float(snr_threshold)
    return thresholded.reset_index(drop=True)


def _threshold_extract(df, col):
    return {
        s: df.loc[df['scenario'] == s, col].to_numpy()
        for s in threshold_scenario_order
    }


def _threshold_scenario_histogram(
    ax,
    scenario_vals: dict,
    xlabel: str,
    logx: bool = False,
    show_legend: bool = True,
    vline: float = None,
    loc: str = 'upper left',
    show_ci_band: bool = False,
    ci_alpha: float = 0.08,
):
    all_vals = np.concatenate([v for v in scenario_vals.values() if len(v) > 0])
    all_vals = all_vals[np.isfinite(all_vals)]
    if len(all_vals) == 0:
        return

    if logx:
        bins = np.logspace(np.log10(all_vals.min()), np.log10(all_vals.max()), 25)
    else:
        bins = np.linspace(all_vals.min(), all_vals.max(), 25)
    if show_ci_band:
        print(f"\n{xlabel}:")
    for s in threshold_scenario_order:
        vals = scenario_vals.get(s, np.array([]))
        vals = vals[np.isfinite(vals)]
        if len(vals) == 0:
            continue
        style = threshold_scenario_styles.get(s, {'color': '#888888', 'linestyle': '-', 'linewidth': 1.5})
        if show_ci_band:
            lo, hi = np.nanpercentile(vals, [2.5, 97.5])
            median = np.nanmedian(vals)
            print(f"  {s}: median = {median:.3g}, 95% CI = [{lo:.3g}, {hi:.3g}]")
            counts, edges = np.histogram(vals, bins=bins)
            lo_idx = np.clip(np.searchsorted(edges, lo, side='right') - 1, 0, len(counts) - 1)
            hi_idx = np.clip(np.searchsorted(edges, hi, side='right') - 1, 0, len(counts) - 1)
            for i in range(lo_idx, hi_idx + 1):
                ax.bar(
                    edges[i],
                    counts[i],
                    width=edges[i + 1] - edges[i],
                    align='edge',
                    color=style['color'],
                    alpha=0.12,
                    zorder=1,
                    linewidth=0,
                )
        ax.hist(
            vals,
            bins=bins,
            histtype='step',
            color=style['color'],
            linestyle=style['linestyle'],
            linewidth=style['linewidth'],
            label=s,
            density=False,
            zorder=2,
        )

    ax.set_xlabel(xlabel, fontsize=11)
    ax.set_ylabel('Number of simulations', fontsize=11)
    ax.tick_params(labelsize=8)
    ax.spines['top'].set_visible(True)
    ax.spines['right'].set_visible(True)
    if logx:
        ax.set_xscale('log')
    if vline is not None:
        ax.axvline(vline, color='black', linestyle='--', linewidth=1.5, zorder=3)
    if show_legend:
        handles = [
            mlines.Line2D(
                [], [],
                color=threshold_scenario_styles[s]['color'],
                linestyle=threshold_scenario_styles[s]['linestyle'],
                linewidth=threshold_scenario_styles[s]['linewidth'],
                label=threshold_scenario_label_map[s],
            )
            for s in threshold_scenario_order if s in threshold_scenario_styles
        ]
        if vline is not None:
            handles.append(mlines.Line2D(
                [], [],
                color='black',
                linestyle='dashed',
                linewidth=1.5,
                label='Threshold',
            ))
        ax.legend(handles=handles, frameon=False, fontsize=8, loc=loc)


cgw_threshold_sim_df = build_cgw_threshold_simulation_table(cgw_sim_df, CGW_SNR_THRESHOLD)
print('CGW simulations with thresholded top-source diagnostics:', len(cgw_threshold_sim_df))
if cgw_threshold_sim_df.empty:
    print('No per-simulation CGW diagnostics met the SNR threshold.')
else:
    threshold_scenario_order = [s for s in SCENARIOS if s in set(cgw_threshold_sim_df.get('scenario', []))]
    threshold_scenario_label_map = {s: scenario_label_map.get(s, s) for s in threshold_scenario_order}
    threshold_scenario_styles = {
        s: scenario_styles.get(s, {'color': '#888888', 'linestyle': '-', 'linewidth': 1.5})
        for s in threshold_scenario_order
    }
    
threshold_scenario_styles = {
    'optimistic':  {'color': 'lime',    'linestyle': '-',  'linewidth': 2.2},
    'realistic':   {'color': 'magenta', 'linestyle': '--', 'linewidth': 2.2},
    'pessimistic': {'color': 'navy',    'linestyle': ':',  'linewidth': 2.5},
}


CGW simulations with thresholded top-source diagnostics: 409


In [172]:
print("SCENARIOS:", SCENARIOS)
print("scenario_styles keys:", list(scenario_styles.keys()))
print("threshold_scenario_order:", threshold_scenario_order)
print("threshold_scenario_styles:", threshold_scenario_styles)
print("scenario_styles:", scenario_styles)

SCENARIOS: ('optimistic', 'realistic', 'pessimistic')
scenario_styles keys: ['optimistic', 'realistic', 'pessimistic']
threshold_scenario_order: ['optimistic', 'realistic', 'pessimistic']
threshold_scenario_styles: {'optimistic': {'color': '#888888', 'linestyle': '-', 'linewidth': 1.5}, 'realistic': {'color': '#888888', 'linestyle': '-', 'linewidth': 1.5}, 'pessimistic': {'color': '#888888', 'linestyle': '-', 'linewidth': 1.5}}
scenario_styles: {'optimistic': {'color': 'lime', 'linestyle': '-', 'linewidth': 2.2}, 'realistic': {'color': 'magenta', 'linestyle': '--', 'linewidth': 2.2}, 'pessimistic': {'color': 'navy', 'linestyle': ':', 'linewidth': 2.5}}


In [174]:
# --- SNR threshold detection summary ---
# Count sims above threshold per scenario (already filtered in cgw_threshold_sim_df)
if not cgw_sim_df.empty:
    total_sims_df = (
        cgw_sim_df
        .groupby('scenario')['sim_key']
        .nunique()
        .rename('n_sims_total')
    )

    if not cgw_threshold_sim_df.empty and 'scenario' in cgw_threshold_sim_df.columns:
        above_threshold = (
            cgw_threshold_sim_df
            .groupby('scenario')['sim_key']
            .nunique()
            .rename('n_sims_above_threshold')
        )
    else:
        above_threshold = pd.Series(dtype=int, name='n_sims_above_threshold')

    threshold_summary = (
        total_sims_df
        .to_frame()
        .join(above_threshold, how='left')
        .fillna({'n_sims_above_threshold': 0})
        .assign(n_sims_above_threshold=lambda df: df['n_sims_above_threshold'].astype(int))
        .assign(detection_fraction=lambda df: df['n_sims_above_threshold'] / df['n_sims_total'])
        .reset_index()
    )

    # Preserve scenario ordering
    threshold_summary['scenario'] = pd.Categorical(
        threshold_summary['scenario'],
        categories=[s for s in SCENARIOS if s in threshold_summary['scenario'].values],
        ordered=True,
    )
    threshold_summary = threshold_summary.sort_values('scenario')

    print(f"\nSNR threshold: {CGW_SNR_THRESHOLD}")
    print("Simulations with at least one binary above SNR threshold, by scenario:\n")
    display(threshold_summary)
else:
    print('No `cgw_sim_df` available or it is empty.')



SNR threshold: 5.94
Simulations with at least one binary above SNR threshold, by scenario:



,scenario,n_sims_total,n_sims_above_threshold,detection_fraction
0,optimistic,400,171,0.4275
2,realistic,400,135,0.3375
1,pessimistic,400,103,0.2575


In [175]:
if cgw_threshold_sim_df.empty:
    raise RuntimeError('No CGW simulation table available to plot (all simulations below threshold).')
figure_dir = Path('figures')
figure_dir.mkdir(parents=True, exist_ok=True)

# 1) nearest SMBHB distance
fig, ax = plt.subplots(figsize=(3.5, 2.8))
_threshold_scenario_histogram(
    ax,
    scenario_vals=_threshold_extract(cgw_threshold_sim_df, 'nearest_D'),
    xlabel=r'$D_{\rm{comov}}$ [Mpc]',
    logx=True,
    show_legend=True,
    loc='upper left',
)
fig.tight_layout()
fig.savefig(figure_dir / 'nearest_SMBHB_threshold.pdf', dpi=300, bbox_inches='tight')
plt.close(fig)

# 2) loudest-by-CGW binary parameters saved individually
threshold_param_specs = [
    ('loudest_h0', r'$h_0$',                                    'loudest_h0_CGW_threshold.pdf', True),
    ('loudest_Mc', r'$\mathcal{M}_c$ [$M_\odot$]',              'loudest_Mc_CGW_threshold.pdf', True),
    ('loudest_D',  r'$D_{\rm{comov}}$ [Mpc]',                   'loudest_D_CGW_threshold.pdf',  True),
    ('loudest_f',  r'$f$ [Hz]',                                  'loudest_f_CGW_threshold.pdf',  True),
]
for col, xlabel, filename, use_logx in threshold_param_specs:
    show_leg = (filename == 'loudest_D_CGW_threshold.pdf')
    fig, ax = plt.subplots(figsize=(3.5, 2.8))
    _threshold_scenario_histogram(
        ax,
        scenario_vals=_threshold_extract(cgw_threshold_sim_df, col),
        xlabel=xlabel,
        logx=use_logx,
        show_legend=show_leg,
        loc='upper left',
        show_ci_band=True,
        ci_alpha=0.08,
    )
    fig.tight_layout()
    fig.savefig(figure_dir / filename, dpi=300, bbox_inches='tight')
    plt.close(fig)

# 3) CGW SNR distribution with threshold line
fig, ax = plt.subplots(figsize=(3.5, 2.8))
_threshold_scenario_histogram(
    ax,
    scenario_vals=_threshold_extract(cgw_threshold_sim_df, 'loudest_cgw_snr'),
    xlabel=r'$(\mathrm{S/N})_s$',
    logx=True,
    show_legend=True,
    vline=CGW_SNR_THRESHOLD,
    loc='upper right',
)
fig.tight_layout()
fig.savefig(figure_dir / 'loudest_cgw_snr_threshold.pdf', dpi=300, bbox_inches='tight')
plt.close(fig)


$h_0$:
  optimistic: median = 3.05e-15, 95% CI = [2.84e-17, 1.43e-14]
  realistic: median = 1.67e-16, 95% CI = [2.99e-17, 1.5e-14]
  pessimistic: median = 9.66e-17, 95% CI = [1.36e-17, 5.82e-15]

$\mathcal{M}_c$ [$M_\odot$]:
  optimistic: median = 5.78e+09, 95% CI = [1.51e+09, 1.49e+10]
  realistic: median = 2.2e+09, 95% CI = [1.03e+09, 6.85e+09]
  pessimistic: median = 1.47e+09, 95% CI = [5.48e+08, 2.54e+09]

$D_{\rm{comov}}$ [Mpc]:
  optimistic: median = 903, 95% CI = [27.4, 1.12e+04]
  realistic: median = 1.01e+03, 95% CI = [17.3, 1.12e+04]
  pessimistic: median = 620, 95% CI = [37.1, 1.12e+04]

$f$ [Hz]:
  optimistic: median = 7.55e-09, 95% CI = [2.27e-09, 2.74e-08]
  realistic: median = 5.21e-09, 95% CI = [2.16e-09, 1.73e-08]
  pessimistic: median = 5.96e-09, 95% CI = [2.55e-09, 3.04e-08]


In [181]:
fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.2), sharey=True)

for ax, s in zip(axes, scenario_order):
    sub   = cgw_threshold_sim_df[cgw_threshold_sim_df['scenario'] == s]
    style = scenario_styles.get(s, {'color': '#888888'})

    sc = ax.scatter(
        sub['loudest_Mc'],
        sub['loudest_h0'],
        c=sub['loudest_D'],
        cmap='viridis_r',
        s=4,
        alpha=0.5,
        rasterized=True,
    )
    cb = fig.colorbar(sc, ax=ax, pad=0.02)
    cb.set_label(r'$D_{\mathrm{comov}}$ [Mpc]', fontsize=8)
    cb.ax.tick_params(labelsize=7)

    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel(r'$\mathcal{M}_c\ [M_\odot]$', fontsize=9)
    ax.set_title(scenario_label_map.get(s, s), fontsize=8)
    ax.tick_params(labelsize=8)
    ax.spines['top'].set_visible(True)
    ax.spines['right'].set_visible(True)

axes[0].set_ylabel(r'$h_0$', fontsize=9)

fig.tight_layout(pad=0.4)
fig.savefig('figures/loudest_h0_vs_Mc_scatter_thresh.pdf', dpi=300, bbox_inches='tight')
plt.show(); plt.close(fig)
print('  saved loudest_h0_vs_Mc_scatter_thresh.pdf')

# also plot h0 vs frequency to check the other axis
fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.2), sharey=True)

for ax, s in zip(axes, scenario_order):
    sub   = cgw_threshold_sim_df[cgw_threshold_sim_df['scenario'] == s]
    style = scenario_styles.get(s, {'color': '#888888'})

    sc = ax.scatter(
        sub['loudest_f'],
        sub['loudest_h0'],
        c=sub['loudest_Mc'],
        cmap='plasma',
        norm=plt.matplotlib.colors.LogNorm(),
        s=4,
        alpha=0.5,
        rasterized=True,
    )
    cb = fig.colorbar(sc, ax=ax, pad=0.02)
    cb.set_label(r'$\mathcal{M}_c\ [M_\odot]$', fontsize=8)
    cb.ax.tick_params(labelsize=7)

    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel(r'$f$ [Hz]', fontsize=9)
    ax.set_title(scenario_label_map.get(s, s), fontsize=8)
    ax.tick_params(labelsize=8)
    ax.spines['top'].set_visible(True)
    ax.spines['right'].set_visible(True)

axes[0].set_ylabel(r'$h_0$', fontsize=9)

fig.tight_layout(pad=0.4)
fig.savefig('figures/loudest_h0_vs_f_scatter_thresh.pdf', dpi=300, bbox_inches='tight')
plt.show(); plt.close(fig)
print('  saved loudest_h0_vs_f_scatter_thresh.pdf')

  saved loudest_h0_vs_Mc_scatter_thresh.pdf
  saved loudest_h0_vs_f_scatter_thresh.pdf


In [135]:
# Sky maps for thresholded loudest-by-CGW binaries
threshold_sky_df = cgw_threshold_sim_df[
    np.isfinite(cgw_threshold_sim_df['loudest_ra']) & np.isfinite(cgw_threshold_sim_df['loudest_dec'])
].copy()

if not threshold_sky_df.empty:
    # ── Sky map 1: thresholded loudest binaries colored by scenario ─────────────────
    fig = plt.figure(figsize=(11, 5.5))
    ax  = fig.add_subplot(111, projection='aitoff')

    for scenario in threshold_scenario_order:
        sub = threshold_sky_df[threshold_sky_df['scenario'] == scenario]
        if sub.empty:
            continue
        style = threshold_scenario_styles.get(scenario, {'color': '#888888'})
        ax.scatter(
            _wrap_ra(sub['loudest_ra'].to_numpy()),
            sub['loudest_dec'].to_numpy(),
            s=18,
            alpha=0.65,
            color=style['color'],
            edgecolors='none',
            label=threshold_scenario_label_map.get(scenario, scenario),
            zorder=3,
        )

    if pulsar_ra.size > 0:
        ax.scatter(
            pulsar_ra, pulsar_dec,
            s=25, marker='*',
            color='black',
            alpha=0.9,
            label='Pulsars',
            zorder=4,
        )

    _style_skyax(ax)
    ax.legend(loc='lower center', frameon=False, fontsize=8,
              bbox_to_anchor=(0.5, -0.18), ncol=len(threshold_scenario_order) + 2)
    fig.suptitle('Sky Locations of Thresholded Loudest-by-CGW Binaries (SNR ≥ {:.1f})'.format(CGW_SNR_THRESHOLD), fontsize=12, y=1.01)
    fig.tight_layout()
    fig.savefig("figures/SNR_loudest_binary_CGW_threshold.pdf", dpi=300, bbox_inches='tight')

    # ── Sky map 2: thresholded loudest binaries colored by CGW SNR ────────────────────
    fig = plt.figure(figsize=(11, 5.5))
    ax  = fig.add_subplot(111, projection='aitoff')

    sc = ax.scatter(
        _wrap_ra(threshold_sky_df['loudest_ra'].to_numpy()),
        threshold_sky_df['loudest_dec'].to_numpy(),
        c=threshold_sky_df['loudest_cgw_snr'].to_numpy(),
        cmap='plasma',
        s=18,
        alpha=0.8,
        edgecolors='none',
        zorder=3,
    )

    if pulsar_ra.size > 0:
        ax.scatter(
            pulsar_ra, pulsar_dec,
            s=25, marker='*',
            color='white',
            edgecolors='black',
            linewidths=0.5,
            alpha=0.95,
            label='Pulsars',
            zorder=4,
        )

    cbar = plt.colorbar(sc, ax=ax, pad=0.08, shrink=0.75, orientation='vertical')
    cbar.set_label('Loudest binary CGW SNR', fontsize=9)
    cbar.ax.tick_params(labelsize=8)

    _style_skyax(ax)
    if pulsar_ra.size > 0:
        ax.legend(loc='lower center', frameon=False, fontsize=8,
                  bbox_to_anchor=(0.5, -0.18), ncol=2)
    fig.suptitle('Sky Locations of Thresholded Loudest-by-CGW Binaries (SNR ≥ {:.1f}, coloured by CGW SNR)'.format(CGW_SNR_THRESHOLD), fontsize=12, y=1.01)
    fig.tight_layout()
    fig.savefig("figures/coloured_SNR_loudest_binary_CGW_threshold.pdf", dpi=300, bbox_inches='tight')
else:
    print("No valid RA/Dec values for thresholded loudest-by-CGW sources.")

TypeError: _style_skyax() missing 1 required positional argument: 'fig'

## Multi-Simulation Population Comparisons

This section overlays binary frequency distributions from all simulations of a chosen scenario, filtering optionally by maximum redshift. Select the scenario and redshift cutoff below.

In [81]:
# ========== Configuration ==========
# Select which scenario to analyze
COMPARISON_SCENARIO = "pessimistic"  # Choose from: "optimistic", "pessimistic", "realistic"

# Toggle redshift filtering (set to None to include all redshifts, or specify max z)
MAX_REDSHIFT = None  # Set to a value like 2.0, 5.0, etc., or None for no cutoff

# Streaming parameters for memory efficiency
ALPHA_POPULATION = 0.08  # Transparency for each population line (lower = less clutter)
N_FREQ_BINS = 30         # Number of frequency bins if not supplied explicitly

# Candidate vertical lines, matching the visualisation.py interface
CANDIDATE_FREQUENCIES = None  # e.g. [1e-7, 2e-7] or None
CANDIDATE_LABELS = None       # e.g. ["Candidate 1", "Candidate 2"] or None
CANDIDATE_MASSES = None       # optional log10(Mtot/Msun) annotations for candidates

print("Configuration (Streaming plot_binaries_vs_frequency style):")
print(f"  Scenario: {COMPARISON_SCENARIO}")
print(f"  Max redshift: {MAX_REDSHIFT if MAX_REDSHIFT is not None else 'None (all)'}")
print(f"  Alpha per population: {ALPHA_POPULATION}")
print(f"  Frequency bins: {N_FREQ_BINS}")
print(f"  Candidate frequencies: {CANDIDATE_FREQUENCIES}")
print("\n✓ Will draw one step-line per population, with low alpha, on a single axes")

Configuration (Streaming plot_binaries_vs_frequency style):
  Scenario: pessimistic
  Max redshift: None (all)
  Alpha per population: 0.08
  Frequency bins: 30
  Candidate frequencies: None

✓ Will draw one step-line per population, with low alpha, on a single axes


In [ ]:
def streaming_plot_binaries_vs_frequency(
    files_list,
    scenario_name,
    max_z=None,
    mass_bins=None,
    freq_bins=None,
    candidate_frequencies=None,
    candidate_labels=None,
    candidate_masses=None,
    subset_name='Subset',
    n_freq_bins=30,
    alpha_population=0.08,
):
    """Stream a `plot_binaries_vs_frequency`-style figure without loading all populations.

    Each population is drawn as its own low-alpha step line, with the same mass-bin
    colors, candidate vertical lines, log-log axes, and legend handling as the
    reference implementation in visualisation.py.
    """
    from config import Msun

    if mass_bins is None:
        mass_bins = np.arange(7.5, 10.6, 0.5)

    def _population_mass_and_frequency(population):
        arrays = _population_to_arrays(population)
        if arrays is None or 'f' not in arrays:
            return None, None

        fgw = np.asarray(arrays['f'], dtype=float)
        fgw = fgw[np.isfinite(fgw)]
        if fgw.size == 0:
            return None, None

        if 'Mtot' in arrays and arrays['Mtot'] is not None:
            total_mass = np.asarray(arrays['Mtot'], dtype=float)
        elif 'Mc' in arrays and arrays['Mc'] is not None and 'q' in arrays and arrays['q'] is not None:
            Mc = np.asarray(arrays['Mc'], dtype=float)
            q = np.asarray(arrays['q'], dtype=float)
            total_mass = Mc * (1.0 + q) ** (6.0 / 5.0) / (q ** (3.0 / 5.0))
        else:
            total_mass = None

        if total_mass is not None:
            total_mass = np.asarray(total_mass, dtype=float)
            if total_mass.size != fgw.size:
                n = min(total_mass.size, fgw.size)
                total_mass = total_mass[:n]
                fgw = fgw[:n]
            mask = np.isfinite(total_mass) & np.isfinite(fgw)
            total_mass = total_mass[mask]
            fgw = fgw[mask]
            if total_mass.size == 0:
                return None, None

        return fgw, total_mass

    # Pass 1: determine frequency range without retaining populations.
    if freq_bins is None:
        f_min = np.inf
        f_max = 0.0
        n_populations_seen = 0

        for fp in files_list:
            if infer_scenario(fp) != scenario_name:
                continue
            try:
                payload = _load_payload(fp)
            except Exception:
                continue
            if not isinstance(payload, dict):
                continue
            pops = payload.get('populations', [])
            if not isinstance(pops, list):
                continue

            for entry in pops:
                if not isinstance(entry, dict):
                    continue
                population = entry.get('population')
                if population is None:
                    continue
                if max_z is not None:
                    population = _filter_population_by_redshift(population, max_z)
                fgw, _ = _population_mass_and_frequency(population)
                if fgw is None or fgw.size == 0:
                    continue
                f_min = min(f_min, float(np.nanmin(fgw)))
                f_max = max(f_max, float(np.nanmax(fgw)))
                n_populations_seen += 1

        if n_populations_seen == 0 or not np.isfinite(f_min) or not np.isfinite(f_max):
            print(f"No usable populations found for scenario '{scenario_name}'")
            return None

        freq_bins = np.logspace(np.log10(f_min), np.log10(f_max), n_freq_bins)

    bin_centers = 0.5 * (freq_bins[:-1] + freq_bins[1:])
    n_mass = len(mass_bins) - 1

    colors = [
        "#0072B2",
        "#E69F00",
        "#009E73",
        "#D55E00",
        "#CC79A7",
        "#56B4E9",
        "#000000",
    ]
    linestyles = ['-', '--']
    cand_colors = ['r', 'm', 'c', 'y']

    fig, ax = plt.subplots(figsize=(8, 6))

    # Stream and plot each population immediately.
    n_plotted = 0
    for fp in files_list:
        if infer_scenario(fp) != scenario_name:
            continue
        try:
            payload = _load_payload(fp)
        except Exception:
            continue
        if not isinstance(payload, dict):
            continue
        pops = payload.get('populations', [])
        if not isinstance(pops, list):
            continue

        for entry in pops:
            if not isinstance(entry, dict):
                continue
            population = entry.get('population')
            if population is None:
                continue
            if max_z is not None:
                population = _filter_population_by_redshift(population, max_z)

            fgw, total_mass = _population_mass_and_frequency(population)
            if fgw is None or fgw.size == 0:
                continue

            total_counts, _ = np.histogram(fgw, bins=freq_bins)
            ax.plot(
                bin_centers,
                np.where(total_counts > 0, total_counts, 0.1),
                color='k',
                linewidth=1.0,
                alpha=alpha_population,
                linestyle='-',
                drawstyle='steps-mid',
                label='Total' if n_plotted == 0 else None,
                zorder=1,
            )

            if total_mass is not None and total_mass.size == fgw.size:
                log_mass = np.log10(total_mass)
                for i in range(n_mass):
                    mask = (log_mass >= mass_bins[i]) & (log_mass < mass_bins[i + 1])
                    if not np.any(mask):
                        continue
                    counts, _ = np.histogram(fgw[mask], bins=freq_bins)
                    ax.plot(
                        bin_centers,
                        np.where(counts > 0, counts, 0.1),
                        color=colors[i % len(colors)],
                        linewidth=1.6,
                        alpha=alpha_population,
                        linestyle=linestyles[i % len(linestyles)],
                        drawstyle='steps-mid',
                        label=(r"$%.1f < \log_{10}\!\left(M_{\rm tot}/\mathrm{M}_\odot\right) < %.1f$" % (mass_bins[i], mass_bins[i + 1])) if n_plotted == 0 else None,
                        zorder=2,
                    )
            n_plotted += 1

    if n_plotted == 0:
        print(f"No populations found for scenario '{scenario_name}'")
        return None

    # Candidate frequencies match the reference interface.
    if candidate_frequencies is not None:
        if candidate_labels is None:
            candidate_labels = [f"Candidate {j + 1}" for j in range(len(candidate_frequencies))]
        if candidate_masses is None:
            candidate_masses = [None] * len(candidate_frequencies)

        for i, (f0, label, mass_val) in enumerate(zip(candidate_frequencies, candidate_labels, candidate_masses)):
            if mass_val is not None:
                label = (
                    rf"{label} "
                    rf"$\left[\log_{{10}}\!\left(M_{{\rm tot}}/M_\odot\right)={mass_val:.2f}\right]$"
                )
            ax.axvline(
                f0,
                color=cand_colors[i % len(cand_colors)],
                linestyle='-',
                lw=3,
                alpha=1,
                label=label,
                zorder=5,
            )

    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel('Gravitational Wave Frequency [Hz]', fontsize=12, fontweight='bold')
    ax.set_ylabel('Number of binaries', fontsize=12, fontweight='bold')
    ax.set_title(f'Binaries by GW frequency ({subset_name})', fontsize=13, fontweight='bold')

    from matplotlib.ticker import LogLocator
    ax.xaxis.set_major_locator(LogLocator(base=10))
    ax.xaxis.set_minor_locator(LogLocator(base=10, subs=np.arange(2, 10) * 0.1))
    ax.yaxis.set_major_locator(LogLocator(base=10))
    ax.yaxis.set_minor_locator(LogLocator(base=10, subs=np.arange(2, 10) * 0.1))
    ax.tick_params(which='both', direction='in', top=True, right=True)

    ax.set_xlim(freq_bins[0], freq_bins[-1] * 1.2)
    ax.set_ylim(0.2, ax.get_ylim()[1])

    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color('black')
        spine.set_linewidth(1)

    handles, labels = ax.get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    ax.legend(by_label.values(), by_label.keys(), fontsize=9, frameon=True, edgecolor='black')

    plt.tight_layout()
    return fig


In [ ]:
# Execute the streaming comparison with the exact plot style from visualisation.py
print("\n" + "=" * 70)
print(f"Generating streaming multi-simulation frequency comparison for {COMPARISON_SCENARIO}...")
if MAX_REDSHIFT is not None:
    print(f"Redshift filter: z ≤ {MAX_REDSHIFT}")
print("=" * 70)

fig = streaming_plot_binaries_vs_frequency(
    result_files_CGW,
    scenario_name=COMPARISON_SCENARIO,
    max_z=MAX_REDSHIFT,
    mass_bins=np.arange(7.5, 10.6, 0.5),
    freq_bins=None,
    candidate_frequencies=CANDIDATE_FREQUENCIES,
    candidate_labels=CANDIDATE_LABELS,
    candidate_masses=CANDIDATE_MASSES,
    subset_name=COMPARISON_SCENARIO,
    n_freq_bins=N_FREQ_BINS,
    alpha_population=ALPHA_POPULATION,
)

if fig is not None:
    fname = f"figures/multi_sim_frequency_comparison_{COMPARISON_SCENARIO}"
    if MAX_REDSHIFT is not None:
        fname += f"_z{MAX_REDSHIFT}"
    fname += ".pdf"
    fig.savefig(fname, dpi=300, bbox_inches='tight')
    print(f"\n✓ Saved to {fname}")
    plt.close(fig)
else:
    print(f"\n✗ Could not generate plot for {COMPARISON_SCENARIO}")


Generating multi-simulation frequency comparison...
[PASS 1] Scanning 2398 files for frequency range...
  ✓ Found 800 populations in f ∈ [1.98e-09, 3.8e-07] Hz
[PASS 2] Computing mean histograms (sampling 50 individual curves)...
    Processed 50 simulations...
    Processed 100 simulations...
    Processed 150 simulations...
    Processed 200 simulations...
    Processed 250 simulations...
    Processed 300 simulations...
    Processed 350 simulations...
    Processed 400 simulations...
    Processed 450 simulations...
    Processed 500 simulations...


In [ ]:
# Summary printout: counts and min/max for CGW SNR histograms
import numpy as np

print('\n=== CGW SNR Summary ===')
if 'cgw_sim_df' in globals() and not cgw_sim_df.empty:
    col = 'loudest_cgw_snr'
    scenarios = sorted(cgw_sim_df['scenario'].unique())
    total_vals = []
    for s in scenarios:
        arr = cgw_sim_df.loc[cgw_sim_df['scenario'] == s, col].to_numpy(dtype=float)
        finite = arr[np.isfinite(arr)]
        total_vals.append(finite)
        if finite.size:
            print(f"Scenario '{s}': plotted {finite.size} SNR values; min={finite.min():.3g}, max={finite.max():.3g}")
        else:
            print(f"Scenario '{s}': plotted 0 SNR values")
    all_vals = np.concatenate([a for a in total_vals if a.size > 0]) if total_vals else np.array([])
    if all_vals.size:
        print(f"Global SNR: n={all_vals.size}, min={all_vals.min():.3g}, max={all_vals.max():.3g}")
    else:
        print('Global SNR: no finite values to plot')
else:
    print('No `cgw_sim_df` available or it is empty.')

print('\n=== Thresholded CGW SNR Summary ===')
if 'cgw_threshold_sim_df' in globals() and not cgw_threshold_sim_df.empty:
    col = 'loudest_cgw_snr'
    scenarios = sorted(cgw_threshold_sim_df['scenario'].unique())
    total_vals = []
    for s in scenarios:
        arr = cgw_threshold_sim_df.loc[cgw_threshold_sim_df['scenario'] == s, col].to_numpy(dtype=float)
        finite = arr[np.isfinite(arr)]
        total_vals.append(finite)
        if finite.size:
            print(f"Scenario '{s}' (thresholded): plotted {finite.size} SNR values; min={finite.min():.3g}, max={finite.max():.3g}")
        else:
            print(f"Scenario '{s}' (thresholded): plotted 0 SNR values")
    all_vals = np.concatenate([a for a in total_vals if a.size > 0]) if total_vals else np.array([])
    if all_vals.size:
        print(f"Global thresholded SNR: n={all_vals.size}, min={all_vals.min():.3g}, max={all_vals.max():.3g}")
    else:
        print('Global thresholded SNR: no finite values to plot')
else:
    print('No `cgw_threshold_sim_df` available or it is empty.')

## Cumulative SNR as a function of the number of pulsars

In [136]:
# Cumulative SNR contribution from the loudest binary in each simulation.
#
# Each curve is built from the saved `top_cgw_breakdowns` metadata in
# `summary.pkl.gz`, where `per_pulsar_rho_sq` stores the per-pulsar SNR^2
# contribution for the loudest binary in that simulation.
#
# Layout: forest-plot style. Pulsars listed top-to-bottom ranked by median
# per-pulsar rho^2 contribution. For each pulsar, a filled circle shows the
# median cumulative SNR fraction and horizontal error bars span the 95%
# credible interval across simulations. Font size is scaled to fit all pulsars.

import gzip
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D
from matplotlib.patches import Patch


# ---------------------------------------------------------------------------
# Path / data helpers
# ---------------------------------------------------------------------------

def _resolve_summary_path(source_value):
    if source_value is None or pd.isna(source_value):
        return None
    source_path = Path(str(source_value))
    if source_path.is_dir():
        candidate = source_path / 'summary.pkl.gz'
        return candidate if candidate.is_file() else None
    if source_path.name == 'summary.pkl.gz' and source_path.is_file():
        return source_path
    candidate = source_path.parent / 'summary.pkl.gz'
    return candidate if candidate.is_file() else None


def _load_loudest_breakdown(summary_path, scenario_name):
    summary_path = Path(summary_path)
    if not summary_path.is_file():
        return None

    with gzip.open(summary_path, 'rb') as fh:
        payload = pickle.load(fh)

    top_breakdowns = payload.get('meta', {}).get('top_cgw_breakdowns', []) or []
    if not top_breakdowns:
        return None

    loudest = max(top_breakdowns, key=lambda item: float(item.get('cgw_snr', np.nan)))
    per_pulsar = loudest.get('per_pulsar_rho_sq', {}) or {}
    if not per_pulsar:
        return None

    ordered = sorted(per_pulsar.items(), key=lambda kv: float(kv[1]), reverse=True)
    rho_sq = np.array([max(float(v), 0.0) for _, v in ordered], dtype=float)
    total = np.sum(rho_sq)
    cumulative_snr = np.sqrt(np.cumsum(rho_sq)) / np.sqrt(total) if total > 0 else np.zeros_like(rho_sq)

    return {
        'scenario':       scenario_name,
        'summary_path':   summary_path,
        'sim_key':        summary_path.parent.name,
        'loudest_cgw_snr': float(loudest.get('cgw_snr', np.nan)),
        'pulsar_names':   [name for name, _ in ordered],
        'rho_sq':         rho_sq,
        'cumulative_snr': cumulative_snr,
    }


def _pad_curves(curves):
    if not curves:
        return np.empty((0, 0), dtype=float)
    max_len = max(len(c) for c in curves)
    padded = np.full((len(curves), max_len), np.nan, dtype=float)
    for i, c in enumerate(curves):
        if len(c) == 0:
            continue
        padded[i, :len(c)] = c
        padded[i, len(c):] = c[-1]
    return padded


# ---------------------------------------------------------------------------
# Forest-plot helpers
# ---------------------------------------------------------------------------

def _median_pulsar_order(scenario_rows, n_pulsars):
    """
    Return a list of exactly n_pulsars unique pulsar names, ordered by each
    pulsar's median rank across simulations (rank 1 = highest rho^2 contributor).

    Strategy: collect every pulsar name seen across all simulations, compute
    each pulsar's median rank position, then sort by that median rank. This
    guarantees no duplicates and that every pulsar that appears in any
    simulation is included exactly once.
    """
    all_name_lists = [row for row in scenario_rows['pulsar_names'].tolist() if row]
    if not all_name_lists:
        return [f'Rank {i + 1}' for i in range(n_pulsars)]

    # Collect all unique pulsar names seen across all simulations.
    all_pulsars = set()
    for name_list in all_name_lists:
        all_pulsars.update(name_list)

    # For each pulsar, record the rank (0-based) it appears at in each sim.
    # If it is absent from a sim, treat it as worst-possible rank.
    worst_rank = max(len(nl) for nl in all_name_lists)
    rank_records = {p: [] for p in all_pulsars}
    for name_list in all_name_lists:
        seen = {name: idx for idx, name in enumerate(name_list)}
        for p in all_pulsars:
            rank_records[p].append(seen.get(p, worst_rank))

    median_rank = {p: np.median(ranks) for p, ranks in rank_records.items()}

    # Sort all pulsars by median rank; take the top n_pulsars.
    ordered = sorted(all_pulsars, key=lambda p: median_rank[p])
    ordered = ordered[:n_pulsars]

    # Pad with fallback labels if somehow fewer pulsars than expected.
    while len(ordered) < n_pulsars:
        ordered.append(f'Rank {len(ordered) + 1}')

    return ordered


def _build_forest_data(padded, bands):
    """
    Build per-pulsar (row) statistics for the forest plot from the padded
    cumulative-SNR matrix and precomputed band arrays.

    Returns arrays of length n_pulsars:
      - median_cumul  : median cumulative SNR fraction at each rank
      - lo95, hi95    : 2.5th / 97.5th percentile of cumulative SNR
      - median_marginal: median *marginal* SNR fraction (contribution of that
                         pulsar alone, not cumulative) — used to drive the
                         fade-out alpha
    """
    p50   = bands['p50']
    p2_5  = bands['p2_5']
    p97_5 = bands['p97_5']

    # Marginal contribution = difference in cumulative SNR between consecutive ranks
    marginal = np.concatenate([[p50[0]], np.diff(p50)])

    return p50, p2_5, p97_5, marginal


def _alpha_ramp(marginal, min_alpha=0.15, threshold=0.003):
    """
    Per-pulsar alpha values. Full opacity while marginal >= threshold,
    then linearly ramps to min_alpha.
    """
    alphas = np.ones(len(marginal))
    significant = marginal >= threshold
    if not np.any(~significant):
        return alphas

    first_insig = np.argmax(~significant)
    n_tail = len(marginal) - first_insig
    if n_tail > 0:
        ramp = np.linspace(1.0, min_alpha, n_tail)
        alphas[first_insig:] = ramp
    return alphas


def _draw_forest(ax, pulsar_names, median_cumul, lo95, hi95, alphas, color):
    """
    Draw one forest-plot panel onto ax.

    y positions: 0 (top) = rank 1, increasing downward.
    x axis: cumulative SNR fraction [0, 1].
    """
    n = len(pulsar_names)
    y_pos = np.arange(n, dtype=float)

    for i in range(n):
        a = float(alphas[i])
        # Error bar (95% CI as horizontal caps)
        ax.plot(
            [lo95[i], hi95[i]], [y_pos[i], y_pos[i]],
            color=color, alpha=a, linewidth=1.2, solid_capstyle='butt',
        )
        # Cap ticks
        cap_size = 0.3
        for x_cap in (lo95[i], hi95[i]):
            ax.plot(
                [x_cap, x_cap],
                [y_pos[i] - cap_size, y_pos[i] + cap_size],
                color=color, alpha=a, linewidth=1.2,
            )
        # Median circle
        ax.scatter(
            median_cumul[i], y_pos[i],
            s=18, color=color, alpha=a, zorder=3, linewidths=0,
        )

    # Dotted reference line at x = 1
    ax.axvline(1.0, color='gray', linewidth=0.6, linestyle=':', alpha=0.4)

    ax.set_yticks(y_pos)
    ax.set_yticklabels(pulsar_names)
    ax.set_ylim(n - 0.5, -0.5)   # rank 1 at top
    ax.set_xlim(-0.02, 1.08)
    ax.set_xlabel('Cumulative fraction of total SNR')
    ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
    ax.grid(True, axis='x', alpha=0.2, linewidth=0.6)
    ax.tick_params(axis='y', length=0)


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

if 'cgw_sim_df' not in globals() or cgw_sim_df.empty:
    print('No cgw_sim_df available yet, so the cumulative SNR plots were skipped.')
else:
    breakdown_records = []
    for _, row in cgw_sim_df.iterrows():
        summary_path = _resolve_summary_path(row.get('source_file'))
        if summary_path is None:
            continue
        scenario_name = str(row.get('scenario', 'unknown'))
        record = _load_loudest_breakdown(summary_path, scenario_name)
        if record is None:
            continue
        record['sim_key']     = str(row.get('sim_key', record['sim_key']))
        record['source_file'] = str(row.get('source_file', summary_path))
        breakdown_records.append(record)

    breakdown_df = pd.DataFrame(breakdown_records)

    if breakdown_df.empty:
        print('No loudest-binary breakdowns were found in the loaded summaries.')
    else:
        scenario_values = breakdown_df['scenario'].dropna().unique().tolist()
        if 'scenario_order' in globals():
            scenarios = [s for s in scenario_order if s in scenario_values]
            scenarios.extend(s for s in scenario_values if s not in scenarios)
        else:
            scenarios = sorted(scenario_values)

        color_map = {
            s: color
            for s, color in zip(scenarios, ['lime', 'magenta', 'navy'])
        }

        print(f'Loaded loudest-binary breakdowns for {len(breakdown_df)} simulations.')
        print('Breakdowns by scenario:')
        for s in scenarios:
            print(f'  {s}: {int((breakdown_df["scenario"] == s).sum())}')

        figure_dir = Path('figures')
        figure_dir.mkdir(parents=True, exist_ok=True)

        legend_elements = [
            Line2D([0], [0], marker='o', color='gray', markersize=5,
                   linewidth=0, label='Median'),
            Line2D([0], [0], color='gray', linewidth=1.2,
                   label='95% credible interval'),
        ]

        # ------------------------------------------------------------------
        # One figure per scenario
        # ------------------------------------------------------------------
        for scenario in scenarios:
            scenario_rows = breakdown_df[breakdown_df['scenario'] == scenario].copy()
            curves = [c for c in scenario_rows['cumulative_snr'].tolist() if len(c) > 0]
            if not curves:
                continue

            padded   = _pad_curves(curves)
            n_pulsars = padded.shape[1]
            bands = {
                'p2_5':  np.nanpercentile(padded,  2.5, axis=0),
                'p16':   np.nanpercentile(padded, 16.0, axis=0),
                'p50':   np.nanmedian(padded, axis=0),
                'p84':   np.nanpercentile(padded, 84.0, axis=0),
                'p97_5': np.nanpercentile(padded, 97.5, axis=0),
            }

            median_cumul, lo95, hi95, marginal = _build_forest_data(padded, bands)
            alphas       = _alpha_ramp(marginal)
            pulsar_names = _median_pulsar_order(scenario_rows, n_pulsars)
            color        = color_map[scenario]

            # Each pulsar gets a fixed row height so labels never overlap.
            # 0.20 inches/row comfortably fits 7 pt text at 300 dpi.
            ROW_HEIGHT = 0.20  # inches per pulsar row
            fontsize   = 7.0
            fig_height = max(4.0, n_pulsars * ROW_HEIGHT + 1.5)

            fig, ax = plt.subplots(figsize=(6.5, fig_height))
            plt.rcParams.update({'ytick.labelsize': fontsize})

            _draw_forest(ax, pulsar_names, median_cumul, lo95, hi95, alphas, color)

            # ax.set_title(
            #     f'Per-pulsar cumulative SNR contribution\n'
            #     f'(loudest binary, 95% CI across sims): {scenario}',
            #     fontsize=11,
            # )
            ax.legend(handles=legend_elements, frameon=False,
                      loc='lower right', fontsize=8)
            fig.tight_layout()

            out_path = figure_dir / f'cgw_cumulative_snr_forest_{scenario}.png'
            fig.savefig(out_path, dpi=300, bbox_inches='tight')
            plt.close(fig)
            print(f'  saved {out_path}')

        # ------------------------------------------------------------------
        # Combined figure — side-by-side panels, one per scenario
        # ------------------------------------------------------------------
        scenario_data = []
        for scenario in scenarios:
            scenario_rows = breakdown_df[breakdown_df['scenario'] == scenario].copy()
            curves = [c for c in scenario_rows['cumulative_snr'].tolist() if len(c) > 0]
            if not curves:
                continue
            padded    = _pad_curves(curves)
            n_pulsars = padded.shape[1]
            bands = {
                'p2_5':  np.nanpercentile(padded,  2.5, axis=0),
                'p50':   np.nanmedian(padded, axis=0),
                'p97_5': np.nanpercentile(padded, 97.5, axis=0),
            }
            # marginal from p50 only (for alpha)
            p50      = bands['p50']
            marginal = np.concatenate([[p50[0]], np.diff(p50)])
            scenario_data.append({
                'scenario':     scenario,
                'n_pulsars':    n_pulsars,
                'median_cumul': p50,
                'lo95':         bands['p2_5'],
                'hi95':         bands['p97_5'],
                'marginal':     marginal,
                'alphas':       _alpha_ramp(marginal),
                'pulsar_names': _median_pulsar_order(
                    breakdown_df[breakdown_df['scenario'] == scenario], n_pulsars
                ),
                'color':        color_map[scenario],
            })

        if scenario_data:
            max_n = max(d['n_pulsars'] for d in scenario_data)
            ROW_HEIGHT = 0.20  # inches per pulsar row
            fontsize   = 7.0
            fig_height = max(4.0, max_n * ROW_HEIGHT + 1.5)

            fig, axes = plt.subplots(
                1, len(scenario_data),
                figsize=(6.0 * len(scenario_data), fig_height),
                sharey=False,
            )
            if len(scenario_data) == 1:
                axes = [axes]

            plt.rcParams.update({'ytick.labelsize': fontsize})

            for ax, d in zip(axes, scenario_data):
                _draw_forest(
                    ax,
                    d['pulsar_names'],
                    d['median_cumul'],
                    d['lo95'],
                    d['hi95'],
                    d['alphas'],
                    d['color'],
                )
                ax.set_title(d['scenario'], fontsize=11)

            axes[0].set_ylabel(
                'Pulsar (ranked by median $\\rho^2$ contribution)', fontsize=9
            )
            # fig.suptitle(
            #     'Per-pulsar cumulative SNR contribution\n'
            #     '(loudest binary, 95% CI across simulations)',
            #     fontsize=12, y=1.01,
            # )
            fig.legend(
                handles=legend_elements, frameon=False,
                loc='upper center', bbox_to_anchor=(0.5, 1.0),
                ncol=2, fontsize=9,
            )
            fig.tight_layout()

            combined_out = figure_dir / 'cgw_cumulative_snr_forest_combined.pdf'
            fig.savefig(combined_out, dpi=300, bbox_inches='tight')
            plt.close(fig)
            print(f'  saved {combined_out}')

Loaded loudest-binary breakdowns for 1200 simulations.
Breakdowns by scenario:
  optimistic: 400
  realistic: 400
  pessimistic: 400
  saved figures/cgw_cumulative_snr_forest_optimistic.png
  saved figures/cgw_cumulative_snr_forest_realistic.png
  saved figures/cgw_cumulative_snr_forest_pessimistic.png
  saved figures/cgw_cumulative_snr_forest_combined.pdf


In [137]:
# Cumulative SNR contribution from the loudest binary in each simulation.
#
# Each curve is built from the saved `top_cgw_breakdowns` metadata in
# `summary.pkl.gz`, where `per_pulsar_rho_sq` stores the per-pulsar SNR^2
# contribution for the loudest binary in that simulation.
#
# Layout: forest-plot style sized for a full-page ApJ single-column figure
# (7.1 x 9.0 inches, 300 dpi). All pulsars are listed top-to-bottom ranked
# by median per-pulsar rho^2. Each row shows a median circle and a 95%
# credible-interval error bar. Font size is derived from figure height so
# rows never overlap.

import gzip
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D


# ---------------------------------------------------------------------------
# ApJ figure geometry  (single-column full-page)
# ---------------------------------------------------------------------------
APJ_WIDTH   = 7.1    # inches  (single-column text width)
APJ_HEIGHT  = 9.0    # inches  (safe full-page plot height)
APJ_DPI     = 300

# Fraction of the figure height dedicated to the plot area (axes), after
# reserving space for title (~0.25 in), xlabel (~0.30 in) and legend (~0.20 in).
_RESERVED_INCHES = 0.75   # title + xlabel + legend + breathing room


# ---------------------------------------------------------------------------
# Path / data helpers
# ---------------------------------------------------------------------------

def _resolve_summary_path(source_value):
    if source_value is None or pd.isna(source_value):
        return None
    source_path = Path(str(source_value))
    if source_path.is_dir():
        candidate = source_path / 'summary.pkl.gz'
        return candidate if candidate.is_file() else None
    if source_path.name == 'summary.pkl.gz' and source_path.is_file():
        return source_path
    candidate = source_path.parent / 'summary.pkl.gz'
    return candidate if candidate.is_file() else None


def _load_loudest_breakdown(summary_path, scenario_name):
    summary_path = Path(summary_path)
    if not summary_path.is_file():
        return None

    with gzip.open(summary_path, 'rb') as fh:
        payload = pickle.load(fh)

    top_breakdowns = payload.get('meta', {}).get('top_cgw_breakdowns', []) or []
    if not top_breakdowns:
        return None

    loudest = max(top_breakdowns, key=lambda item: float(item.get('cgw_snr', np.nan)))
    per_pulsar = loudest.get('per_pulsar_rho_sq', {}) or {}
    if not per_pulsar:
        return None

    ordered = sorted(per_pulsar.items(), key=lambda kv: float(kv[1]), reverse=True)
    rho_sq  = np.array([max(float(v), 0.0) for _, v in ordered], dtype=float)
    total   = np.sum(rho_sq)
    cumulative_snr = (
        np.sqrt(np.cumsum(rho_sq)) / np.sqrt(total) if total > 0
        else np.zeros_like(rho_sq)
    )

    return {
        'scenario':        scenario_name,
        'summary_path':    summary_path,
        'sim_key':         summary_path.parent.name,
        'loudest_cgw_snr': float(loudest.get('cgw_snr', np.nan)),
        'pulsar_names':    [name for name, _ in ordered],
        'rho_sq':          rho_sq,
        'cumulative_snr':  cumulative_snr,
    }


def _pad_curves(curves):
    if not curves:
        return np.empty((0, 0), dtype=float)
    max_len = max(len(c) for c in curves)
    padded  = np.full((len(curves), max_len), np.nan, dtype=float)
    for i, c in enumerate(curves):
        if len(c) == 0:
            continue
        padded[i, :len(c)] = c
        padded[i, len(c):] = c[-1]
    return padded


# ---------------------------------------------------------------------------
# Pulsar ordering — unique, by median rank across simulations
# ---------------------------------------------------------------------------

def _median_pulsar_order(scenario_rows, n_pulsars):
    """
    Return a list of exactly n_pulsars *unique* pulsar names ordered by each
    pulsar's median rank across simulations (rank 1 = highest rho^2).

    For every pulsar seen in any simulation we record the 0-based rank at
    which it appears (or worst_rank if absent). We then sort by median of
    those ranks, guaranteeing no duplicates and full coverage.
    """
    all_name_lists = [row for row in scenario_rows['pulsar_names'].tolist() if row]
    if not all_name_lists:
        return [f'Rank {i + 1}' for i in range(n_pulsars)]

    all_pulsars = set()
    for nl in all_name_lists:
        all_pulsars.update(nl)

    worst_rank   = max(len(nl) for nl in all_name_lists)
    rank_records = {p: [] for p in all_pulsars}
    for nl in all_name_lists:
        seen = {name: idx for idx, name in enumerate(nl)}
        for p in all_pulsars:
            rank_records[p].append(seen.get(p, worst_rank))

    median_rank = {p: np.median(ranks) for p, ranks in rank_records.items()}
    ordered     = sorted(all_pulsars, key=lambda p: median_rank[p])[:n_pulsars]

    while len(ordered) < n_pulsars:
        ordered.append(f'Rank {len(ordered) + 1}')

    return ordered


# ---------------------------------------------------------------------------
# Forest-plot helpers
# ---------------------------------------------------------------------------

def _build_forest_data(padded, bands):
    p50   = bands['p50']
    p2_5  = bands['p2_5']
    p97_5 = bands['p97_5']
    marginal = np.concatenate([[p50[0]], np.diff(p50)])
    return p50, p2_5, p97_5, marginal


def _alpha_ramp(marginal, min_alpha=0.15, threshold=0.003):
    alphas      = np.ones(len(marginal))
    significant = marginal >= threshold
    if not np.any(~significant):
        return alphas
    first_insig = int(np.argmax(~significant))
    n_tail      = len(marginal) - first_insig
    if n_tail > 0:
        alphas[first_insig:] = np.linspace(1.0, min_alpha, n_tail)
    return alphas


def _fontsize_for_height(n_rows, axes_height_inches):
    """
    Choose the largest integer font size (points) such that n_rows of text
    fit within axes_height_inches without vertical overlap.
    One point = 1/72 inch; add ~20 % leading.
    """
    max_pt = (axes_height_inches * 72.0) / (n_rows * 1.20)
    return float(np.clip(max_pt, 4.0, 8.0))


def _draw_forest(ax, pulsar_names, median_cumul, lo95, hi95, alphas, color,
                 fontsize):
    n     = len(pulsar_names)
    y_pos = np.arange(n, dtype=float)

    cap_size = 0.30

    for i in range(n):
        a = float(alphas[i])
        ax.plot(
            [lo95[i], hi95[i]], [y_pos[i], y_pos[i]],
            color=color, alpha=a, linewidth=0.9, solid_capstyle='butt',
        )
        for x_cap in (lo95[i], hi95[i]):
            ax.plot(
                [x_cap, x_cap],
                [y_pos[i] - cap_size, y_pos[i] + cap_size],
                color=color, alpha=a, linewidth=0.9,
            )
        ax.scatter(
            median_cumul[i], y_pos[i],
            s=10, color=color, alpha=a, zorder=3, linewidths=0,
        )

    ax.axvline(1.0, color='gray', linewidth=0.5, linestyle=':', alpha=0.4)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(pulsar_names, fontsize=fontsize)
    ax.set_ylim(n - 0.5, -0.5)
    ax.set_xlim(-0.02, 1.08)
    ax.set_xlabel('Cumulative fraction of total SNR', fontsize=fontsize + 1)
    ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
    ax.tick_params(axis='x', labelsize=fontsize)
    ax.tick_params(axis='y', length=0)
    ax.grid(True, axis='x', alpha=0.2, linewidth=0.5)
    # Subtle alternating row shading to aid reading across the page.
    for i in range(0, n, 2):
        ax.axhspan(i - 0.5, i + 0.5, color='gray', alpha=0.04, linewidth=0)


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

if 'cgw_sim_df' not in globals() or cgw_sim_df.empty:
    print('No cgw_sim_df available yet, so the cumulative SNR plots were skipped.')
else:
    breakdown_records = []
    for _, row in cgw_sim_df.iterrows():
        summary_path  = _resolve_summary_path(row.get('source_file'))
        if summary_path is None:
            continue
        scenario_name = str(row.get('scenario', 'unknown'))
        record        = _load_loudest_breakdown(summary_path, scenario_name)
        if record is None:
            continue
        record['sim_key']     = str(row.get('sim_key', record['sim_key']))
        record['source_file'] = str(row.get('source_file', summary_path))
        breakdown_records.append(record)

    breakdown_df = pd.DataFrame(breakdown_records)

    if breakdown_df.empty:
        print('No loudest-binary breakdowns were found in the loaded summaries.')
    else:
        scenario_values = breakdown_df['scenario'].dropna().unique().tolist()
        if 'scenario_order' in globals():
            scenarios = [s for s in scenario_order if s in scenario_values]
            scenarios.extend(s for s in scenario_values if s not in scenarios)
        else:
            scenarios = sorted(scenario_values)

        color_map = {
            s: color
            for s, color in zip(scenarios, ['lime', 'magenta', 'navy'])
        }
        print(f'Loaded loudest-binary breakdowns for {len(breakdown_df)} simulations.')
        print('Breakdowns by scenario:')
        for s in scenarios:
            print(f'  {s}: {int((breakdown_df["scenario"] == s).sum())}')

        figure_dir = Path('figures')
        figure_dir.mkdir(parents=True, exist_ok=True)

        legend_elements = [
            Line2D([0], [0], marker='o', color='gray', markersize=4,
                   linewidth=0, label='Median'),
            Line2D([0], [0], color='gray', linewidth=0.9,
                   label='95% credible interval'),
        ]

        # ------------------------------------------------------------------
        # One full-page ApJ figure per scenario
        # ------------------------------------------------------------------
        for scenario in scenarios:
            
            scenario_rows = breakdown_df[breakdown_df['scenario'] == scenario].copy()
            curves = [c for c in scenario_rows['cumulative_snr'].tolist() if len(c) > 0]
            if not curves:
                continue

            padded    = _pad_curves(curves)
            n_pulsars = padded.shape[1]
            bands = {
                'p2_5':  np.nanpercentile(padded,  2.5, axis=0),
                'p50':   np.nanmedian(padded,             axis=0),
                'p97_5': np.nanpercentile(padded, 97.5, axis=0),
            }

            median_cumul, lo95, hi95, marginal = _build_forest_data(padded, bands)
            alphas       = _alpha_ramp(marginal)
            pulsar_names = _median_pulsar_order(scenario_rows, n_pulsars)
            color        = color_map[scenario]
            legend_elements_scenario = [
                Line2D([0], [0], marker='o', color=color, markersize=4,
                       linewidth=0, label='Median'),
                Line2D([0], [0], color=color, linewidth=0.9,
                       label='95% credible interval'),
            ]
            axes_height = APJ_HEIGHT - _RESERVED_INCHES
            fontsize    = _fontsize_for_height(n_pulsars, axes_height)

            fig, ax = plt.subplots(figsize=(APJ_WIDTH, APJ_HEIGHT))

            # Tight margins: left wide enough for pulsar names, right minimal.
            # We estimate name width as longest_name * fontsize * 0.60 pt/char,
            # converted to a figure fraction.
            max_name_len   = max((len(n) for n in pulsar_names), default=10)
            left_inches    = max(1.0, max_name_len * fontsize * 0.60 / 72.0)
            left_frac      = left_inches / APJ_WIDTH
            right_frac     = 0.02
            bottom_frac    = _RESERVED_INCHES * 0.40 / APJ_HEIGHT
            top_frac       = 1.0 - _RESERVED_INCHES * 0.35 / APJ_HEIGHT

            fig.subplots_adjust(
                left=left_frac, right=1.0 - right_frac,
                bottom=bottom_frac, top=top_frac,
            )

            _draw_forest(ax, pulsar_names, median_cumul, lo95, hi95,
                         alphas, color, fontsize)

            
            ax.legend(handles=legend_elements_scenario, frameon=False,
                      loc='lower left', fontsize=fontsize + 0.5)

            out_path = figure_dir / f'cgw_cumulative_snr_forest_{scenario}.pdf'
            fig.savefig(out_path, dpi=APJ_DPI, bbox_inches='tight')
            plt.close(fig)
            print(f'  saved {out_path}')

        # ------------------------------------------------------------------
        # Combined figure — side-by-side panels, one per scenario
        # ------------------------------------------------------------------
        scenario_data = []
        for scenario in scenarios:
            scenario_rows = breakdown_df[breakdown_df['scenario'] == scenario].copy()
            curves = [c for c in scenario_rows['cumulative_snr'].tolist() if len(c) > 0]
            if not curves:
                continue
            padded    = _pad_curves(curves)
            n_pulsars = padded.shape[1]
            bands = {
                'p2_5':  np.nanpercentile(padded,  2.5, axis=0),
                'p50':   np.nanmedian(padded,             axis=0),
                'p97_5': np.nanpercentile(padded, 97.5, axis=0),
            }
            p50      = bands['p50']
            marginal = np.concatenate([[p50[0]], np.diff(p50)])
            scenario_data.append({
                'scenario':     scenario,
                'n_pulsars':    n_pulsars,
                'median_cumul': p50,
                'lo95':         bands['p2_5'],
                'hi95':         bands['p97_5'],
                'marginal':     marginal,
                'alphas':       _alpha_ramp(marginal),
                'pulsar_names': _median_pulsar_order(
                    breakdown_df[breakdown_df['scenario'] == scenario], n_pulsars
                ),
                'color':        color_map[scenario],
            })

        if scenario_data:
            n_panels  = len(scenario_data)
            max_n     = max(d['n_pulsars'] for d in scenario_data)

            axes_height = APJ_HEIGHT - _RESERVED_INCHES
            fontsize    = _fontsize_for_height(max_n, axes_height)

            # Width: each panel gets APJ_WIDTH inches; panels share y-labels
            # only on the leftmost panel so others can be narrower.
            panel_width = APJ_WIDTH
            fig_width   = panel_width * n_panels

            fig, axes = plt.subplots(
                1, n_panels,
                figsize=(fig_width, APJ_HEIGHT),
                sharey=False,
            )
            if n_panels == 1:
                axes = [axes]

            max_name_len = max(
                max((len(nm) for nm in d['pulsar_names']), default=10)
                for d in scenario_data
            )
            left_inches  = max(1.0, max_name_len * fontsize * 0.60 / 72.0)
            # Express margins as fractions of the *total* figure width.
            left_frac    = left_inches / fig_width
            right_frac   = 0.01
            bottom_frac  = _RESERVED_INCHES * 0.40 / APJ_HEIGHT
            top_frac     = 1.0 - _RESERVED_INCHES * 0.35 / APJ_HEIGHT
            wspace       = 0.35  # relative gap between panels

            fig.subplots_adjust(
                left=left_frac, right=1.0 - right_frac,
                bottom=bottom_frac, top=top_frac,
                wspace=wspace,
            )

            for ax, d in zip(axes, scenario_data):
                _draw_forest(
                    ax,
                    d['pulsar_names'],
                    d['median_cumul'],
                    d['lo95'],
                    d['hi95'],
                    d['alphas'],
                    d['color'],
                    fontsize,
                )
                ax.set_title(d['scenario'], fontsize=fontsize + 2, pad=4)


            fig.legend(
                handles=legend_elements, frameon=False,
                loc='upper center', bbox_to_anchor=(0.5, 1.0),
                ncol=2, fontsize=fontsize + 0.5,
            )

            combined_out = figure_dir / 'cgw_cumulative_snr_forest_combined.pdf'
            fig.savefig(combined_out, dpi=APJ_DPI, bbox_inches='tight')
            plt.close(fig)
            print(f'  saved {combined_out}')

Loaded loudest-binary breakdowns for 1200 simulations.
Breakdowns by scenario:
  optimistic: 400
  realistic: 400
  pessimistic: 400
  saved figures/cgw_cumulative_snr_forest_optimistic.pdf
  saved figures/cgw_cumulative_snr_forest_realistic.pdf
  saved figures/cgw_cumulative_snr_forest_pessimistic.pdf
  saved figures/cgw_cumulative_snr_forest_combined.pdf


In [138]:
# Thresholded cumulative SNR contribution from the loudest binary in each simulation.
# This mirrors the pre-threshold figure block above, but uses cgw_threshold_sim_df.

if 'cgw_threshold_sim_df' not in globals() or cgw_threshold_sim_df.empty:
    print('No cgw_threshold_sim_df available yet, so the thresholded cumulative SNR plots were skipped.')
else:
    threshold_breakdown_records = []
    for _, row in cgw_threshold_sim_df.iterrows():
        summary_path = _resolve_summary_path(row.get('source_file'))
        if summary_path is None:
            continue

        scenario_name = str(row.get('scenario', 'unknown'))
        record = _load_loudest_breakdown(summary_path, scenario_name)
        if record is None:
            continue

        record['sim_key'] = str(row.get('sim_key', record['sim_key']))
        record['source_file'] = str(row.get('source_file', summary_path))
        threshold_breakdown_records.append(record)

    threshold_breakdown_df = pd.DataFrame(threshold_breakdown_records)

    if threshold_breakdown_df.empty:
        print('No thresholded loudest-binary breakdowns were found in the loaded summaries.')
    else:
        threshold_scenarios = sorted(threshold_breakdown_df['scenario'].dropna().unique().tolist())
        threshold_color_map = {s: color
                                    for s, color in zip(scenarios, ['lime', 'magenta', 'navy'])
                                }
        threshold_line_style_map = {
            scenario: style
            for scenario, style in zip(threshold_scenarios, ['-', '--', '-.', ':'])
        }
        while len(threshold_line_style_map) < len(threshold_scenarios):
            threshold_line_style_map[threshold_scenarios[len(threshold_line_style_map)]] = '-'

        print(f'Loaded thresholded loudest-binary breakdowns for {len(threshold_breakdown_df)} simulations.')
        print('Thresholded breakdowns by scenario:')
        for scenario in threshold_scenarios:
            scenario_count = int((threshold_breakdown_df['scenario'] == scenario).sum())
            print(f'  {scenario}: {scenario_count}')

        figure_dir = Path('figures')
        figure_dir.mkdir(parents=True, exist_ok=True)

        # One figure per configuration.
        for scenario in threshold_scenarios:
            scenario_rows = threshold_breakdown_df[threshold_breakdown_df['scenario'] == scenario].copy()
            if scenario_rows.empty:
                continue

            curves = [curve for curve in scenario_rows['cumulative_snr'].tolist() if len(curve) > 0]
            if not curves:
                continue

            padded = _pad_curves(curves)
            median_curve = np.nanmedian(padded, axis=0)
            x_vals = np.arange(1, len(median_curve) + 1)

            fig, ax = plt.subplots(figsize=(9.5, 6.0))
            color = threshold_color_map[scenario]
            line_style = threshold_line_style_map[scenario]

            for curve in curves:
                ax.plot(
                    np.arange(1, len(curve) + 1),
                    curve,
                    color=color,
                    linestyle=line_style,
                    alpha=0.12,
                    linewidth=1.0,
                )

            ax.plot(
                x_vals,
                median_curve,
                color=color,
                linestyle=line_style,
                alpha=0.95,
                linewidth=2.8,
                label=f'{scenario} median',
            )

            ax.set_title(f'Thresholded cumulative loudest-binary SNR fraction by pulsar rank: {scenario}', fontsize=13)
            ax.set_xlabel('Pulsar rank after sorting by per-pulsar $\\rho^2$ contribution')
            ax.set_ylabel('Fraction of total SNR')
            ax.grid(True, alpha=0.25)
            ax.legend(frameon=False)
            fig.tight_layout()

            out_path = figure_dir / f'cgw_cumulative_snr_fraction_by_pulsar_rank_threshold_{scenario}.pdf'
            fig.savefig(out_path, dpi=300, bbox_inches='tight')
            plt.close(fig)
            print(f'  saved {out_path}')

        # Combined figure with all configurations on the same axes.
        fig, ax = plt.subplots(figsize=(10.5, 6.5))
        legend_handles = []

        for scenario in threshold_scenarios:
            scenario_rows = threshold_breakdown_df[threshold_breakdown_df['scenario'] == scenario].copy()
            if scenario_rows.empty:
                continue

            curves = [curve for curve in scenario_rows['cumulative_snr'].tolist() if len(curve) > 0]
            if not curves:
                continue

            padded = _pad_curves(curves)
            median_curve = np.nanmedian(padded, axis=0)
            x_vals = np.arange(1, len(median_curve) + 1)
            color = threshold_color_map[scenario]
            line_style = threshold_line_style_map[scenario]

            for curve in curves:
                ax.plot(
                    np.arange(1, len(curve) + 1),
                    curve,
                    color=color,
                    linestyle=line_style,
                    alpha=0.08,
                    linewidth=0.9,
                )

            ax.plot(
                x_vals,
                median_curve,
                color=color,
                linestyle=line_style,
                alpha=0.98,
                linewidth=3.0,
            )
            legend_handles.append(
                Line2D(
                    [0], [0],
                    color=color,
                    linestyle=line_style,
                    linewidth=3.0,
                    label=f'{scenario} median',
                )
            )

        ax.set_title('Thresholded cumulative loudest-binary SNR fraction by pulsar rank across all configurations', fontsize=13)
        ax.set_xlabel('Pulsar rank after sorting by per-pulsar $\\rho^2$ contribution')
        ax.set_ylabel('Fraction of total SNR')
        ax.grid(True, alpha=0.25)
        if legend_handles:
            ax.legend(handles=legend_handles, frameon=False, loc='best')
        fig.tight_layout()

        combined_out = figure_dir / 'cgw_cumulative_snr_fraction_by_pulsar_rank_combined_threshold.pdf'
        fig.savefig(combined_out, dpi=300, bbox_inches='tight')
        plt.close(fig)
        print(f'  saved {combined_out}')

Loaded thresholded loudest-binary breakdowns for 409 simulations.
Thresholded breakdowns by scenario:
  optimistic: 171
  pessimistic: 103
  realistic: 135
  saved figures/cgw_cumulative_snr_fraction_by_pulsar_rank_threshold_optimistic.pdf
  saved figures/cgw_cumulative_snr_fraction_by_pulsar_rank_threshold_pessimistic.pdf
  saved figures/cgw_cumulative_snr_fraction_by_pulsar_rank_threshold_realistic.pdf
  saved figures/cgw_cumulative_snr_fraction_by_pulsar_rank_combined_threshold.pdf


---
## Synthetic PTA Analysis

The following sections analyse the **synthetic (upgraded) PTA scenarios** produced by Stage 2.
Each summary file may contain additional SNR fields `cgw_snr_<scenario>` alongside the baseline `cgw_snr`.
We load these extra fields, pair each simulation's synthetic-PTA loudest source against its baseline loudest source,
and produce:

1. **SNR improvement histograms** — Δρ (or ρ_syn / ρ_baseline) across simulations, one histogram per synthetic scenario, coloured by population scenario.
2. **Binary parameter distributions** for the loudest source in each synthetic-PTA scenario.
3. **Detection-fraction gain** — how many additional simulations cross the SNR threshold with each upgraded PTA.
4. **Correlation / scatter plots** — baseline vs synthetic SNR to characterise which sources benefit most.
5. **Sky-map of loudest sources** per scenario.

In [139]:
# ============================================================
# SYNTHETIC PTA — STEP 1: discover extra SNR fields
# ============================================================
import gzip, pickle, re
import numpy as np
import pandas as pd
from pathlib import Path

def _discover_synthetic_snr_fields(result_files, max_probe=30):
    """Scan summary files to find all cgw_snr_<scenario> array keys."""
    found = set()
    for fp in result_files[:max_probe]:
        try:
            payload = _load_payload(fp, verbose=False)
            if not isinstance(payload, dict):
                continue
            arrays = payload.get('arrays', {})
            if not isinstance(arrays, dict):
                continue
            for k in arrays:
                if k.startswith('cgw_snr_'):
                    found.add(k)
        except Exception:
            pass
    return sorted(found)


synthetic_snr_fields = _discover_synthetic_snr_fields(result_files)
synthetic_pta_labels = [f[len('cgw_snr_'):] for f in synthetic_snr_fields]

print(f'Discovered synthetic PTA SNR fields ({len(synthetic_snr_fields)}):',
      synthetic_snr_fields if synthetic_snr_fields else '  none')
if not synthetic_snr_fields:
    print('\n  NOTE: No synthetic PTA SNR fields found in any summary file.')
    print('  Run Stage 2 with --synthetic-ptas to generate them.')
    print('  The cells below will print warnings but not crash.')

Discovered synthetic PTA SNR fields (3): ['cgw_snr_4x_precision', 'cgw_snr_5x_cad_4x_prec', 'cgw_snr_5x_cadence']


In [140]:
# ============================================================
# SYNTHETIC PTA — STEP 2: build per-simulation summary table
# ============================================================

def build_synthetic_pta_sim_table(result_files, synthetic_snr_fields, verbose=False):
    """
    Iterate over summary files and, for each synthetic SNR field, record
    the loudest binary's SNR and its physical parameters.
    Returns a DataFrame with one row per simulation.
    """
    if not synthetic_snr_fields:
        return pd.DataFrame()

    records = []

    for fp in result_files:
        try:
            scenario  = infer_scenario(fp)
            run_id    = infer_run_id(fp)
            run_scope = _infer_run_scope_from_path(fp)
            sim_name  = fp.parent.name
            sim_index = _infer_sim_index_from_path(fp)
            sim_key   = f'{run_scope}/{sim_name}' if run_scope else sim_name

            payload = _load_payload(fp, verbose=False)
            if not isinstance(payload, dict) or 'arrays' not in payload:
                continue
            arrays = payload['arrays']
            if not isinstance(arrays, dict) or 'f' not in arrays:
                continue

            n = len(arrays['f'])

            # Baseline loudest
            base_snr_arr     = np.asarray(arrays.get('cgw_snr', np.zeros(n)), dtype=float)
            base_loudest_idx = int(np.nanargmax(base_snr_arr))
            base_snr         = float(base_snr_arr[base_loudest_idx])

            row = {
                'scenario':   scenario,
                'run_id':     run_id,
                'run_scope':  run_scope,
                'sim_name':   sim_name,
                'sim_key':    sim_key,
                'sim_index':  int(sim_index),
                'source_file': str(fp),
                'loudest_cgw_snr': base_snr,
            }

            def _param(key, idx):
                arr = arrays.get(key)
                if arr is not None and len(arr) > idx:
                    return float(arr[idx])
                return np.nan

            for param in ('f', 'Mc', 'D_comov', 'h0', 'z', 'ra', 'dec', 'psi', 'iota', 'Mtot'):
                row[f'baseline_{param}'] = _param(param, base_loudest_idx)

            # Synthetic PTA fields
            for field in synthetic_snr_fields:
                syn_arr = np.asarray(arrays.get(field, np.zeros(n)), dtype=float)
                if np.all(syn_arr == 0):
                    row[f'loudest_{field}'] = np.nan
                    for param in ('f', 'Mc', 'D_comov', 'h0', 'z', 'ra', 'dec'):
                        row[f'{field}_{param}'] = np.nan
                else:
                    syn_idx = int(np.nanargmax(syn_arr))
                    row[f'loudest_{field}'] = float(syn_arr[syn_idx])
                    for param in ('f', 'Mc', 'D_comov', 'h0', 'z', 'ra', 'dec'):
                        row[f'{field}_{param}'] = _param(param, syn_idx)

            records.append(row)

        except Exception as e:
            if verbose:
                print(f'  Error processing {fp}: {e}')

    return pd.DataFrame(records) if records else pd.DataFrame()


print('Building synthetic PTA simulation table...')
synpta_sim_df = build_synthetic_pta_sim_table(result_files, synthetic_snr_fields, verbose=False)

if synpta_sim_df.empty:
    print('  ✗ No synthetic PTA data found — table is empty.')
else:
    print(f'  ✓ {len(synpta_sim_df)} simulations × {len(synthetic_snr_fields)} synthetic scenarios')
    print('  Synthetic scenario labels:', synthetic_pta_labels)
    print('  Population scenarios:', sorted(synpta_sim_df['scenario'].unique().tolist()))
    print(synpta_sim_df[['scenario', 'sim_key', 'loudest_cgw_snr'] +
                         [f'loudest_{f}' for f in synthetic_snr_fields]].head(5).to_string())

Building synthetic PTA simulation table...
  ✓ 1200 simulations × 3 synthetic scenarios
  Synthetic scenario labels: ['4x_precision', '5x_cad_4x_prec', '5x_cadence']
  Population scenarios: ['optimistic', 'pessimistic', 'realistic']
     scenario                       sim_key  loudest_cgw_snr  loudest_cgw_snr_4x_precision  loudest_cgw_snr_5x_cad_4x_prec  loudest_cgw_snr_5x_cadence
0  optimistic  2026-06-05_optimistic/sim000        13.384206                     16.807667                       22.255690                   18.195097
1  optimistic  2026-06-05_optimistic/sim002         3.299688                      3.977569                        7.287083                    4.171178
2  optimistic  2026-06-05_optimistic/sim003        15.722937                     15.721641                       15.721719                   15.721663
3  optimistic  2026-06-05_optimistic/sim004        11.315403                     11.817843                       16.536543                   12.018096
4  optimisti

In [141]:
# ============================================================
# SYNTHETIC PTA — STEP 3: compute SNR ratio & delta columns
# ============================================================

if not synpta_sim_df.empty:
    for field, label in zip(synthetic_snr_fields, synthetic_pta_labels):
        syn_col  = f'loudest_{field}'
        base_col = 'loudest_cgw_snr'
        synpta_sim_df[f'snr_ratio_{label}'] = (
            synpta_sim_df[syn_col] / synpta_sim_df[base_col].replace(0, np.nan)
        )
        synpta_sim_df[f'snr_delta_{label}'] = (
            synpta_sim_df[syn_col] - synpta_sim_df[base_col]
        )

    print('Added SNR ratio and delta columns.')
    ratio_cols = [f'snr_ratio_{l}' for l in synthetic_pta_labels]
    delta_cols = [f'snr_delta_{l}' for l in synthetic_pta_labels]
    print(synpta_sim_df[['scenario', 'loudest_cgw_snr'] + ratio_cols].describe().round(3).to_string())
else:
    print('synpta_sim_df is empty — skipping ratio computation.')

Added SNR ratio and delta columns.
       loudest_cgw_snr  snr_ratio_4x_precision  snr_ratio_5x_cad_4x_prec  snr_ratio_5x_cadence
count         1200.000                1200.000                  1200.000              1200.000
mean            11.439                   1.128                     1.508                 1.156
std             38.870                   0.160                     0.593                 0.186
min              0.056                   0.852                     0.917                 0.852
25%              1.780                   1.000                     1.014                 1.000
50%              3.754                   1.059                     1.320                 1.076
75%              7.904                   1.216                     1.784                 1.266
max            559.733                   2.185                     4.314                 1.903


### 1 — SNR improvement: histograms across simulations

Each subplot shows the distribution of the **loudest-source SNR** for one PTA configuration
across all simulations.  Colours correspond to the three SMBHB population scenarios.
The left column shows **absolute SNR**, the right column shows **SNR ratio** (synthetic / baseline).

In [142]:
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import matplotlib.patches as mpatches
import numpy as np
from pathlib import Path

CGW_SNR_THRESHOLD = 5.94

figure_dir = Path('figures')
figure_dir.mkdir(parents=True, exist_ok=True)

POP_SCENARIO_STYLES = {
    'optimistic':  {'color': '#0072B2', 'linestyle': '-',  'linewidth': 2.0},
    'realistic':   {'color': '#E69F00', 'linestyle': '--', 'linewidth': 2.0},
    'pessimistic': {'color': '#D55E00', 'linestyle': ':',  'linewidth': 2.2},
}
SYN_COLOURS = ['#009E73', '#CC79A7', '#F0E442', '#56B4E9', '#999999']

if synpta_sim_df.empty or not synthetic_pta_labels:
    print('No synthetic PTA data available — skipping SNR improvement histograms.')
else:
    pop_scenarios = [s for s in ('optimistic', 'realistic', 'pessimistic')
                     if s in synpta_sim_df['scenario'].unique()]
    n_syn = len(synthetic_pta_labels)

    # ── Figure A: one row per synthetic scenario ──────────────────────────────
    fig, axes = plt.subplots(n_syn, 2, figsize=(7.0, 2.8 * n_syn), squeeze=False)

    for row_i, (label, field) in enumerate(zip(synthetic_pta_labels, synthetic_snr_fields)):
        ax_abs   = axes[row_i][0]
        ax_ratio = axes[row_i][1]

        abs_vals_by_pop   = {}
        ratio_vals_by_pop = {}
        for pop in pop_scenarios:
            sub = synpta_sim_df[synpta_sim_df['scenario'] == pop]
            a = sub[f'loudest_{field}'].to_numpy(float)
            r = sub[f'snr_ratio_{label}'].to_numpy(float)
            abs_vals_by_pop[pop]   = a[np.isfinite(a) & (a > 0)]
            ratio_vals_by_pop[pop] = r[np.isfinite(r) & (r > 0)]

        all_abs = np.concatenate(list(abs_vals_by_pop.values()))
        if all_abs.size > 0:
            bins_abs = np.logspace(np.log10(max(all_abs.min(), 1e-3)), np.log10(all_abs.max()), 30)
            for pop in pop_scenarios:
                st = POP_SCENARIO_STYLES.get(pop, {})
                if abs_vals_by_pop[pop].size:
                    ax_abs.hist(abs_vals_by_pop[pop], bins=bins_abs, histtype='step', label=pop,
                                color=st.get('color','#888'), linestyle=st.get('linestyle','-'),
                                linewidth=st.get('linewidth',1.5), density=False)
            ax_abs.axvline(CGW_SNR_THRESHOLD, color='black', linestyle='--', linewidth=1.5,
                           label=f'threshold ({CGW_SNR_THRESHOLD})')
            ax_abs.set_xscale('log')
            ax_abs.set_xlabel(r'Loudest-source $\rho$ (synthetic PTA)', fontsize=10)
            ax_abs.set_ylabel('N simulations', fontsize=9)
            ax_abs.set_title(label, fontsize=10, style='italic')

        all_ratio = np.concatenate(list(ratio_vals_by_pop.values()))
        if all_ratio.size > 0:
            bins_ratio = np.logspace(np.log10(max(all_ratio.min(), 1e-2)), np.log10(all_ratio.max()), 30)
            for pop in pop_scenarios:
                st = POP_SCENARIO_STYLES.get(pop, {})
                if ratio_vals_by_pop[pop].size:
                    ax_ratio.hist(ratio_vals_by_pop[pop], bins=bins_ratio, histtype='step', label=pop,
                                  color=st.get('color','#888'), linestyle=st.get('linestyle','-'),
                                  linewidth=st.get('linewidth',1.5), density=False)
            ax_ratio.axvline(1.0, color='black', linestyle='--', linewidth=1.5, label='ratio = 1')
            ax_ratio.set_xscale('log')
            ax_ratio.set_xlabel(r'SNR ratio $\rho_{\rm syn} / \rho_{\rm baseline}$', fontsize=10)
            ax_ratio.set_ylabel('N simulations', fontsize=9)
            ax_ratio.set_title(f'{label} (ratio)', fontsize=10, style='italic')

        if row_i == 0:
            handles_pop = [
                mlines.Line2D([], [], color=POP_SCENARIO_STYLES.get(p,{}).get('color','k'),
                              linestyle=POP_SCENARIO_STYLES.get(p,{}).get('linestyle','-'),
                              linewidth=2, label=p)
                for p in pop_scenarios
            ]
            ax_abs.legend(handles=handles_pop, frameon=False, fontsize=8, loc='upper right')
            ax_ratio.legend(frameon=False, fontsize=7, loc='upper right')

    plt.tight_layout()
    fig.savefig(figure_dir / 'synpta_snr_improvement_panels.pdf', dpi=300, bbox_inches='tight')
    plt.show(); plt.close(fig)
    print(f'  saved synpta_snr_improvement_panels.pdf')

    # ── Figure B: all synthetic scenarios overlaid, coloured by scenario label ─
    fig2, (ax_l, ax_r) = plt.subplots(1, 2, figsize=(7.0, 3.2))

    for i, (label, field) in enumerate(zip(synthetic_pta_labels, synthetic_snr_fields)):
        colour = SYN_COLOURS[i % len(SYN_COLOURS)]
        a = synpta_sim_df[f'loudest_{field}'].to_numpy(float)
        r = synpta_sim_df[f'snr_ratio_{label}'].to_numpy(float)
        a = a[np.isfinite(a) & (a > 0)]
        r = r[np.isfinite(r) & (r > 0)]
        if a.size:
            ax_l.hist(a, bins=np.logspace(np.log10(max(a.min(),1e-3)), np.log10(a.max()), 25),
                      histtype='step', color=colour, linewidth=1.8, label=label, density=True)
        if r.size:
            ax_r.hist(r, bins=np.logspace(np.log10(max(r.min(),1e-2)), np.log10(r.max()), 25),
                      histtype='step', color=colour, linewidth=1.8, label=label, density=True)

    base_vals = synpta_sim_df['loudest_cgw_snr'].to_numpy(float)
    base_vals = base_vals[np.isfinite(base_vals) & (base_vals > 0)]
    if base_vals.size:
        ax_l.hist(base_vals, bins=np.logspace(np.log10(max(base_vals.min(),1e-3)),
                  np.log10(base_vals.max()), 25), histtype='step', color='black',
                  linewidth=2.0, linestyle='--', label='baseline', density=True)

    ax_l.axvline(CGW_SNR_THRESHOLD, color='gray', linestyle=':', linewidth=1.5)
    ax_l.set_xscale('log')
    ax_l.set_xlabel(r'Loudest-source SNR $\rho$', fontsize=11)
    ax_l.set_ylabel('Probability density', fontsize=9)
    ax_l.legend(frameon=False, fontsize=8)

    ax_r.axvline(1.0, color='gray', linestyle=':', linewidth=1.5)
    ax_r.set_xscale('log')
    ax_r.set_xlabel(r'SNR ratio $\rho_{\rm syn} / \rho_{\rm baseline}$', fontsize=11)
    ax_r.set_ylabel('Probability density', fontsize=9)
    ax_r.legend(frameon=False, fontsize=8)

    plt.tight_layout()
    fig2.savefig(figure_dir / 'synpta_snr_ratio_combined.pdf', dpi=300, bbox_inches='tight')
    plt.show(); plt.close(fig2)
    print(f'  saved synpta_snr_ratio_combined.pdf')

  saved synpta_snr_improvement_panels.pdf
  saved synpta_snr_ratio_combined.pdf


### 2 — Binary parameters of the loudest source in each synthetic PTA

Histograms of the physical parameters ($f$, $\mathcal{M}_c$, $h_0$, $D_{\rm comov}$, inclination $\iota$)
of the **loudest** source in each simulation, separated by synthetic PTA scenario.
This mirrors the equivalent block for the baseline PTA already in the notebook.

In [143]:
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

figure_dir = Path('figures')

PARAM_SPECS = [
    ('f',       r'GW frequency $f$ [Hz]',                  True),
    ('Mc',      r'Chirp mass $\mathcal{M}_c$ [$M_\odot$]', True),
    ('h0',      r'Strain amplitude $h_0$',                  True),
    ('D_comov', r'Comoving distance $D$ [Mpc]',             True),
    ('iota',    r'Inclination $\iota$ [rad]',               False),
]

if synpta_sim_df.empty or not synthetic_pta_labels:
    print('No synthetic PTA data — skipping binary parameter plots.')
else:
    pop_scenarios = [s for s in ('optimistic', 'realistic', 'pessimistic')
                     if s in synpta_sim_df['scenario'].unique()]

    for label, field in zip(synthetic_pta_labels, synthetic_snr_fields):
        n_params = len(PARAM_SPECS)
        fig, axes = plt.subplots(1, n_params, figsize=(3.2 * n_params, 2.8), squeeze=True)

        for ax, (param_key, xlabel, use_log) in zip(axes, PARAM_SPECS):
            col      = f'{field}_{param_key}'
            base_col = f'baseline_{param_key}'

            all_vals_list = []
            for pop in pop_scenarios:
                sub = synpta_sim_df[synpta_sim_df['scenario'] == pop]
                if col in sub.columns:
                    v = sub[col].to_numpy(float)
                    all_vals_list.append(v[np.isfinite(v)])
            if base_col in synpta_sim_df.columns:
                vb = synpta_sim_df[base_col].to_numpy(float)
                all_vals_list.append(vb[np.isfinite(vb)])

            all_vals = np.concatenate(all_vals_list) if all_vals_list else np.array([])
            if all_vals.size == 0 or col not in synpta_sim_df.columns:
                ax.set_visible(False)
                continue

            bins = (np.logspace(np.log10(all_vals.min()), np.log10(all_vals.max()), 25)
                    if use_log and all_vals.min() > 0
                    else np.linspace(all_vals.min(), all_vals.max(), 25))

            if base_col in synpta_sim_df.columns:
                bv = synpta_sim_df[base_col].to_numpy(float)
                bv = bv[np.isfinite(bv)]
                if bv.size:
                    ax.hist(bv, bins=bins, histtype='step', color='black',
                            linestyle='--', linewidth=1.5, label='baseline', density=True)

            for pop in pop_scenarios:
                sub = synpta_sim_df[synpta_sim_df['scenario'] == pop]
                if col not in sub.columns:
                    continue
                v = sub[col].to_numpy(float)
                v = v[np.isfinite(v)]
                if v.size == 0:
                    continue
                st = POP_SCENARIO_STYLES.get(pop, {'color': '#888', 'linestyle': '-', 'linewidth': 1.5})
                ax.hist(v, bins=bins, histtype='step', color=st['color'],
                        linestyle=st['linestyle'], linewidth=st['linewidth'],
                        label=pop, density=True)

            if use_log and all_vals.min() > 0:
                ax.set_xscale('log')
            ax.set_xlabel(xlabel, fontsize=9)
            ax.set_ylabel('Density', fontsize=8)
            ax.tick_params(labelsize=7)
            ax.spines['top'].set_visible(True)
            ax.spines['right'].set_visible(True)

        axes[0].legend(frameon=False, fontsize=7, loc='upper right')
        fig.suptitle(f'Loudest binary parameters — synthetic PTA: {label}', fontsize=11, y=1.01)
        plt.tight_layout()
        fname = figure_dir / f'synpta_loudest_params_{label}.pdf'
        fig.savefig(fname, dpi=300, bbox_inches='tight')
        plt.show(); plt.close(fig)
        print(f'  saved {fname}')

  saved figures/synpta_loudest_params_4x_precision.pdf
  saved figures/synpta_loudest_params_5x_cad_4x_prec.pdf
  saved figures/synpta_loudest_params_5x_cadence.pdf


### 3 — Detection-fraction gain

How many additional simulations cross the SNR detection threshold with each upgraded PTA?
Reported as absolute counts and fractional gain relative to the baseline detection fraction.

In [144]:
_SYNPTA_POP_STYLES = {
    'optimistic':  {'color': 'lime',    'linestyle': '-',  'linewidth': 2.2},
    'realistic':   {'color': 'magenta', 'linestyle': '--', 'linewidth': 2.2},
    'pessimistic': {'color': 'navy',    'linestyle': ':',  'linewidth': 2.5},
}

_SYNPTA_LABEL_MAP = {
    'optimistic':  r'$m_{\mathrm{char}} = 10^{9.3}\;\mathrm{M}_{\odot}$',
    'realistic':   r'$m_{\mathrm{char}} = 10^{9.0}\;\mathrm{M}_{\odot}$',
    'pessimistic': r'$m_{\mathrm{char}} = 10^{8.7}\;\mathrm{M}_{\odot}$',
}

_BASELINE_COL_MAP = {
    'f':       'loudest_f',
    'Mc':      'loudest_Mc',
    'h0':      'loudest_h0',
    'D_comov': 'loudest_D',
    'iota':    None,
    'snr':     'loudest_cgw_snr',   # ← key used below for the SNR param
}

PARAM_SPECS = [
    ('f',       r'$f$ [Hz]',                  True),
    ('Mc',      r'$\mathcal{M}_c$ [$M_\odot$]', True),
    ('h0',      r'$h_0$',                  True),
    ('D_comov', r'$D_{\rm{comov}}$ [Mpc]',             True),
    ('iota',    r'$\iota$ [rad]',               False),
    ('snr',     r'$(\mathrm{S/N})_s$',                                     True),   # ← renamed key
]


In [145]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path

figure_dir = Path('figures')

if synpta_sim_df.empty or not synthetic_pta_labels:
    print('No synthetic PTA data — skipping detection fraction analysis.')
else:
    pop_scenarios = [s for s in ('optimistic', 'realistic', 'pessimistic')
                     if s in synpta_sim_df['scenario'].unique()]

    det_records = []
    for pop in pop_scenarios:
        sub     = synpta_sim_df[synpta_sim_df['scenario'] == pop]
        n_total = len(sub)
        if n_total == 0:
            continue
        base_above    = (sub['loudest_cgw_snr'] >= CGW_SNR_THRESHOLD).sum()
        base_fraction = base_above / n_total
        det_records.append({
            'scenario': pop, 'pta': 'baseline',
            'n_total': n_total, 'n_detected': int(base_above),
            'det_fraction': base_fraction,
            'gain_abs': 0,
            'gain_frac': 0.0,
        })
        for label, field in zip(synthetic_pta_labels, synthetic_snr_fields):
            syn_col = f'loudest_{field}'
            if syn_col not in sub.columns:
                continue
            syn_above    = (sub[syn_col] >= CGW_SNR_THRESHOLD).sum()
            syn_fraction = syn_above / n_total
            det_records.append({
                'scenario': pop, 'pta': label,
                'n_total': n_total, 'n_detected': int(syn_above),
                'det_fraction': syn_fraction,
                'gain_abs':  int(syn_above - base_above),
                'gain_frac': (syn_above - base_above) / base_above if base_above > 0 else np.nan,
            })

    det_df = pd.DataFrame(det_records)

    print(f'Detection threshold: SNR ≥ {CGW_SNR_THRESHOLD}')
    print(det_df.to_string(index=False))

    pta_labels_all = ['baseline'] + synthetic_pta_labels
    n_pta  = len(pta_labels_all)
    n_pop  = len(pop_scenarios)
    x_all  = np.arange(n_pta)
    x_syn  = np.arange(len(synthetic_pta_labels))   # right panel: synthetic only
    width  = 0.8 / n_pop

    fig, axes = plt.subplots(1, 2, figsize=(max(5.5, 2.5 * n_pta), 3.5))

    for pi, pop in enumerate(pop_scenarios):
        sub = det_df[det_df['scenario'] == pop].set_index('pta')
        st  = _SYNPTA_POP_STYLES.get(pop, {'color': '#888'})
        offset = (pi - n_pop / 2 + 0.5) * width

        # left panel: detection fraction for all PTAs including baseline
        fracs = [float(sub.loc[p, 'det_fraction']) if p in sub.index else 0
                 for p in pta_labels_all]
        axes[0].bar(x_all + offset, fracs, width, color=st['color'], alpha=0.8,
                    label=_SYNPTA_LABEL_MAP.get(pop, pop))

        # right panel: fractional gain vs baseline, synthetic PTAs only
        gains = [float(sub.loc[p, 'gain_frac']) if p in sub.index else 0
                 for p in synthetic_pta_labels]
        axes[1].bar(x_syn + offset, gains, width, color=st['color'], alpha=0.8,
                    label=_SYNPTA_LABEL_MAP.get(pop, pop))

    axes[0].set_xticks(x_all)
    axes[0].set_xticklabels(pta_labels_all, rotation=20, ha='right', fontsize=8)
    axes[0].set_ylabel('Detection fraction', fontsize=10)
    axes[0].set_ylim(0, 1)

    axes[1].set_xticks(x_syn)
    axes[1].set_xticklabels(synthetic_pta_labels, rotation=20, ha='right', fontsize=8)
    axes[1].set_ylabel(r'$\Delta$ detections / baseline detections', fontsize=10)
    axes[1].axhline(0, color='black', linewidth=0.8)

    for ax in axes:
        ax.legend(frameon=False, fontsize=7)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    plt.tight_layout()
    fig.savefig(figure_dir / 'synpta_detection_fraction_gain.pdf', dpi=300, bbox_inches='tight')
    plt.show(); plt.close(fig)
    print('  saved synpta_detection_fraction_gain.pdf')

# ── scatter: detection fraction ──────────────────────────────────────────────
_SCATTER_STYLES = {
    'optimistic':  {'color': 'limegreen',  'linestyle': '-',  'marker': 'o',  'linewidth': 1.6, 'ms': 5},
    'realistic':   {'color': 'magenta',    'linestyle': '--', 'marker': 's',  'linewidth': 1.6, 'ms': 5},
    'pessimistic': {'color': 'navy',       'linestyle': ':',  'marker': '^',  'linewidth': 1.8, 'ms': 5},
}

_PTA_DISPLAY_NAMES = {
    'baseline':       'NG15',
    '4x_precision':   '4×_prec',
    '5x_cadence':     '5×_cad',
    '5x_cad_4x_prec': '5×_cad\n_4×_prec',
}

if not det_df.empty:
    syn_order = [p for p in synthetic_pta_labels if p != '5x_cad_4x_prec']
    if '5x_cad_4x_prec' in synthetic_pta_labels:
        syn_order.append('5x_cad_4x_prec')
    pta_order = ['baseline'] + syn_order
    x_pos = np.arange(len(pta_order))

    fig, ax = plt.subplots(figsize=(3.5, 2.8))

    handles = []
    for pop in ('optimistic', 'realistic', 'pessimistic'):
        if pop not in pop_scenarios:
            continue
        sub = det_df[det_df['scenario'] == pop].set_index('pta')
        st  = _SCATTER_STYLES[pop]
        lab = _SYNPTA_LABEL_MAP.get(pop, pop)

        fracs = np.array([float(sub.loc[p, 'det_fraction']) if p in sub.index else np.nan
                          for p in pta_order])

        h, = ax.plot(x_pos, fracs,
                     color=st['color'], linestyle=st['linestyle'],
                     linewidth=st['linewidth'], marker=st['marker'],
                     markersize=st['ms'], label=lab, clip_on=False, zorder=3)
        handles.append(h)

    # x-axis
    tick_labels = [_PTA_DISPLAY_NAMES.get(p, p) for p in pta_order]
    ax.set_xticks(x_pos)
    ax.set_xticklabels(tick_labels, rotation=25, ha='right', fontsize=8)
    ax.set_xlim(-0.5, len(pta_order) - 0.5)

    # y-axis
    ax.set_ylim(0, 0.75)
    ax.set_ylabel('Detection fraction', fontsize=9)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.1f}'))
    ax.yaxis.set_major_locator(plt.MultipleLocator(0.1))

    # close the box
    ax.spines['top'].set_visible(True)
    ax.spines['right'].set_visible(True)

    ax.legend(handles=handles, frameon=False, fontsize=7,
              loc='upper left', handlelength=2.2)

    ax.tick_params(labelsize=8)

    fig.tight_layout()
    fig.savefig(figure_dir / 'synpta_detection_fraction_scatter.pdf',
                dpi=300, bbox_inches='tight')
    plt.show(); plt.close(fig)
    print('  saved synpta_detection_fraction_scatter.pdf')

Detection threshold: SNR ≥ 5.94
   scenario            pta  n_total  n_detected  det_fraction  gain_abs  gain_frac
 optimistic       baseline      400         171        0.4275         0   0.000000
 optimistic   4x_precision      400         190        0.4750        19   0.111111
 optimistic 5x_cad_4x_prec      400         245        0.6125        74   0.432749
 optimistic     5x_cadence      400         193        0.4825        22   0.128655
  realistic       baseline      400         135        0.3375         0   0.000000
  realistic   4x_precision      400         146        0.3650        11   0.081481
  realistic 5x_cad_4x_prec      400         190        0.4750        55   0.407407
  realistic     5x_cadence      400         147        0.3675        12   0.088889
pessimistic       baseline      400         103        0.2575         0   0.000000
pessimistic   4x_precision      400         107        0.2675         4   0.038835
pessimistic 5x_cad_4x_prec      400         130        

In [146]:
# ── LaTeX detection fraction table ───────────────────────────────────────────
pta_display = {
    'baseline':       'NG15',
    '4x_precision':   r'4x\_precision',
    '5x_cadence':     r'5x\_cadence',
    '5x_cad_4x_prec': r'5x\_cad\_4x\_prec',
}

scenario_mass_map = {
    'optimistic':  r'$10^{9.3}$',
    'realistic':   r'$10^{9.0}$',
    'pessimistic': r'$10^{8.7}$',
}

pta_row_order = ['baseline'] + [p for p in synthetic_pta_labels if p != '5x_cad_4x_prec']
if '5x_cad_4x_prec' in synthetic_pta_labels:
    pta_row_order.append('5x_cad_4x_prec')

lines = []
lines.append(r'\begin{table}[htbp]')
lines.append(r'\centering')
lines.append(r'\begin{tabular}{l l c c}')
lines.append(r'\toprule')
lines.append(
    r'$m_{\mathrm{char}}\ [\mathrm{M}_\odot]$'
    r' & PTA'
    r' & \multicolumn{1}{c}{Det.\ fraction}'
    r' & \multicolumn{1}{c}{\shortstack{\% Rel.\\Gain}} \\'
)
lines.append(r'\midrule')

for si, pop in enumerate(('optimistic', 'realistic', 'pessimistic')):
    if pop not in pop_scenarios:
        continue
    sub = det_df[det_df['scenario'] == pop].set_index('pta')
    mass_label = scenario_mass_map[pop]
    n_rows = len(pta_row_order)

    # baseline detection fraction for this scenario
    base_frac = float(sub.loc['baseline', 'det_fraction']) if 'baseline' in sub.index else np.nan

    for ri, pta in enumerate(pta_row_order):
        if pta not in sub.index:
            continue
        frac  = float(sub.loc[pta, 'det_fraction'])
        n_det = int(sub.loc[pta, 'n_detected'])
        n_tot = int(sub.loc[pta, 'n_total'])
        frac_str = f'${frac:.3f}$'

        if pta == 'baseline':
            gain_str = r'---'
        elif not np.isnan(base_frac) and base_frac > 0:
            gain = (frac - base_frac) / base_frac * 100
            gain_str = f'${gain:+.1f}\\%$'
        else:
            gain_str = r'---'

        pta_str  = pta_display.get(pta, pta)
        mass_col = rf'\multirow{{{n_rows}}}{{*}}{{{mass_label}}}' if ri == 0 else ''

        lines.append(rf' {mass_col} & {pta_str} & {frac_str} & {gain_str} \\')

    if si < len(pop_scenarios) - 1:
        lines.append(r'\addlinespace')
        lines.append(r'\hline')

lines.append(r'\bottomrule')
lines.append(r'\end{tabular}')
lines.append(
    r'\caption{Detection fraction (number detected / total simulations) and relative gain '
    r'compared to NG15 for each PTA configuration and population scenario, '
    f'using an SNR threshold of {CGW_SNR_THRESHOLD}' + r'.}'
)
lines.append(r'\label{tab:detection_fraction}')
lines.append(r'\end{table}')

print('\n'.join(lines))

\begin{table}[htbp]
\centering
\begin{tabular}{l l c c}
\toprule
$m_{\mathrm{char}}\ [\mathrm{M}_\odot]$ & PTA & \multicolumn{1}{c}{Det.\ fraction} & \multicolumn{1}{c}{\shortstack{\% Rel.\\Gain}} \\
\midrule
 \multirow{4}{*}{$10^{9.3}$} & NG15 & $0.427$ & --- \\
  & 4x\_precision & $0.475$ & $+11.1\%$ \\
  & 5x\_cadence & $0.482$ & $+12.9\%$ \\
  & 5x\_cad\_4x\_prec & $0.613$ & $+43.3\%$ \\
\addlinespace
\hline
 \multirow{4}{*}{$10^{9.0}$} & NG15 & $0.338$ & --- \\
  & 4x\_precision & $0.365$ & $+8.1\%$ \\
  & 5x\_cadence & $0.367$ & $+8.9\%$ \\
  & 5x\_cad\_4x\_prec & $0.475$ & $+40.7\%$ \\
\addlinespace
\hline
 \multirow{4}{*}{$10^{8.7}$} & NG15 & $0.258$ & --- \\
  & 4x\_precision & $0.268$ & $+3.9\%$ \\
  & 5x\_cadence & $0.278$ & $+7.8\%$ \\
  & 5x\_cad\_4x\_prec & $0.325$ & $+26.2\%$ \\
\bottomrule
\end{tabular}
\caption{Detection fraction (number detected / total simulations) and relative gain compared to NG15 for each PTA configuration and population scenario, using an SNR thr

### 4 — Baseline vs synthetic SNR scatter

For each synthetic scenario a scatter plot shows the loudest-source SNR in the baseline PTA
($x$-axis) against the same quantity in the synthetic PTA ($y$-axis).
Points lying above the diagonal gain SNR with the upgraded array; those below lose it.
Colour encodes the SMBHB population scenario.

In [147]:
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

figure_dir = Path('figures')

if synpta_sim_df.empty or not synthetic_pta_labels:
    print('No synthetic PTA data — skipping scatter plots.')
else:
    pop_scenarios = [s for s in ('optimistic', 'realistic', 'pessimistic')
                     if s in synpta_sim_df['scenario'].unique()]
    n_syn = len(synthetic_pta_labels)
    fig, axes = plt.subplots(1, n_syn, figsize=(3.8 * n_syn, 3.5), squeeze=False)

    for col_i, (label, field) in enumerate(zip(synthetic_pta_labels, synthetic_snr_fields)):
        ax      = axes[0][col_i]
        syn_col = f'loudest_{field}'

        for pop in pop_scenarios:
            sub  = synpta_sim_df[synpta_sim_df['scenario'] == pop]
            x    = sub['loudest_cgw_snr'].to_numpy(float)
            y    = sub[syn_col].to_numpy(float) if syn_col in sub.columns else np.full(len(sub), np.nan)
            mask = np.isfinite(x) & np.isfinite(y) & (x > 0) & (y > 0)
            st   = POP_SCENARIO_STYLES.get(pop, {'color': '#888'})
            ax.scatter(x[mask], y[mask], s=6, alpha=0.4, color=st['color'], label=pop, rasterized=True)

        all_x = synpta_sim_df['loudest_cgw_snr'].to_numpy(float)
        all_y = synpta_sim_df[syn_col].to_numpy(float) if syn_col in synpta_sim_df.columns else all_x
        lim = max(np.nanmax(all_x), np.nanmax(all_y)) * 1.1 if np.any(np.isfinite(all_x)) else 10
        lo  = min(np.nanmin(all_x[all_x > 0]), np.nanmin(all_y[all_y > 0])) * 0.9 \
              if np.any(all_x > 0) else 0.1
        ax.plot([lo, lim], [lo, lim], 'k--', linewidth=1.2, alpha=0.7)
        ax.axhline(CGW_SNR_THRESHOLD, color='grey', linestyle=':', linewidth=1, alpha=0.8)
        ax.axvline(CGW_SNR_THRESHOLD, color='grey', linestyle=':', linewidth=1, alpha=0.8)
        ax.set_xscale('log'); ax.set_yscale('log')
        ax.set_xlim(lo, lim); ax.set_ylim(lo, lim)
        ax.set_xlabel(r'Baseline loudest $\rho$', fontsize=10)
        ax.set_ylabel(fr'Synthetic ({label}) loudest $\rho$', fontsize=10)
        ax.set_title(label, fontsize=9, style='italic')
        if col_i == 0:
            ax.legend(frameon=False, fontsize=8, markerscale=2)

    plt.tight_layout()
    fig.savefig(figure_dir / 'synpta_baseline_vs_synthetic_scatter.pdf', dpi=300, bbox_inches='tight')
    plt.show(); plt.close(fig)
    print('  saved synpta_baseline_vs_synthetic_scatter.pdf')

  saved synpta_baseline_vs_synthetic_scatter.pdf


### 5 — SNR improvement CDF

Cumulative distribution of the **SNR ratio** $\rho_{\rm syn}/\rho_{\rm baseline}$
for each synthetic PTA and each population scenario.  The vertical dashed line marks
ratio = 1 (no improvement); the fraction of simulations to the right shows how often
the upgraded PTA finds a louder source.

In [148]:
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

figure_dir = Path('figures')

if synpta_sim_df.empty or not synthetic_pta_labels:
    print('No synthetic PTA data — skipping CDF plots.')
else:
    pop_scenarios = [s for s in ('optimistic', 'realistic', 'pessimistic')
                     if s in synpta_sim_df['scenario'].unique()]
    n_syn = len(synthetic_pta_labels)
    fig, axes = plt.subplots(1, n_syn, figsize=(3.8 * n_syn, 3.2), squeeze=False)

    for col_i, (label, field) in enumerate(zip(synthetic_pta_labels, synthetic_snr_fields)):
        ax = axes[0][col_i]
        for pop in pop_scenarios:
            sub      = synpta_sim_df[synpta_sim_df['scenario'] == pop]
            r        = sub[f'snr_ratio_{label}'].to_numpy(float)
            r        = r[np.isfinite(r) & (r > 0)]
            if r.size == 0:
                continue
            r_sorted = np.sort(r)
            cdf      = np.arange(1, len(r_sorted) + 1) / len(r_sorted)
            st       = POP_SCENARIO_STYLES.get(pop, {})
            ax.plot(r_sorted, cdf, color=st.get('color','#888'),
                    linestyle=st.get('linestyle','-'), linewidth=st.get('linewidth',1.5), label=pop)

        ax.axvline(1.0, color='black', linestyle='--', linewidth=1.5, label='ratio = 1')
        ax.set_xscale('log')
        ax.set_xlabel(r'SNR ratio $\rho_{\rm syn}/\rho_{\rm baseline}$', fontsize=10)
        ax.set_ylabel('CDF', fontsize=10)
        ax.set_title(label, fontsize=10, style='italic')
        ax.set_ylim(0, 1)
        if col_i == 0:
            ax.legend(frameon=False, fontsize=8, loc='upper left')

    plt.tight_layout()
    fig.savefig(figure_dir / 'synpta_snr_ratio_cdf.pdf', dpi=300, bbox_inches='tight')
    plt.show(); plt.close(fig)
    print('  saved synpta_snr_ratio_cdf.pdf')

  saved synpta_snr_ratio_cdf.pdf


### 6 — Sky distribution of loudest sources in synthetic PTAs

Mollweide projection sky maps showing where the loudest binary lies in each simulation,
colour-coded by SNR.  One map per synthetic PTA scenario (plus baseline for comparison).

In [149]:
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

figure_dir = Path('figures')

if synpta_sim_df.empty or not synthetic_pta_labels:
    print('No synthetic PTA data — skipping sky maps.')
else:
    panels = [('baseline', 'loudest_cgw_snr', 'baseline_ra', 'baseline_dec')] + [
        (label, f'loudest_{field}', f'{field}_ra', f'{field}_dec')
        for label, field in zip(synthetic_pta_labels, synthetic_snr_fields)
    ]
    n_panels = len(panels)
    fig = plt.figure(figsize=(5.5 * n_panels, 3.5))

    for pi, (label, snr_col, ra_col, dec_col) in enumerate(panels):
        ax  = fig.add_subplot(1, n_panels, pi + 1, projection='mollweide')
        sub = synpta_sim_df.copy()

        if snr_col not in sub.columns or ra_col not in sub.columns:
            ax.set_title(f'{label}\n(no sky data)', fontsize=9)
            continue

        ra  = sub[ra_col].to_numpy(float)
        dec = sub[dec_col].to_numpy(float)
        snr = sub[snr_col].to_numpy(float)
        mask = np.isfinite(ra) & np.isfinite(dec) & np.isfinite(snr) & (snr > 0)

        if mask.sum() == 0:
            ax.set_title(f'{label}\n(no valid positions)', fontsize=9)
            continue

        ra_moll = ra[mask] - np.pi   # shift [0, 2π] → [-π, π] for Mollweide
        sc = ax.scatter(ra_moll, dec[mask], c=np.log10(snr[mask]),
                        cmap='viridis', s=3, alpha=0.5, rasterized=True)
        ax.set_xticklabels([])
        ax.set_yticklabels([])
        ax.set_title(label, fontsize=9, style='italic')
        ax.grid(True, alpha=0.3)
        plt.colorbar(sc, ax=ax, label=r'$\log_{10}(\rho)$', shrink=0.7, pad=0.05)

    fig.suptitle('Sky distribution of loudest sources (synthetic PTAs)', fontsize=11, y=1.01)
    plt.tight_layout()
    fig.savefig(figure_dir / 'synpta_sky_maps.pdf', dpi=300, bbox_inches='tight')
    plt.show(); plt.close(fig)
    print('  saved synpta_sky_maps.pdf')

  saved synpta_sky_maps.pdf


### 7 - Fixed up binary params plots


In [155]:
# ============================================================
# SYNTHETIC PTA — STEP 2 (revised): individual parameter plots
# per synthetic PTA scenario, mirroring _save_hist_figure style.
# ============================================================

SHOW_BASELINE_OVERLAY = True
BASELINE_ALPHA        = 0.15

if synpta_sim_df.empty or not synthetic_pta_labels:
    print('No synthetic PTA data — skipping binary parameter plots.')
else:
    pop_scenarios = [s for s in ('optimistic', 'realistic', 'pessimistic')
                     if s in synpta_sim_df['scenario'].unique()]

    for label, field in zip(synthetic_pta_labels, synthetic_snr_fields):
        syn_detected = synpta_sim_df[synpta_sim_df[f'loudest_{field}'] >= CGW_SNR_THRESHOLD]

        for param_key, xlabel, use_log in PARAM_SPECS:

            # ── choose the right subset ────────────────────────────────────
            if param_key == 'snr':
                subsets = [
                    (synpta_sim_df, 'all'),        # all simulations
                    (syn_detected,  'detected'),   # above threshold only
                ]
            else:
                subsets = [(syn_detected, 'detected')]
            # ──────────────────────────────────────────────────────────────

            for df_subset, subset_tag in subsets:

                if param_key == 'snr':
                    col = f'loudest_{field}'
                else:
                    col = f'{field}_{param_key}'

                base_cgw_col = _BASELINE_COL_MAP.get(param_key)

                pop_vals = {}
                for pop in pop_scenarios:
                    sub = df_subset[df_subset['scenario'] == pop]
                    if col not in sub.columns:
                        continue
                    v = sub[col].to_numpy(float)
                    v = v[np.isfinite(v)]
                    if v.size:
                        pop_vals[pop] = v

                if not pop_vals:
                    continue

                base_pop_vals = {}
                if SHOW_BASELINE_OVERLAY and base_cgw_col and not cgw_sim_df.empty:
                    for pop in pop_scenarios:
                        # for SNR 'all' panel: no threshold filter on baseline either
                        if param_key == 'snr' and subset_tag == 'all':
                            base_sub = cgw_sim_df[cgw_sim_df['scenario'] == pop]
                        else:
                            base_sub = cgw_sim_df[
                                (cgw_sim_df['scenario'] == pop) &
                                (cgw_sim_df['loudest_cgw_snr'] >= CGW_SNR_THRESHOLD)
                            ]
                        bv = base_sub[base_cgw_col].to_numpy(float)
                        if bv.size:
                            base_pop_vals[pop] = bv

                # ── 95% credible intervals ─────────────────────────────────
                print(f'\n95% credible intervals — {label} | {param_key} | {subset_tag}')
                print(f'  {"Scenario":<15}  {"2.5%":>12}  {"Median":>12}  {"97.5%":>12}  {"N":>6}')
                print(f'  {"-"*15}  {"-"*12}  {"-"*12}  {"-"*12}  {"-"*6}')
                for pop in pop_scenarios:
                    if pop in pop_vals:
                        v = pop_vals[pop]
                        lo, med, hi = np.percentile(v, [2.5, 50, 97.5])
                        pop_label = _SYNPTA_LABEL_MAP.get(pop, pop)
                        print(f'  {pop_label:<15}  {lo:>12.4g}  {med:>12.4g}  {hi:>12.4g}  {len(v):>6}')
                if base_pop_vals:
                    print(f'  {"--- NG15 baseline ---":<15}')
                    for pop in pop_scenarios:
                        if pop in base_pop_vals:
                            bv = base_pop_vals[pop]
                            lo, med, hi = np.percentile(bv, [2.5, 50, 97.5])
                            pop_label = _SYNPTA_LABEL_MAP.get(pop, pop)
                            print(f'  {pop_label:<15}  {lo:>12.4g}  {med:>12.4g}  {hi:>12.4g}  {len(bv):>6}')
                # ──────────────────────────────────────────────────────────

                all_vals = np.concatenate(
                    list(pop_vals.values()) + list(base_pop_vals.values())
                )
                pos_mask = (all_vals > 0) if use_log else np.ones(len(all_vals), dtype=bool)
                all_vals = all_vals[np.isfinite(all_vals) & pos_mask]
                if all_vals.size == 0:
                    continue

                bins = (np.logspace(np.log10(all_vals.min()), np.log10(all_vals.max()), 25)
                        if use_log and all_vals.min() > 0
                        else np.linspace(all_vals.min(), all_vals.max(), 25))

                fig, ax = plt.subplots(figsize=(3.5, 2.8))

                if SHOW_BASELINE_OVERLAY:
                    for pop, bv in base_pop_vals.items():
                        st = _SYNPTA_POP_STYLES.get(pop, {'color': '#888'})
                        ax.hist(bv, bins=bins, histtype='stepfilled',
                                color=st['color'], alpha=BASELINE_ALPHA,
                                density=False, zorder=1)

                for pop in pop_scenarios:
                    vals = pop_vals.get(pop)
                    if vals is None:
                        continue
                    st = _SYNPTA_POP_STYLES.get(pop, {'color': '#888', 'linestyle': '-', 'linewidth': 1.5})
                    ax.hist(vals, bins=bins, histtype='step',
                            color=st['color'], linestyle=st['linestyle'],
                            linewidth=st['linewidth'], density=False, zorder=2)

                # vertical threshold line on SNR plots
                if param_key == 'snr':
                    ax.axvline(CGW_SNR_THRESHOLD, color='black', linestyle='--',
                               linewidth=1.5, zorder=3)

                if use_log:
                    ax.set_xscale('log')
                ax.set_xlabel(xlabel, fontsize=11)
                ax.set_ylabel('Number of simulations', fontsize=11)
                ax.tick_params(labelsize=8)
                ax.spines['top'].set_visible(True)
                ax.spines['right'].set_visible(True)

                import matplotlib.patches as mpatches
                handles = []
                if SHOW_BASELINE_OVERLAY and base_pop_vals:
                    handles.append(mpatches.Patch(
                        facecolor='grey', alpha=BASELINE_ALPHA + 0.25, label='NG15'))
                for pop in pop_scenarios:
                    if pop not in pop_vals:
                        continue
                    st = _SYNPTA_POP_STYLES.get(pop, {'color': '#888', 'linestyle': '-', 'linewidth': 1.5})
                    handles.append(mlines.Line2D([], [],
                        color=st['color'], linestyle=st['linestyle'],
                        linewidth=st['linewidth'],
                        label=_SYNPTA_LABEL_MAP.get(pop, pop)))
                ax.legend(handles=handles, frameon=False, fontsize=7, loc='upper right')

                fig.tight_layout()
                # ── filename includes subset tag for SNR panels ────────────
                fname = Path('figures') / f'synpta_{label}_{param_key}_{subset_tag}.pdf'
                fig.savefig(fname, dpi=300, bbox_inches='tight')
                plt.close(fig)
                print(f'  saved {fname}')


95% credible intervals — 4x_precision | f | detected
  Scenario                 2.5%        Median         97.5%       N
  ---------------  ------------  ------------  ------------  ------
  $m_{\mathrm{char}} = 10^{9.3}\;\mathrm{M}_{\odot}$     2.293e-09     1.131e-08     3.626e-08     190
  $m_{\mathrm{char}} = 10^{9.0}\;\mathrm{M}_{\odot}$     2.163e-09      6.15e-09     2.602e-08     146
  $m_{\mathrm{char}} = 10^{8.7}\;\mathrm{M}_{\odot}$     2.555e-09      6.38e-09     3.174e-08     107
  --- NG15 baseline ---
  $m_{\mathrm{char}} = 10^{9.3}\;\mathrm{M}_{\odot}$     2.265e-09     7.551e-09     2.742e-08     171
  $m_{\mathrm{char}} = 10^{9.0}\;\mathrm{M}_{\odot}$     2.157e-09     5.211e-09     1.725e-08     135
  $m_{\mathrm{char}} = 10^{8.7}\;\mathrm{M}_{\odot}$     2.548e-09     5.958e-09     3.038e-08     103
  saved figures/synpta_4x_precision_f_detected.pdf

95% credible intervals — 4x_precision | Mc | detected
  Scenario                 2.5%        Median         97.5%   

In [158]:
from collections import defaultdict

ci_data = defaultdict(lambda: defaultdict(lambda: defaultdict(dict)))

for param_key, xlabel, use_log in PARAM_SPECS:
    for label, field in zip(synthetic_pta_labels, synthetic_snr_fields):

        if param_key == 'snr':
            col = f'loudest_{field}'
        else:
            col = f'{field}_{param_key}'

        for subset_tag, df_subset in [
            ('all',      synpta_sim_df),
            ('detected', synpta_sim_df[synpta_sim_df[f'loudest_{field}'] >= CGW_SNR_THRESHOLD]),
        ]:
            if col not in df_subset.columns:
                continue
            for pop in pop_scenarios:
                vals = df_subset.loc[df_subset['scenario'] == pop, col].to_numpy(float)
                vals = vals[np.isfinite(vals)]
                if len(vals) == 0:
                    continue
                lo, med, hi = np.nanpercentile(vals, [2.5, 50, 97.5])
                ci_data[param_key][subset_tag][label][pop] = (lo, med, hi, len(vals))

    # NG15 baseline from cgw_sim_df
    base_col = _BASELINE_COL_MAP.get(param_key)
    if base_col and not cgw_sim_df.empty:
        for subset_tag, df_subset in [
            ('all',      cgw_sim_df),
            ('detected', cgw_sim_df[cgw_sim_df['loudest_cgw_snr'] >= CGW_SNR_THRESHOLD]),
        ]:
            if base_col not in df_subset.columns:
                continue
            for pop in pop_scenarios:
                vals = df_subset.loc[df_subset['scenario'] == pop, base_col].to_numpy(float)
                vals = vals[np.isfinite(vals)]
                if len(vals) == 0:
                    continue
                lo, med, hi = np.nanpercentile(vals, [2.5, 50, 97.5])
                ci_data[param_key][subset_tag]['NG15'][pop] = (lo, med, hi, len(vals))

print("ci_data rebuilt. counts:")
for param_key in ci_data:
    for subset_tag in ci_data[param_key]:
        for pta_label in ci_data[param_key][subset_tag]:
            for pop, entry in ci_data[param_key][subset_tag][pta_label].items():
                print(f"  {param_key:6s} {subset_tag:8s} {pta_label:20s} {pop:12s}  n={entry[3]}")

ci_data rebuilt. counts:
  f      all      4x_precision         optimistic    n=400
  f      all      4x_precision         realistic     n=400
  f      all      4x_precision         pessimistic   n=400
  f      all      5x_cad_4x_prec       optimistic    n=400
  f      all      5x_cad_4x_prec       realistic     n=400
  f      all      5x_cad_4x_prec       pessimistic   n=400
  f      all      5x_cadence           optimistic    n=400
  f      all      5x_cadence           realistic     n=400
  f      all      5x_cadence           pessimistic   n=400
  f      all      NG15                 optimistic    n=400
  f      all      NG15                 realistic     n=400
  f      all      NG15                 pessimistic   n=400
  f      detected 4x_precision         optimistic    n=190
  f      detected 4x_precision         realistic     n=146
  f      detected 4x_precision         pessimistic   n=107
  f      detected 5x_cad_4x_prec       optimistic    n=245
  f      detected 5x_cad_4x_pre

In [160]:
Path('tables').mkdir(exist_ok=True)
all_pta_labels = list(synthetic_pta_labels) + ['NG15']

import math

def _log10_mchar_str(mc_med):
    if mc_med <= 0:
        return f'${mc_med:.2e}$'
    return r'$10^{' + f'{math.log10(mc_med):.1f}' + r'}$'

def _fmt_val(med, lo, hi, decimals=2):
    fmt = f'{{:.{decimals}f}}'
    return (r'$' + fmt.format(med)
            + r'^{+' + fmt.format(hi - med) + r'}'
            + r'_{-' + fmt.format(med - lo) + r'}$')

_SCENARIO_MCHAR_LABEL = {
    'optimistic':  r'$10^{9.3}$',
    'realistic':   r'$10^{9.0}$',
    'pessimistic': r'$10^{8.7}$',
}

def _build_table(param_key, subset_tag, col_header, chirp_key='Mc',
                 conv=1.0, decimals=2, caption=None, label=None):
    param_data = ci_data.get(param_key, {}).get(subset_tag, {})

    if not param_data:
        print(f"ERROR: ci_data['{param_key}']['{subset_tag}'] is empty.")
        print("Available param keys:", list(ci_data.keys()))
        return

    present_labels = [l for l in all_pta_labels if l in param_data]
    if not present_labels:
        print("ERROR: no matching labels found in param_data.")
        print("Labels in param_data:", list(param_data.keys()))
        print("all_pta_labels:", all_pta_labels)
        return

    lines = []
    lines.append(r'\begin{table}[htbp]')
    lines.append(r'\centering')
    lines.append(r'\begin{tabular}{lll}')
    lines.append(r'\toprule')
    lines.append(r'$m_{\mathrm{char}}\ [M_\odot]$ & PTA & ' + col_header + r' \\')
    lines.append(r'\midrule')

    for pop in pop_scenarios:
        # use ground-truth label, not inferred from data
        mc_str = _SCENARIO_MCHAR_LABEL.get(pop, pop)

        n_rows = sum(
            1 for pl in present_labels
            if param_data.get(pl, {}).get(pop) is not None
        )
        if n_rows == 0:
            print(f"  WARNING: no data for scenario '{pop}', skipping.")
            continue

        first_row = True
        for pta_label in present_labels:
            entry = param_data.get(pta_label, {}).get(pop)
            if entry is None:
                continue

            lo, med, hi, n = entry
            lo, med, hi = lo * conv, med * conv, hi * conv
            val_str = _fmt_val(med, lo, hi, decimals=decimals)
            pta_str = str(pta_label).replace('_', r'\_')

            mchar_cell = (r'\multirow{' + str(n_rows) + r'}{*}{' + mc_str + r'}'
                          if first_row else '')
            first_row = False
            lines.append(f'{mchar_cell} & {pta_str} & {val_str} \\\\')

        lines.append(r'\addlinespace')

    if lines[-1] == r'\addlinespace':
        lines.pop()

    lines.append(r'\bottomrule')
    lines.append(r'\end{tabular}')
    lines.append(r'\caption{' + (caption or col_header) + r'}')
    lines.append(r'\label{' + (label or f'tab:{param_key}_{subset_tag}') + r'}')
    lines.append(r'\end{table}')

    tex = '\n'.join(lines)
    fname = Path('tables') / f'ci_{param_key}_{subset_tag}.tex'
    fname.write_text(tex)
    print(f'  saved {fname}')
    print(tex)
for param_key in ('snr', 'f'):
    for subset_tag in ('all', 'detected'):
        d = ci_data.get(param_key, {}).get(subset_tag, {})
        for pta_label, pop_dict in d.items():
            for pop, entry in pop_dict.items():
                lo, med, hi, n = entry
                print(f"  {param_key:6s} {subset_tag:8s} {pta_label:20s} {pop:12s}  "
                      f"med={med:.3g}  [{lo:.3g}, {hi:.3g}]  n={n}")    
    
# ── frequency (detected only) ─────────────────────────────────────────────
_build_table(
    param_key  = 'f',
    subset_tag = 'detected',
    col_header = 'Frequency [nHz]',
    conv       = 1e9,
    decimals   = 2,
    caption    = (r'Summary of 95\% credible intervals for the gravitational wave'
                  r' frequencies of detectable SMBHBs for each of the pulsar'
                  r' timing array observation settings used.'),
    label      = 'tab:f_pta',
)

# ── SNR detected ──────────────────────────────────────────────────────────
_build_table(
    param_key  = 'snr',
    subset_tag = 'detected',
    col_header = r'(S/N)$_{s}$',
    conv       = 1.0,
    decimals   = 2,
    caption    = (r'Summary of 95\% credible intervals for the SNR of detectable'
                  r' SMBHBs for each of the pulsar timing array observation'
                  r' settings used.'),
    label      = 'tab:snr_detected_pta',
)

# ── SNR all (includes non-detections) ────────────────────────────────────
_build_table(
    param_key  = 'snr',
    subset_tag = 'all',
    col_header = r'(S/N)$_{s}$',
    conv       = 1.0,
    decimals   = 2,
    caption    = (r'Summary of 95\% credible intervals for the SNR of all simulated'
                  r' SMBHBs (including sub-threshold) for each of the pulsar timing'
                  r' array observation settings used.'),
    label      = 'tab:snr_all_pta',
)

  snr    all      4x_precision         optimistic    med=5.67  [0.602, 39.4]  n=400
  snr    all      4x_precision         realistic     med=4.28  [0.485, 119]  n=400
  snr    all      4x_precision         pessimistic   med=2.88  [0.344, 51.1]  n=400
  snr    all      5x_cad_4x_prec       optimistic    med=7.59  [0.778, 49.6]  n=400
  snr    all      5x_cad_4x_prec       realistic     med=5.64  [0.709, 119]  n=400
  snr    all      5x_cad_4x_prec       pessimistic   med=3.65  [0.44, 53.2]  n=400
  snr    all      5x_cadence           optimistic    med=5.74  [0.615, 39.4]  n=400
  snr    all      5x_cadence           realistic     med=4.46  [0.504, 119]  n=400
  snr    all      5x_cadence           pessimistic   med=2.92  [0.352, 51.1]  n=400
  snr    all      NG15                 optimistic    med=4.88  [0.568, 39.5]  n=400
  snr    all      NG15                 realistic     med=3.83  [0.404, 119]  n=400
  snr    all      NG15                 pessimistic   med=2.52  [0.281, 48.4]  n=4

In [157]:
print("Keys in ci_data:", list(ci_data.keys()))
print("Keys in ci_data['frequency']:", list(ci_data.get('frequency', {}).keys()))
print("Keys in ci_data['frequency']['detected']:", 
      list(ci_data.get('frequency', {}).get('detected', {}).keys()))
print("Keys in ci_data['chirp_mass']['detected']:", 
      list(ci_data.get('chirp_mass', {}).get('detected', {}).keys()))
print()
print("all_pta_labels:", all_pta_labels)
print("present_labels:", present_labels)

Keys in ci_data: ['f', 'Mc', 'h0', 'D_comov', 'snr', 'frequency']
Keys in ci_data['frequency']: ['detected']
Keys in ci_data['frequency']['detected']: []
Keys in ci_data['chirp_mass']['detected']: []

all_pta_labels: ['4x_precision', '5x_cad_4x_prec', '5x_cadence', 'NG15']
present_labels: ['4x_precision', '5x_cad_4x_prec', '5x_cadence', 'NG15']


### 8 — Summary statistics table

Concise numerical summary of loudest-source SNR for baseline and each synthetic scenario,
broken down by population scenario.

In [152]:
import pandas as pd
import numpy as np

if synpta_sim_df.empty:
    print('No synthetic PTA data — skipping summary table.')
else:
    pop_scenarios = [s for s in ('optimistic', 'realistic', 'pessimistic')
                     if s in synpta_sim_df['scenario'].unique()]

    all_cols = (['loudest_cgw_snr'] +
                [f'loudest_{f}' for f in synthetic_snr_fields] +
                [f'snr_ratio_{l}' for l in synthetic_pta_labels])

    rows = []
    for pop in pop_scenarios:
        sub = synpta_sim_df[synpta_sim_df['scenario'] == pop]
        for col in all_cols:
            if col not in sub.columns:
                continue
            v = sub[col].to_numpy(float)
            v = v[np.isfinite(v)]
            if v.size == 0:
                continue
            if col == 'loudest_cgw_snr':
                pta = 'baseline'
            elif col.startswith('loudest_cgw_snr_'):
                pta = col[len('loudest_cgw_snr_'):]
            else:
                pta = col
            rows.append({
                'pop_scenario': pop,
                'PTA': pta,
                'n_sims': v.size,
                'median': np.median(v),
                'mean': np.mean(v),
                'p16': np.percentile(v, 16),
                'p84': np.percentile(v, 84),
                'max': v.max(),
                f'frac_above_{CGW_SNR_THRESHOLD:.1f}': (v >= CGW_SNR_THRESHOLD).mean(),
            })

    summary_tbl = pd.DataFrame(rows)
    print('\nSynthetic PTA Summary Statistics')
    print('=' * 80)
    with pd.option_context('display.float_format', '{:.3g}'.format, 'display.max_columns', 20):
        display(summary_tbl)


Synthetic PTA Summary Statistics


,pop_scenario,PTA,n_sims,median,mean,p16,p84,max,frac_above_5.9
0,optimistic,baseline,400,4.88,11.3,1.7,12.7,560,0.427
1,optimistic,4x_precision,400,5.67,12.1,1.99,14.5,559,0.475
2,optimistic,5x_cad_4x_prec,400,7.59,14.2,2.76,17.4,559,0.613
3,optimistic,5x_cadence,400,5.74,12.3,2.04,14.5,559,0.482
4,optimistic,snr_ratio_4x_precision,400,1.11,1.15,1,1.31,1.6,0
5,optimistic,snr_ratio_5x_cad_4x_prec,400,1.44,1.58,1.02,2.14,3.83,0
6,optimistic,snr_ratio_5x_cadence,400,1.13,1.18,1,1.37,1.75,0
7,realistic,baseline,400,3.83,14.4,1.56,13.2,550,0.338
8,realistic,4x_precision,400,4.28,14.9,1.78,13.6,550,0.365
9,realistic,5x_cad_4x_prec,400,5.64,16.2,2.48,15,550,0.475


In [153]:
# ============================================================
# SYNTHETIC PTA — DIAGNOSTIC: did the loudest source change?
#
# For each simulation, compares the identity of the loudest binary
# in the synthetic PTA against the loudest binary in the baseline.
# Uses global_idx as the binary identifier. If the synthetic PTA
# promotes a previously-quiet source to the top, that simulation
# is flagged as "source changed".
#
# Requires binary_df to be loaded (has global_idx per binary).
# ============================================================

if synpta_sim_df.empty or not synthetic_snr_fields:
    print('No synthetic PTA data available.')
elif 'global_idx' not in binary_df.columns:
    print('global_idx not in binary_df — cannot match sources across PTAs.')
else:
    change_records = []

    for fp in result_files:
        try:
            scenario  = infer_scenario(fp)
            run_id    = infer_run_id(fp)
            sim_index = _infer_sim_index_from_path(fp)
            run_scope = _infer_run_scope_from_path(fp)
            sim_name  = fp.parent.name
            sim_key   = f'{run_scope}/{sim_name}' if run_scope else sim_name

            payload = _load_payload(fp, verbose=False)
            if not isinstance(payload, dict) or 'arrays' not in payload:
                continue
            arrays = payload['arrays']
            if not isinstance(arrays, dict) or 'f' not in arrays:
                continue

            n = len(arrays['f'])
            global_idx = np.asarray(arrays.get('global_idx', np.arange(n)), dtype=np.int64)

            # baseline loudest binary
            base_snr = np.asarray(arrays.get('cgw_snr', np.zeros(n)), dtype=float)
            base_loudest_idx = int(np.nanargmax(base_snr))
            base_loudest_gidx = int(global_idx[base_loudest_idx])
            base_loudest_snr  = float(base_snr[base_loudest_idx])
            base_loudest_f    = float(arrays['f'][base_loudest_idx])

            row = {
                'scenario':        scenario,
                'sim_key':         sim_key,
                'sim_index':       sim_index,
                'base_loudest_gidx': base_loudest_gidx,
                'base_loudest_snr':  base_loudest_snr,
                'base_loudest_f':    base_loudest_f,
            }

            for field in synthetic_snr_fields:
                syn_label = field[len('cgw_snr_'):]
                syn_snr   = np.asarray(arrays.get(field, np.zeros(n)), dtype=float)

                if np.all(syn_snr == 0) or np.all(~np.isfinite(syn_snr)):
                    row[f'{syn_label}_loudest_gidx']   = np.nan
                    row[f'{syn_label}_loudest_snr']    = np.nan
                    row[f'{syn_label}_loudest_f']      = np.nan
                    row[f'{syn_label}_source_changed'] = np.nan
                    row[f'{syn_label}_base_snr_rank']  = np.nan
                    row[f'{syn_label}_snr_ratio']      = np.nan
                    continue

                syn_loudest_idx  = int(np.nanargmax(syn_snr))
                syn_loudest_gidx = int(global_idx[syn_loudest_idx])
                syn_loudest_snr  = float(syn_snr[syn_loudest_idx])
                syn_loudest_f    = float(arrays['f'][syn_loudest_idx])

                source_changed = (syn_loudest_gidx != base_loudest_gidx)

                # what rank was the new loudest source in the *baseline* SNR ordering?
                # rank 1 = was already the loudest in baseline
                if source_changed:
                    base_snr_of_new_source = float(base_snr[syn_loudest_idx])
                    base_rank = int(np.sum(base_snr >= base_snr_of_new_source))
                else:
                    base_rank = 1

                row[f'{syn_label}_loudest_gidx']   = syn_loudest_gidx
                row[f'{syn_label}_loudest_snr']    = syn_loudest_snr
                row[f'{syn_label}_loudest_f']      = syn_loudest_f
                row[f'{syn_label}_source_changed'] = source_changed
                row[f'{syn_label}_base_snr_rank']  = base_rank
                row[f'{syn_label}_snr_ratio']      = (syn_loudest_snr / base_loudest_snr
                                                      if base_loudest_snr > 0 else np.nan)

            change_records.append(row)

        except Exception as e:
            continue

    change_df = pd.DataFrame(change_records)

    if change_df.empty:
        print('No records built — check that arrays contain global_idx and cgw_snr fields.')
    else:
        print(f'Analysed {len(change_df)} simulations across '
              f'{change_df["scenario"].nunique()} population scenarios.\n')

        # ── summary table ────────────────────────────────────────────────────
        summary_rows = []
        for syn_label in synthetic_pta_labels:
            changed_col = f'{syn_label}_source_changed'
            rank_col    = f'{syn_label}_base_snr_rank'
            ratio_col   = f'{syn_label}_snr_ratio'
            if changed_col not in change_df.columns:
                continue

            for pop in sorted(change_df['scenario'].unique()):
                sub = change_df[change_df['scenario'] == pop]
                valid = sub[changed_col].notna()
                n_sims      = valid.sum()
                n_changed   = sub.loc[valid, changed_col].sum()
                pct_changed = 100 * n_changed / n_sims if n_sims else np.nan

                # median baseline rank of the newly-promoted source
                # (only for sims where source changed)
                changed_mask = valid & sub[changed_col].astype(bool)
                med_rank = (sub.loc[changed_mask, rank_col].median()
                            if changed_mask.sum() else np.nan)

                # median SNR ratio (synthetic / baseline loudest)
                med_ratio = sub.loc[valid, ratio_col].median()

                summary_rows.append({
                    'synthetic PTA':      syn_label,
                    'population scenario': pop,
                    'n sims':             int(n_sims),
                    'source changed (n)': int(n_changed),
                    'source changed (%)': round(pct_changed, 1),
                    'median baseline rank\nof promoted source': (round(med_rank, 1)
                                                                  if not np.isnan(med_rank)
                                                                  else '—'),
                    'median SNR ratio\n(syn / baseline)':      round(med_ratio, 3),
                })

        summary_tbl = pd.DataFrame(summary_rows)
        print('=== Did the loudest source change between baseline and synthetic PTA? ===\n')
        display(summary_tbl)

        # ── frequency shift for changed sources ──────────────────────────────
        print('\n=== Frequency of the promoted source vs baseline loudest (changed sims only) ===\n')
        freq_rows = []
        for syn_label in synthetic_pta_labels:
            changed_col  = f'{syn_label}_source_changed'
            syn_f_col    = f'{syn_label}_loudest_f'
            if changed_col not in change_df.columns:
                continue
            for pop in sorted(change_df['scenario'].unique()):
                sub          = change_df[change_df['scenario'] == pop]
                changed_mask = sub[changed_col].astype(bool)
                if not changed_mask.any():
                    continue
                base_f = sub.loc[changed_mask, 'base_loudest_f']
                syn_f  = sub.loc[changed_mask, syn_f_col]
                freq_rows.append({
                    'synthetic PTA':       syn_label,
                    'population scenario': pop,
                    'n changed sims':      int(changed_mask.sum()),
                    'median baseline loudest f [Hz]': f'{base_f.median():.3e}',
                    'median promoted source f [Hz]':  f'{syn_f.median():.3e}',
                    'promoted source higher f (%)':   round(
                        100 * (syn_f.to_numpy() > base_f.to_numpy()).mean(), 1),
                })

        display(pd.DataFrame(freq_rows))

Analysed 1200 simulations across 3 population scenarios.

=== Did the loudest source change between baseline and synthetic PTA? ===



,synthetic PTA,population scenario,n sims,source changed (n),source changed (%),median baseline rank\nof promoted source,median SNR ratio\n(syn / baseline)
0,4x_precision,optimistic,400,71,17.8,2.0,1.106
1,4x_precision,pessimistic,400,49,12.2,3.0,1.024
2,4x_precision,realistic,400,63,15.8,2.0,1.043
3,5x_cad_4x_prec,optimistic,400,196,49.0,3.0,1.440
4,5x_cad_4x_prec,pessimistic,400,137,34.2,4.0,1.181
5,5x_cad_4x_prec,realistic,400,168,42.0,4.0,1.324
6,5x_cadence,optimistic,400,83,20.8,2.0,1.128
7,5x_cadence,pessimistic,400,57,14.2,2.0,1.035
8,5x_cadence,realistic,400,72,18.0,2.0,1.057



=== Frequency of the promoted source vs baseline loudest (changed sims only) ===



,synthetic PTA,population scenario,n changed sims,median baseline loudest f [Hz],median promoted source f [Hz],promoted source higher f (%)
0,4x_precision,optimistic,71,6.639e-09,2.413e-08,100.0
1,4x_precision,pessimistic,49,6.692e-09,2.388e-08,89.8
2,4x_precision,realistic,63,7.396e-09,2.334e-08,100.0
3,5x_cad_4x_prec,optimistic,196,7.590e-09,2.730e-08,100.0
4,5x_cad_4x_prec,pessimistic,137,6.676e-09,3.418e-08,97.1
5,5x_cad_4x_prec,realistic,168,7.946e-09,3.713e-08,100.0
6,5x_cadence,optimistic,83,6.465e-09,2.413e-08,100.0
7,5x_cadence,pessimistic,57,6.683e-09,2.406e-08,91.2
8,5x_cadence,realistic,72,8.631e-09,2.325e-08,100.0
